# ViT-MAE Base Pre-training Pipeline — DIMER E2E masked-image-modelling tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/tutorials/vit_mae_pretraining_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fvit--mae--base-ffcc4d?style=flat)](https://huggingface.co/facebook/vit-mae-base) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fmae-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/mae) [![arXiv](https://img.shields.io/badge/arXiv-2111.06377-b31b1b.svg)](https://arxiv.org/abs/2111.06377)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** masked-patch reconstruction (masked image modelling), mean-pooled encoder embeddings, a linear probe on those embeddings and bounded continuation of the masked-autoencoding objective on a photograph set, using the pinned `facebook/vit-mae-base` weights

**This notebook is standalone.** It carries the repository's package (6 modules under `src/vit_mae_pipeline/`, at revision `828abad1fc4c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `25b184bea5538bf5c4c852c79d221195fdd2778d` (~448 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (nothing is installed into the notebook's own Python, so no restart is needed and Run all completes in one pass), stages and digest-verifies the pinned `facebook/vit-mae-base` snapshot (a 448 MB `model.safetensors`; no pickle is opened anywhere), fetches the 360 pinned iNaturalist photographs from the project's open-data bucket (about 39 MB, each refused on any byte-size or SHA-256 mismatch), cuts them per species into 216 / 48 / 96 training, validation and test photographs, masks and reconstructs three drawn shapes through the inference contract with an input manifest and a rejection probe, measures the frozen model's masked-patch MSE over the 96 test photographs beside the mean-patch and blur fills and its linear-probe accuracy beside the majority floor, a colour nearest neighbour and a k-NN on the same features, runs a bounded continuation of the masked-autoencoding objective on the decoder and the last two encoder blocks with validation-MSE epoch selection, reconstructs and probes the held-out photographs again, re-reconstructs the drawn shapes with the adapted model, exports the adapter and the probe head as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify reconstruction and probe parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). On CPU the whole path took about four minutes of model time on the build workstation's CPU after the downloads (expect longer on a 2-vCPU hosted runtime); a CUDA runtime is used automatically when present and finishes in a few minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and either set `BYOD_PATH` to a zip or folder in the runtime (Colab, Kaggle or Jupyter) or leave it empty to upload one zip in Colab, then choose **Run after** from that cell (it first puts the model back to the pinned base) to supply a `labels.csv` (columns `id`, `file`, `label`) beside the image files — at least **12 photographs for two labels** (6 per label), 15 for three, and 4 per label from four labels up (`min_byod_records(n_labels)`); the labels feed only the linear probe, the reconstruction objective never sees them. They pass through the same validation, seeded stratified split, baselines, continuation, held-out evaluation, artifact export and reload-parity cells as the iNaturalist sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`facebook/vit-mae-base` is the ViT-B/16 masked autoencoder of He, Chen, Xie, Li, Dollár and Girshick (CVPR 2022): a 12-layer, 768-wide encoder that sees only the visible 25 % of an image's 196 16×16 patches, and an 8-layer, 512-wide decoder that predicts the pixels of the hidden 75 % from the encoder's tokens plus a shared mask token — 111,907,840 parameters in total, pre-trained on ImageNet-1K for 1,600 epochs with per-patch pixel MSE on the hidden patches only, published under the **Apache-2.0** licence. It has **no user-facing task output**: the model returns a reconstruction and its own loss, and the encoder's features are what downstream fine-tuning consumes. This notebook therefore reads the model two ways — the **masked-patch MSE** of its reconstructions (the objective, in the processor's normalised pixel space, lower is better, not a quality judgement) and the **accuracy of a linear probe** on its mean-pooled patch tokens (the paper's own readout of representation quality) — and neither is calibrated or a human judgement.

What this notebook adds to inference is **continuation of the pre-training objective on a photograph set**. The dataset is real: 360 CC0-licensed, research-grade iNaturalist photographs of six common North American birds (**CC0 1.0**; three sparrows, a junco and two finches, 60 per species, one per observer), pinned by photo id, byte size and SHA-256 and fetched from the project's open-data bucket at run time. The honest question is narrow: does a bounded continuation of masked autoencoding — the decoder and the last two encoder blocks, on 216 photographs, with the epoch chosen on the validation masked MSE — move the reconstruction error on an image-disjoint test split, against two **non-neural fills** (the **mean colour of the visible patches** and a **blur of the visible neighbours**), and does it move the probe against three **non-neural classifiers** (the **majority floor**, a **colour nearest neighbour** and a **k-NN on the same features**)? The build record's answer is that it barely does — the pre-trained model is already converged on natural photographs, and 216 more of them at a learning rate small enough not to hurt move the held-out mean masked MSE from 0.2281 to 0.2280 while the median rose from 0.1801 to 0.1827 — so what the notebook demonstrates is the adaptation *contract* (bounded training, validation selection that never returns an epoch worse than the frozen model *on the validation split* — the held-out error can still move either way — an artifact that reloads with verified parity) and how to read the numbers around it. Nothing here is a quality claim about your photographs: it is one seeded split of one sample.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 4-file manifest: model card, config, processor config and weights) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled photograph set, validate it and split it per species without leakage; mask and reconstruct drawn shapes through the public API and read a reconstruction loss correctly (the model's own objective, seeded masks, hidden-patch and visible-patch error, uncalibrated); measure the frozen model's masked-patch MSE beside two non-neural fills and its linear-probe accuracy beside three non-neural classifiers; run a bounded continuation of the masked-autoencoding objective with explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split both ways; re-reconstruct drawings from a different image family with the adapted model; and export a safetensors adapter with its probe head that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** image classification as a product (the probe is a readout of the features, not a classifier to ship), object detection, segmentation, generation of new images, inpainting of user-chosen regions (the mask is random, as in pre-training), full fine-tuning of the encoder for a downstream task, pre-training from scratch, normalised-pixel targets (`norm_pix_loss` is off in this checkpoint), training on photographs that are not the pinned sample or your own uploads, evaluation on ImageNet or any benchmark proper (only one seeded 360-photograph sample is scored here), and any claim that six bird species stand in for your images. The repository exposes none of these.

**Who this notebook is for.** A learner who knows basic Python, has used Colab or Jupyter, and wants to see what a masked autoencoder does, how to read its reconstruction loss and its features honestly, and what continuing its pre-training on a small photograph set does — and does not — change. No prior experience with masked autoencoders or fine-tuning is assumed; each term is explained where it first matters and again in the **Glossary** at the end. CPU is adequate (about four minutes of model time); a GPU is faster.

**Input → Model → Output.**

| | Reconstruction | Representation | Bounded continuation |
|---|---|---|---|
| Input | one image (16..4,096 px), resized to 224 × 224, with 75 % of its 196 patches hidden by a seeded random mask | photographs with nothing hidden | unlabelled photographs (216 training and 48 validation in the sample) |
| Model | ViT-MAE base: a 12-layer encoder over the visible patches and an 8-layer decoder that predicts the hidden ones | the encoder's mean-pooled patch tokens, read by a linear probe | the decoder and the last two encoder blocks trained on the model's own masked-patch loss; validation MSE chooses the epoch |
| Output | a reconstruction, its masked and visible MSE and PSNR — the model's own objective, not a perceptual judgement | probe accuracy and macro F1 beside three non-neural classifiers | a safetensors adapter, and held-out reconstruction and probe numbers beside the frozen model |

**How to use this notebook.** Choose a runtime (CPU works; a GPU is faster), then **Runtime → Run all**. Run all completes in one pass: Section 1 installs nothing into the notebook's own Python, so no restart is needed. Sections 1–3 are **infrastructure** — the isolated environment, the carried package (including the 360-photograph table) and the model snapshot — and their cells are collapsed; you may run them without studying them. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited, and the defaults reproduce the recorded run. Before each principal result the notebook asks you to **Predict**; after it come **What to notice** and a collapsible **Check your reasoning** with a worked answer that names the run it quotes — the Kaggle T4 release run of 21 September 2026. Section 10 is a **change-one-thing experiment**, off by default. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Writing your predictions down is optional.

**Roadmap:** 1–3 infrastructure → 4 the photographs, validation and a stratified split *(evaluation practice)* → 5 masking and reconstruction through the inference contract *(core concept: what an MAE predicts)* → 6 two fills, three classifiers and the frozen model *(evaluation practice)* → 7 bounded continuation and validation selection *(core concept)* → 8 held-out evaluation, mean and median *(evaluation practice)* → 9 re-reconstruct, export and reload *(engineering)* → 10 change one thing (optional) → conclude.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; no prior experience with masked autoencoders or fine-tuning. The notebook explains masking, masked and visible MSE, PSNR, the linear probe, k-NN, validation selection and the adapter where they are first used; the Glossary repeats them.
- **Runtime:** a fresh supported **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter). Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter and nothing is installed into it. The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is slow but adequate: the build record measured about 4.4 s to reconstruct the 96 test photographs, 28.8 s to embed the 312 photographs and fit the probe, and 159.1 s for the 5 epochs of continuation (216 photographs per epoch through the encoder and decoder, the decoder and the last two encoder blocks training) including the per-epoch validation reconstruction, so the whole default path is about four minutes of model time on the build workstation's CPU with the snapshot and photographs already cached (a 2-vCPU hosted runtime will be several times slower); a hosted T4 finishes it in a few minutes. The pinned `torch==2.14.0` install and the 448 MB checkpoint are the large downloads of the run; the photographs add about 39 MB.
- **Knowledge:** basic Python and PIL; what a mean squared error and a PSNR are; what a masked autoencoder hides and predicts; what accuracy and macro F1 measure and why a linear probe is a readout of features rather than a product classifier.
- **Data contract:** records are `{id, image, label}` — a PIL image or a file decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px, and a label of at most 64 plain characters (used by the probe only). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a training split needs 8..20,000 records over 2..100 labels, and every label needs one training, one validation and one test photograph, so with the default 15 % + 20 % stratified hold-out the effective BYOD minimum is **12 photographs for two labels** (6 per label), 15 for three, and 4 per label from four labels up (`min_byod_records(n_labels)`); splitting is stratified per label after pixel-digest de-duplication so no photograph lands in two splits. Every image is resized to 224×224 by the processor. BYOD accepts one zip of images plus a `labels.csv` in that shape.
- **Validation is structural, not semantic:** every image is opened and decoded and every label checked, but nothing checks that a label is right — a mislabelled set is probed without complaint, and the reconstruction objective never reads the labels at all.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path fetches 360 JPEG/PNG files from `https://inaturalist-open-data.s3.amazonaws.com/photos/<id>/medium.<ext>` (about 39 MB in total), each pinned by byte size and SHA-256 in the carried `samples.py` and refused on any mismatch; every photograph's iNaturalist observation page and observer login are kept in its record. Each photograph carries the CC0 1.0 licence its observer chose (nothing is committed to the repository).
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/vit-mae-base` snapshot (~448 MB in total) at revision `25b184bea553…`. No GitHub access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 47 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 47 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = 'a5ce92fef4ab5c8d82f9b6fdc84d6a5a045bb22fab979ac9536ca9d40dbe9239'
LOCKED_PACKAGES = 47
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.11 \
    --hash=sha256:1cbdc4ace7fb11d2f6d073f5c3d91537a23b3d58d2bf2fdffc12c6d717a589c6 \
    --hash=sha256:9dc453a1edd25626a6fc97c2cb1bd9f7e46345843d40b3181c071761fb9a5f7a
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   vit-mae-pretraining-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   vit-mae-pretraining-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   vit-mae-pretraining-pipeline (pyproject.toml)
    #   torchvision
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   vit-mae-pretraining-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   vit-mae-pretraining-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via vit-mae-pretraining-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via vit-mae-pretraining-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via vit-mae-pretraining-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

{'isolated_environment': '/content/dimer_isolated_env_a5ce92fef4ab', 'reused': False, 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 47, 'setup_seconds': 58}
{'routed_to': '/content/dimer_isolated_env_a5ce92fef4ab/bin/python', 'worker_pid': 1955, 'worker_reused': False}


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers`, `numpy` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'vit-mae-pretraining-pipeline',
    'repository_revision': '828abad1fc4c4ed398a5da3b5937a1a0f98b35d3',
    'embedded_module': 'src/vit_mae_pipeline/config.py',
    'embedded_modules': ['src/vit_mae_pipeline/config.py', 'src/vit_mae_pipeline/metrics.py', 'src/vit_mae_pipeline/model.py', 'src/vit_mae_pipeline/provenance.py', 'src/vit_mae_pipeline/samples.py', 'src/vit_mae_pipeline/pipeline.py'],
    'module_sha256': 'b1f4baa397e280d44d104c88b0a6a560c452fec6be5c3271fec5d0d8647bdc26',
    'generator': 'build_notebook.py/2.2',
    'notebook_spec': '2.2',
}
import torch, transformers, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'vit-mae-pretraining-pipeline', 'repository_revision': '828abad1fc4c4ed398a5da3b5937a1a0f98b35d3', 'embedded_module': 'src/vit_mae_pipeline/config.py', 'embedded_modules': ['src/vit_mae_pipeline/config.py', 'src/vit_mae_pipeline/metrics.py', 'src/vit_mae_pipeline/model.py', 'src/vit_mae_pipeline/provenance.py', 'src/vit_mae_pipeline/samples.py', 'src/vit_mae_pipeline/pipeline.py'], 'module_sha256': 'b1f4baa397e280d44d104c88b0a6a560c452fec6be5c3271fec5d0d8647bdc26', 'generator': 'build_notebook.py/2.2', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'executable': '/content/dimer_isolated_env_a5ce92fef4ab/bin/python', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'numpy': '2.5.3', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/vit_mae_pipeline/` @ `828abad1fc4c`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 6 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/6:** `src/vit_mae_pipeline/config.py`

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

MODEL_ID = "facebook/vit-mae-base"
MODEL_REVISION = "25b184bea5538bf5c4c852c79d221195fdd2778d"
MODEL_FILENAME = "model.safetensors"
MODEL_SHA256 = "479dcef4bd5df06259399027b789f21e9d9a1b79f37155a64176d55bc26fdae8"
MODEL_SIZE_BYTES = 447_670_680
MODEL_LICENSE = "Apache-2.0"

DEFAULT_MODEL_KEY = "vit-mae-base"
UNSAFE_WEIGHT_EXTENSIONS = (
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".h5",
    ".msgpack",
)

ALLOWED_CHECKPOINT_FILES = (
    "config.json",
    MODEL_FILENAME,
    "preprocessor_config.json",
)

# The checkpoint's own geometry (config.json at the pinned revision).
IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 196
HIDDEN_SIZE = 768
ENCODER_LAYERS = 12
DECODER_HIDDEN_SIZE = 512
DECODER_LAYERS = 8
DEFAULT_MASK_RATIO = 0.75  # config.json mask_ratio: 147 of the 196 patches are hidden from the encoder
NORM_PIX_LOSS = False  # config.json norm_pix_loss: the loss is a raw-pixel MSE on the masked patches

**Module 2/6:** `src/vit_mae_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level measures for the two things this pipeline can be read on, in numpy.

**Reconstruction** (what a masked autoencoder is trained for): the mean squared error between the decoder's
prediction and the true pixels of the patches that were hidden from the encoder, averaged over the hidden
patches of an image and then over images — the checkpoint's own training loss (`norm_pix_loss` false: raw
pixels in the processor's normalised space). Two non-neural references a fine-tuned model must beat: the
**mean-patch fill** (every hidden patch predicted as the mean colour of the visible patches — a decoder that
knows only the image's average) and the **blur fill** (every hidden patch predicted as the mean of its visible
neighbours in the 14 x 14 patch grid — the cheapest spatial prior).

**Linear-probe classification** (how the fleet reads a self-supervised encoder): accuracy and macro F1 of a
linear head on frozen embeddings, with a per-class breakdown, against the **majority floor** and a **colour
nearest neighbour** (the label of the training photograph whose 3 x 3 mean-colour grid is closest — a
classifier that knows the image through 27 numbers) and a **k-nearest-neighbour** on the same embeddings.
"""
# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import math
from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
from PIL import Image

COLOUR_GRID = 3
METRIC_DEFINITIONS = {
    "accuracy": "fraction of photographs whose highest-scoring class is the gold label; in 0..1",
    "macro_f1": "unweighted mean over classes of the F1 of predicting that class; in 0..1",
}
RECONSTRUCTION_DEFINITIONS = {
    "masked_mse": (
        "mean over images of the mean squared error between predicted and true pixels over the patches hidden "
        "from the encoder, in the processor's normalised pixel space (ImageNet mean / std); lower is better"
    ),
    "masked_mse_visible": "the same error over the patches the encoder saw (the decoder also predicts those); lower is better",
    "mask_ratio": "fraction of the patches hidden from the encoder (the seeded random mask)",
    "psnr_masked": "peak signal-to-noise ratio of the hidden patches in 0..1 pixel space, dB; higher is better",
}


# ---------------------------------------------------------------------------------- reconstruction


def masked_mse(prediction: np.ndarray, target: np.ndarray, mask: np.ndarray) -> float:
    """MSE over the patches with mask == 1. `prediction` / `target` are `[patches, values]`, `mask` `[patches]`."""
    pred = np.asarray(prediction, dtype=np.float64)
    true = np.asarray(target, dtype=np.float64)
    m = np.asarray(mask, dtype=np.float64)
    if pred.shape != true.shape or pred.ndim != 2 or m.shape != (pred.shape[0],):
        raise ValueError("prediction and target must be [patches, values] with one mask entry per patch")
    if m.sum() <= 0:
        raise ValueError("at least one patch must be masked")
    per_patch = ((pred - true) ** 2).mean(axis=1)
    return float((per_patch * m).sum() / m.sum())


def psnr_from_mse(mse_normalised: float, *, std: Sequence[float] = (0.229, 0.224, 0.225)) -> float:
    """PSNR (dB) in 0..1 pixel space from an MSE measured in ImageNet-normalised space (per-channel std applied
    as the mean squared std; an approximation stated as such)."""
    scale = float(np.mean(np.square(np.asarray(std, dtype=np.float64))))
    mse_pixels = max(mse_normalised * scale, 1e-12)
    return float(10.0 * math.log10(1.0 / mse_pixels))


def reconstruction_metrics(rows: Sequence[Mapping[str, Any]], *, mask_ratio: float) -> dict[str, Any]:
    """Aggregate per-image rows `{id, masked_mse, visible_mse, ...}` into the corpus reading."""
    if not rows:
        raise ValueError("no images to score")
    masked = np.asarray([r["masked_mse"] for r in rows], dtype=np.float64)
    visible = np.asarray([r["visible_mse"] for r in rows], dtype=np.float64)
    if not (np.all(np.isfinite(masked)) and np.all(np.isfinite(visible))):
        raise ValueError("reconstruction errors must be finite")
    return {
        "n": int(len(rows)),
        "masked_mse": float(masked.mean()),
        "masked_mse_median": float(np.median(masked)),
        "masked_mse_visible": float(visible.mean()),
        "psnr_masked": psnr_from_mse(float(masked.mean())),
        "mask_ratio": float(mask_ratio),
        "definitions": dict(RECONSTRUCTION_DEFINITIONS),
    }


def mean_patch_fill(patches: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Predict every hidden patch as the mean over the visible patches (per value position averaged to one
    colour per channel): the decoder that knows only the image's average."""
    p = np.asarray(patches, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    visible = p[~m]
    if not len(visible):
        raise ValueError("no visible patches")
    values = visible.shape[1]
    channels = 3
    per_channel = visible.reshape(-1, values // channels, channels).mean(axis=(0, 1))
    fill = np.tile(per_channel, values // channels)
    out = p.copy()
    out[m] = fill
    return out


def blur_fill(patches: np.ndarray, mask: np.ndarray, *, grid: int) -> np.ndarray:
    """Predict every hidden patch as the mean colour of its visible 8-neighbours in the patch grid (falling back
    to the mean-patch fill where a patch has no visible neighbour)."""
    p = np.asarray(patches, dtype=np.float64)
    m = np.asarray(mask, dtype=bool)
    if p.shape[0] != grid * grid:
        raise ValueError("patch count must equal grid * grid")
    values = p.shape[1]
    channels = 3
    colours = p.reshape(grid * grid, values // channels, channels).mean(axis=1)  # one colour per patch
    fallback = colours[~m].mean(axis=0) if (~m).any() else np.zeros(channels)
    out = p.copy()
    for index in np.flatnonzero(m):
        row, col = divmod(int(index), grid)
        neighbours = []
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == dc == 0:
                    continue
                r2, c2 = row + dr, col + dc
                if 0 <= r2 < grid and 0 <= c2 < grid and not m[r2 * grid + c2]:
                    neighbours.append(colours[r2 * grid + c2])
        colour = np.mean(neighbours, axis=0) if neighbours else fallback
        out[index] = np.tile(colour, values // channels)
    return out


# ---------------------------------------------------------------------------------- classification


def _per_class(pred: np.ndarray, gold: np.ndarray, n_classes: int) -> dict[str, Any]:
    out = {}
    for c in range(n_classes):
        tp = int(np.sum((pred == c) & (gold == c)))
        fp = int(np.sum((pred == c) & (gold != c)))
        fn = int(np.sum((pred != c) & (gold == c)))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        out[c] = {"support": int(np.sum(gold == c)), "precision": precision, "recall": recall, "f1": f1}
    return out


def classification_metrics(scores: Any, gold: Sequence[int], classes: Sequence[str]) -> dict[str, Any]:
    """Accuracy, macro F1 and a per-class breakdown for an `[image][class]` score grid."""
    grid = np.asarray(scores, dtype=np.float64)
    gold_arr = np.asarray(gold, dtype=int)
    if grid.ndim != 2 or grid.shape != (len(gold_arr), len(classes)):
        raise ValueError(f"scores must be an [images x classes] grid, got {grid.shape}")
    if not len(gold_arr) or gold_arr.min() < 0 or gold_arr.max() >= len(classes):
        raise ValueError("gold must hold class indices for at least one photograph")
    if not np.all(np.isfinite(grid)):
        raise ValueError("scores must be finite")
    pred = grid.argmax(axis=1)
    per = _per_class(pred, gold_arr, len(classes))
    return {
        "n": int(len(gold_arr)),
        "accuracy": float(np.mean(pred == gold_arr)),
        "macro_f1": float(np.mean([per[c]["f1"] for c in range(len(classes))])),
        "per_class": {classes[c]: per[c] for c in range(len(classes))},
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _scores_from_predictions(pred: Sequence[int], n_classes: int) -> np.ndarray:
    grid = np.zeros((len(pred), n_classes))
    for i, c in enumerate(pred):
        grid[i, c] = 1.0
    return grid


def majority_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], classes: Sequence[str]
) -> dict[str, Any]:
    """The most frequent training label for every photograph."""
    if not train:
        raise ValueError("the majority baseline needs training records")
    counts = Counter(str(r["label"]) for r in train)
    label = max(sorted(counts), key=counts.get)
    index = list(classes).index(label)
    gold = [list(classes).index(str(r["label"])) for r in records]
    result = classification_metrics(_scores_from_predictions([index] * len(records), len(classes)), gold, classes)
    result["baseline"] = f"majority floor ({label!r} for every photograph)"
    return result


def colour_signature(image: str | Image.Image, *, grid: int = COLOUR_GRID) -> list[float]:
    """Mean RGB of each cell of a `grid` x `grid` partition of the image, in 0..1 (27 numbers by default)."""
    handle = image if isinstance(image, Image.Image) else Image.open(image)
    small = handle.convert("RGB").resize((grid * 8, grid * 8), Image.BILINEAR)
    pixels = list(small.getdata())
    out: list[float] = []
    for row in range(grid):
        for col in range(grid):
            cell = [pixels[(row * 8 + y) * grid * 8 + col * 8 + x] for y in range(8) for x in range(8)]
            out.extend(sum(p[channel] for p in cell) / (64 * 255.0) for channel in range(3))
    return out


def _distance(a: Sequence[float], b: Sequence[float]) -> float:
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b, strict=True)))


def colour_neighbour_baseline(
    train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], classes: Sequence[str]
) -> dict[str, Any]:
    """Label every photograph with the label of the training photograph whose colour signature is closest;
    the score grid is the negative distance to the nearest training photograph of each class."""
    if not train:
        raise ValueError("the colour-neighbour baseline needs training records")
    class_list = list(classes)
    signatures = [(colour_signature(r["image"]), class_list.index(str(r["label"]))) for r in train]
    grid = np.zeros((len(records), len(class_list)))
    for i, record in enumerate(records):
        query = colour_signature(record["image"])
        best = [math.inf] * len(class_list)
        for sig, c in signatures:
            best[c] = min(best[c], _distance(sig, query))
        grid[i] = [-d for d in best]
    gold = [class_list.index(str(r["label"])) for r in records]
    result = classification_metrics(grid, gold, class_list)
    result["baseline"] = f"colour nearest neighbour ({len(train)} training photographs)"
    return result


def knn_scores(train_features: np.ndarray, train_gold: Sequence[int], features: np.ndarray, n_classes: int, *, k: int = 5) -> np.ndarray:
    """Cosine k-NN vote on L2-normalised features: the score grid is the summed similarity of the k nearest
    training features per class."""
    a = np.asarray(train_features, dtype=np.float64)
    b = np.asarray(features, dtype=np.float64)
    gold = np.asarray(train_gold, dtype=int)
    if a.ndim != 2 or b.ndim != 2 or a.shape[1] != b.shape[1] or len(gold) != len(a):
        raise ValueError("feature matrices must be [images x dim] with one training label per row")
    sims = b @ a.T
    k = min(k, len(a))
    grid = np.zeros((len(b), n_classes))
    for i in range(len(b)):
        top = np.argsort(-sims[i])[:k]
        for j in top:
            grid[i, gold[j]] += sims[i, j]
    return grid

**Module 3/6:** `src/vit_mae_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

import hashlib
import json
import os
from collections.abc import Callable
from pathlib import Path
from typing import Any

import torch
from huggingface_hub import snapshot_download
from transformers import AutoImageProcessor, ViTMAEForPreTraining

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

MANIFEST_NAME = "dimer-base-manifest.json"
#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in a repository-
#: local snapshot directory named by the model key and described by the committed manifest; a
#: standalone notebook carries that manifest inline and stages/verifies a working-directory copy.
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / DEFAULT_MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory snapshot, no repository checkout


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_checkpoint(
    snapshot_path: str | Path,
    *,
    require_configs: bool = False,
    return_manifest_verified: bool = False,
) -> Path | tuple[Path, bool]:
    root = Path(snapshot_path)
    if not root.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {root}")

    weight_path = root / MODEL_FILENAME
    if not weight_path.is_file():
        raise RuntimeError(f"Pinned checkpoint is missing {MODEL_FILENAME}")

    unsafe = sorted(
        p.name
        for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS
    )
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files: {unsafe}")

    manifest_path = root / MANIFEST_NAME
    manifest_verified = False

    if manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc

        files = manifest.get("files") or []
        if not files:
            raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")

        for entry in files:
            rel_path = entry.get("path")
            if not rel_path:
                continue
            target = root / rel_path
            if not target.is_file():
                raise RuntimeError(f"Manifest file missing: {rel_path}")
            exp_bytes = entry.get("bytes")
            if exp_bytes is not None and target.stat().st_size != exp_bytes:
                raise RuntimeError(
                    f"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}"
                )
            exp_sha = entry.get("sha256")
            if exp_sha is not None and _sha256(target) != exp_sha:
                raise RuntimeError(f"SHA-256 mismatch for {rel_path}")

        manifest_verified = True

    size = weight_path.stat().st_size
    if size != MODEL_SIZE_BYTES:
        raise RuntimeError(f"Unexpected {MODEL_FILENAME} size: {size}; expected {MODEL_SIZE_BYTES}")

    digest = _sha256(weight_path)
    if digest != MODEL_SHA256:
        raise RuntimeError(
            f"Unexpected {MODEL_FILENAME} SHA-256: {digest}; expected {MODEL_SHA256}"
        )

    if require_configs:
        for req in ("config.json", "preprocessor_config.json"):
            if not (root / req).is_file():
                raise RuntimeError(f"Missing required configuration file: {req}")

    if return_manifest_verified:
        return root, manifest_verified
    return root


def _read_manifest(root: Path) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except ValueError as exc:
        raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing"
        )
    if not manifest.get("files"):
        raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of a fleet snapshot directory; raise on the first mismatch.

    The identity in the manifest must be the pinned one; every manifest entry is then size- and
    SHA-256-checked by :func:`verify_checkpoint` (the existing verifier, which also asserts the
    weight file's pinned digest and byte count and refuses unsafe formats). Returns
    ``{"path": ..., **manifest}``.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    _, manifest_verified = verify_checkpoint(
        root, require_configs=True, return_manifest_verified=True
    )
    if not manifest_verified:
        raise RuntimeError(f"manifest at {root} was not verified")  # pragma: no cover
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and
    the small files but git-ignores the weights). Returns the relative paths fetched;
    :func:`verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def resolve_weights_path(
    weights_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[Path, str]:
    """Resolve weights path with precedence:

    1. Explicit argument `weights_path` -> 'explicit_path'
    2. Environment variable `VIT_MAE_WEIGHTS_DIR` -> 'env_var'
    3. Source checkout convention `weights/vit-mae-base` -> 'repo_offline'
       (only if pyproject.toml exists at repo root and weights/ contains model.safetensors)
    4. Hugging Face Hub snapshot download -> 'hf_hub'
    """
    if weights_path is not None:
        return Path(weights_path), "explicit_path"

    env_dir = os.environ.get("VIT_MAE_WEIGHTS_DIR")
    if env_dir:
        return Path(env_dir), "env_var"

    repo_root = Path.cwd()  # standalone rewrite (build_notebook.py): no repository checkout to resolve
    if (repo_root / "pyproject.toml").is_file():
        repo_weights = repo_root / "weights" / DEFAULT_MODEL_KEY
        if (repo_weights / MODEL_FILENAME).is_file():
            return repo_weights, "repo_offline"

    hub_path = Path(
        snapshot_download(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
            allow_patterns=list(ALLOWED_CHECKPOINT_FILES),
            cache_dir=str(cache_dir) if cache_dir is not None else None,
        )
    )
    return hub_path, "hf_hub"


_resolve_weights_path = resolve_weights_path


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_components(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
    return_metadata: bool = False,
) -> tuple[Any, Any, torch.device, Path] | tuple[Any, Any, torch.device, Path, dict[str, Any]]:
    """Acquire, verify, and load the one supported ViT-MAE checkpoint (encoder + decoder, ViTMAEForPreTraining)."""

    candidate_path, source = resolve_weights_path(
        weights_path=weights_path,
        cache_dir=cache_dir,
    )

    verified, manifest_verified = verify_checkpoint(
        candidate_path,
        require_configs=True,
        return_manifest_verified=True,
    )
    target_device = _resolve_device(device)

    processor = AutoImageProcessor.from_pretrained(
        verified,
        local_files_only=True,
        trust_remote_code=False,
    )
    model = ViTMAEForPreTraining.from_pretrained(
        verified,
        local_files_only=True,
        trust_remote_code=False,
        use_safetensors=True,
    )
    model = model.eval().to(target_device)

    weight_file = verified / MODEL_FILENAME
    metadata: dict[str, Any] = {
        "checkpoint_path": verified,
        "checkpoint_source": source,
        "manifest_verified": manifest_verified,
        "weight_sha256": _sha256(weight_file),
        "weight_size_bytes": weight_file.stat().st_size,
        "device": str(target_device),
    }

    if return_metadata:
        return model, processor, target_device, verified, metadata
    return model, processor, target_device, verified

**Module 4/6:** `src/vit_mae_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

_RUNTIME_PACKAGES = ("torch", "transformers", "safetensors", "numpy", "pillow", "huggingface-hub")


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def build_provenance(
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> dict[str, Any]:
    checkpoint_source = None
    manifest_verified = False
    weight_sha256 = MODEL_SHA256
    weight_size = MODEL_SIZE_BYTES
    device_str = None
    resolved_checkpoint_path = None
    adapter: dict[str, Any] | None = None

    if pipeline is not None:
        if getattr(pipeline, "checkpoint_path", None) is not None:
            resolved_checkpoint_path = str(pipeline.checkpoint_path)
        checkpoint_source = getattr(pipeline, "checkpoint_source", None)
        manifest_verified = bool(getattr(pipeline, "manifest_verified", False))
        if getattr(pipeline, "weight_sha256", None):
            weight_sha256 = pipeline.weight_sha256
        if getattr(pipeline, "weight_size_bytes", None):
            weight_size = pipeline.weight_size_bytes
        if getattr(pipeline, "device", None) is not None:
            device_str = str(pipeline.device)
        if getattr(pipeline, "adapter", None):
            skip = ("history", "trainable_names")
            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}

    if checkpoint_path is not None:
        resolved_checkpoint_path = str(checkpoint_path)
        if checkpoint_source is None:
            checkpoint_source = "explicit_path"

    model_record: dict[str, Any] = {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "license": MODEL_LICENSE,
        "weight_file": MODEL_FILENAME,
        "weight_sha256": weight_sha256,
        "weight_size_bytes": weight_size,
        "weight_format": "safetensors",
    }
    if checkpoint_source is not None:
        model_record["checkpoint_source"] = checkpoint_source
    if resolved_checkpoint_path is not None:
        model_record["checkpoint_path"] = resolved_checkpoint_path
    if pipeline is not None or checkpoint_path is not None:
        model_record["manifest_verified"] = manifest_verified

    inference_record: dict[str, Any] = {
        "reconstruction": (
            "masked autoencoding: a seeded random subset of the patches is hidden from the encoder, the decoder "
            "predicts every patch's pixels, the loss is the mean squared error on the hidden patches only"
        ),
        "default_mask_ratio": DEFAULT_MASK_RATIO,
        "norm_pix_loss": NORM_PIX_LOSS,
        "loss_semantics": "raw-pixel MSE over the normalised (ImageNet mean / std) hidden patches; lower is better; not calibrated",
        "embedding": "mean of the encoder's patch tokens after the final LayerNorm with no patch hidden (mask ratio 0), L2-normalised",
        "embedding_dim": 768,
    }
    if device_str is not None:
        inference_record["device"] = device_str

    provenance: dict[str, Any] = {
        "schema_version": 1,
        "model": model_record,
        "processor": {
            "input_resolution": [IMAGE_SIZE, IMAGE_SIZE],
            "patch_size": PATCH_SIZE,
            "num_patches": NUM_PATCHES,
            "normalization": "ImageNet mean / std (preprocessor_config.json)",
        },
        "inference": inference_record,
        "adapter": adapter,
    }
    if include_runtime:
        provenance["runtime"] = {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "packages": {name: _package_version(name) for name in _RUNTIME_PACKAGES},
        }
    return provenance


def write_provenance(
    path: str | Path,
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> Path:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            build_provenance(pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    return target

**Module 5/6:** `src/vit_mae_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Image dataset contract for continuing the masked-autoencoding pretraining and reading the encoder through a
linear probe: the pinned iNaturalist bird sample, validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and out of the checkpoint's ImageNet-1k pretraining domain in the way that
matters here — fine-grained wildlife: 360 CC0-licensed, research-grade iNaturalist photographs of six common
North American birds (60 per species, one per observer per species; four small sparrows, a junco and two
finches), the fleet's shared sample, pinned here by photo id, byte size and SHA-256 of the served `medium`
JPEG. Every file is fetched from the iNaturalist open-data bucket at run time and refused on any byte-size or
SHA-256 mismatch; the repository redistributes none of the photographs. Each record keeps the observation id
and observer login so every image is traceable to its public observation page.

A record is ``{id, image, label}``: a PIL image (or a path to one) and the species key of its gold label —
the label is used by the linear probe only; the reconstruction contract and `adapt` never read it, and
`validate_dataset(..., require_labels=False)` accepts unlabelled ``{id, image}`` records for them. `SPECIES`
maps the key to the scientific and common names for provenance.
"""
# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import MODEL_ID` removed — names are kernel globals defined by the carried modules

MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work

CORPUS_NAME = "iNaturalist CC0 bird photographs (six species)"
CORPUS_RELEASE = (
    "iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species)"
)
CORPUS_BASE_URL = "https://inaturalist-open-data.s3.amazonaws.com/photos/"
CORPUS_LICENSE = (
    "CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records)"
)
CORPUS_BYTES = 39_223_447
DEFAULT_CACHE_DIR = Path("weights") / "inat-birds"
SPECIES: dict[str, tuple[str, str]] = {
    "song_sparrow": ("Melospiza melodia", "Song Sparrow"),
    "chipping_sparrow": ("Spizella passerina", "Chipping Sparrow"),
    "white_throated_sparrow": ("Zonotrichia albicollis", "White-throated Sparrow"),
    "dark_eyed_junco": ("Junco hyemalis", "Dark-eyed Junco"),
    "house_finch": ("Haemorhous mexicanus", "House Finch"),
    "american_goldfinch": ("Spinus tristis", "American Goldfinch"),
}
# (id, label, iNat photo id, iNat observation id, observer login, bytes, sha256 of the served
#  <photo id>/medium.<ext>, ext) — the bucket serves each photo under its original extension
#  (jpg or jpeg); the digest pins the served bytes
SAMPLE_RECORDS: tuple[tuple[str, str, int, int, str, int, str, str], ...] = (
    (
        "song_sparrow-00",
        "song_sparrow",
        129376982,
        79016324,
        "andywilson",
        43427,
        "7a9d9304a82f202e992655ec5f65477cd3d7c1dce03aa89a214c2daa38f9d61d",
        "jpg",
    ),
    (
        "song_sparrow-01",
        "song_sparrow",
        480991086,
        267636534,
        "lyneisfilm",
        162073,
        "11f77ff277dd2703c1000f2c787136ff0c3ca7ffad7017056892fe789d65efec",
        "jpg",
    ),
    (
        "song_sparrow-02",
        "song_sparrow",
        546060381,
        302980489,
        "swpollinators",
        27899,
        "4e70b9519c6e5f7384a4495b91b45465f2b1599f86491d8f9f9b12635a4046f6",
        "jpg",
    ),
    (
        "song_sparrow-03",
        "song_sparrow",
        308625896,
        177450028,
        "radrat",
        70961,
        "1211da4fdb24ae85ef0c6c3e2d03542c430457856aec661fe8f5f2de0027eee5",
        "jpeg",
    ),
    (
        "song_sparrow-04",
        "song_sparrow",
        494793016,
        275349085,
        "k-simpkins",
        58410,
        "255538cf450197257e86ed3d41dc69fb78e594434e9cb338c6314288c6cff26e",
        "jpg",
    ),
    (
        "song_sparrow-05",
        "song_sparrow",
        674054489,
        369029444,
        "ben142",
        220573,
        "1ae24622888d9d449ffd6b5c65cac1b9b14870fd8e12dbed8aa0acc2f5030123",
        "jpg",
    ),
    (
        "song_sparrow-06",
        "song_sparrow",
        339623726,
        193339933,
        "rawcomposition",
        25012,
        "d2cde085277a71886a2bf211a1eec26941a73752375708726a8af29aa4995a8b",
        "jpg",
    ),
    (
        "song_sparrow-07",
        "song_sparrow",
        181658744,
        107953669,
        "gcart043",
        98482,
        "a662a6abb24f42b256fb6e2d6f02c3e128a1053ef534f454461aa5cadcc03fe4",
        "jpeg",
    ),
    (
        "song_sparrow-08",
        "song_sparrow",
        222768957,
        130949329,
        "davidfbird",
        110773,
        "0feee62753f409d0aa365e9aa017ddef436ad70a2673dc847feb3b5386af27ff",
        "jpg",
    ),
    (
        "song_sparrow-09",
        "song_sparrow",
        148994242,
        90171417,
        "glennberry",
        102702,
        "64333977d24957d723a1d26886004f222d810557617685579f72a6b553e508fa",
        "jpg",
    ),
    (
        "song_sparrow-10",
        "song_sparrow",
        637315932,
        349374463,
        "sooji",
        136572,
        "a5cebbc0cc2325d3805c4ac854e103f7ba1c22e3a34fb204f30d58681e865033",
        "jpg",
    ),
    (
        "song_sparrow-11",
        "song_sparrow",
        471146686,
        262252507,
        "jeanpaulboerekamps",
        98601,
        "4301f06b52b8dcf1e137567c32412e384edb71cb90e2e35459a0405b2e56529b",
        "jpg",
    ),
    (
        "song_sparrow-12",
        "song_sparrow",
        640811426,
        351179648,
        "erikschiff",
        107695,
        "27aecce184a485ee888c0e8101cb4a34899b8a185b62dc064f3dc2ec9902182b",
        "jpg",
    ),
    (
        "song_sparrow-13",
        "song_sparrow",
        123859010,
        75689904,
        "w_mark_c",
        190819,
        "6b6057a1c50b83ffeb4d9e34367b3e6a9b355236dbcae9820b509e1484f8fe33",
        "jpg",
    ),
    (
        "song_sparrow-14",
        "song_sparrow",
        108520869,
        67204020,
        "dugald",
        52496,
        "e483364889fb95c540db84b5edf5a2400febf7d2eb62a1e49303b13ad329d1b8",
        "jpg",
    ),
    (
        "song_sparrow-15",
        "song_sparrow",
        63537800,
        40010230,
        "nathanael15",
        51441,
        "5ae55f868e779a4e8ee34f6ad077b40c41f20aa699d967f373ebd659a89137a8",
        "jpg",
    ),
    (
        "song_sparrow-16",
        "song_sparrow",
        435251292,
        244042351,
        "carterdorscht",
        166662,
        "d0cdd9ddcf7202a91ac2c47910a0c230639bce50283e8511fb8311151763233a",
        "jpeg",
    ),
    (
        "song_sparrow-17",
        "song_sparrow",
        120710033,
        73898230,
        "tys_rbg",
        126820,
        "515b3b32b64e88d401990bfd1d8e2c1281443b5d1e763a09e86d3d3664e1df41",
        "jpg",
    ),
    (
        "song_sparrow-18",
        "song_sparrow",
        349361283,
        198243065,
        "sean579",
        42820,
        "5eb0031a8d6066650c66b265fb1724413273e095e4e531054d2817eb75910074",
        "jpg",
    ),
    (
        "song_sparrow-19",
        "song_sparrow",
        393408927,
        222067370,
        "irenemacaulay_",
        101681,
        "22e326807de963352b4307790bd40c5506adb0fab1f9844968edaf4bc5665e1c",
        "jpg",
    ),
    (
        "song_sparrow-20",
        "song_sparrow",
        614258628,
        337847585,
        "joy4birds",
        70552,
        "aa767aa5a74a76cfd985aba5282e589f670a9ce0a8fcdc6a096c0357992dec67",
        "jpg",
    ),
    (
        "song_sparrow-21",
        "song_sparrow",
        608802621,
        335154467,
        "jamesadney",
        112564,
        "6e94bce1b5135f48b0b19b64e21a0c76cff4f9b9a28fcbc2b7f5c8ddda6ef513",
        "jpg",
    ),
    (
        "song_sparrow-22",
        "song_sparrow",
        634100352,
        347744524,
        "zorthesosen",
        214044,
        "6a70dba26dfb3e98a244a9ec9badb812ddaa8b3612e7a36f66aba057bb757ecd",
        "jpg",
    ),
    (
        "song_sparrow-23",
        "song_sparrow",
        50065474,
        31954532,
        "truthseqr",
        82521,
        "4186fbf344e92038358d4338102aa440098bff5f558ab1d197f72fefbcec0b15",
        "jpeg",
    ),
    (
        "song_sparrow-24",
        "song_sparrow",
        8656044,
        6803564,
        "glmory",
        297964,
        "e8cab773436ccfaad4699e5112ef4edb516236d413d6dbf034eea7bf9c88c8f5",
        "jpg",
    ),
    (
        "song_sparrow-25",
        "song_sparrow",
        131051149,
        79988590,
        "funvill",
        72572,
        "0a694b5bb03aee6eb5a3a6132e855d47172b21b84cc4cc7c64367d1a31de7894",
        "jpeg",
    ),
    (
        "song_sparrow-26",
        "song_sparrow",
        12923156,
        9491600,
        "gambolingquail",
        69559,
        "6a92bff4fc76820c6f21e15c5f254385dd976a32264911bfc08417a1c2bf053d",
        "jpg",
    ),
    (
        "song_sparrow-27",
        "song_sparrow",
        12077500,
        8959545,
        "reuvenm",
        74010,
        "3e3c5e94c839f45610ef3ef7ffaf3575f4bfe365fc94454c30bfdb718ca3c593",
        "jpeg",
    ),
    (
        "song_sparrow-28",
        "song_sparrow",
        132730299,
        80955309,
        "steph123456",
        154309,
        "5f8854dd231a302c643b22521b49ed3007203d74b8b8413a23abc79ffe0b0270",
        "jpeg",
    ),
    (
        "song_sparrow-29",
        "song_sparrow",
        124840110,
        76316566,
        "terrimewbornagain",
        81046,
        "2020092b0b67397a78cc2b0df267fb671b3ba18e03fb0e35e664e0784061e3c6",
        "jpeg",
    ),
    (
        "song_sparrow-30",
        "song_sparrow",
        130188079,
        79482300,
        "fake_id",
        180462,
        "1d9ad7b45536b85d46fa0fd9a29d4a8ae828bc223836796a3eb9df1fe8fe4fc4",
        "jpeg",
    ),
    (
        "song_sparrow-31",
        "song_sparrow",
        136405967,
        83075831,
        "tom_lazar",
        178554,
        "f86c3c5899d20ac8fb73afb76d45666c610245636d7285a6d517b0081b9d6cc9",
        "jpeg",
    ),
    (
        "song_sparrow-32",
        "song_sparrow",
        118885385,
        72867352,
        "ellyne",
        105156,
        "aab53fb1e7c12adc697711215778c81983f51af83e20bfffe841b80c1d144673",
        "jpeg",
    ),
    (
        "song_sparrow-33",
        "song_sparrow",
        676467457,
        370293984,
        "sholsenbeck",
        59104,
        "b7c230ca942cedc1c53f42902e1f70bc2d09497e57d2e9f31dfe72a5f6c84d16",
        "jpg",
    ),
    (
        "song_sparrow-34",
        "song_sparrow",
        688316087,
        376461614,
        "doublecritch",
        109416,
        "7f11c69df733f99d6e8663e91d807327f8cc57f404049fdbe95e2fd1ce1066e1",
        "jpg",
    ),
    (
        "song_sparrow-35",
        "song_sparrow",
        691785499,
        378254663,
        "karuquebec",
        144522,
        "9862b079bcf496aa8d9ce9e716e65b03a3d9a70a5c2077a223630cf9dcc6d5cc",
        "jpg",
    ),
    (
        "song_sparrow-36",
        "song_sparrow",
        691561091,
        378140427,
        "vaughnshirey",
        87827,
        "a2c197f91c0543ec7bd3a9aac4867a34417b2eb2444bf3062060f9f1e3580045",
        "jpg",
    ),
    (
        "song_sparrow-37",
        "song_sparrow",
        582182748,
        321880490,
        "dougbrown",
        44795,
        "465ce1214c40efab6afdddfbfd34fc5076e5640d130b0442e8e8ac2731ec8a1f",
        "jpg",
    ),
    (
        "song_sparrow-38",
        "song_sparrow",
        704199014,
        384692464,
        "enspring",
        85269,
        "1aa25fd1ed376b9245127432925832cf9253e5d156f4f7d35298eff9a8002043",
        "jpg",
    ),
    (
        "song_sparrow-39",
        "song_sparrow",
        583754170,
        322666295,
        "rociherrera",
        97743,
        "5e093fc6d59b13bbd7ee0889613cef115ef7bb14432634d0af753a8124951737",
        "jpg",
    ),
    (
        "song_sparrow-40",
        "song_sparrow",
        590270305,
        325961498,
        "damienxw",
        133375,
        "a12a152c94ef9c27e5f6dc557e74bf57f6c9ca3015a5e28ee9c30a27ed0e7e19",
        "jpg",
    ),
    (
        "song_sparrow-41",
        "song_sparrow",
        423534602,
        237940479,
        "don54",
        25229,
        "69b0f0dfb05e8219700e3878c9b7e4a2d43d0a2df3905414630913397a565dea",
        "jpeg",
    ),
    (
        "song_sparrow-42",
        "song_sparrow",
        418537114,
        235351206,
        "memoosborne",
        95540,
        "3432cf5a95939b898989c5a962c66b2d71b92b17195d5e695020a776361af3fa",
        "jpeg",
    ),
    (
        "song_sparrow-43",
        "song_sparrow",
        280724703,
        162358184,
        "shirleymorrison",
        43015,
        "f64af9b3fc4559f251ba0f1e95bf8889bfa629123586532adeb2b81e4c2871b2",
        "jpg",
    ),
    (
        "song_sparrow-44",
        "song_sparrow",
        641009256,
        351278414,
        "robinlanark",
        49640,
        "e4af33fd4a504a876b6e7a111a5024cf4197fc888c90f2039df63e9be13d5f06",
        "jpg",
    ),
    (
        "song_sparrow-45",
        "song_sparrow",
        531314531,
        295151228,
        "steveplumb",
        128710,
        "8806861909c0833b29f43b10e9428e104deaccb99d41b42039b76f7da0ae4600",
        "jpg",
    ),
    (
        "song_sparrow-46",
        "song_sparrow",
        114487462,
        70425556,
        "leahmfulton",
        50091,
        "d0b3c9408b72514f8485b19f61116b94be002be99368db96e126d625a17bd343",
        "jpg",
    ),
    (
        "song_sparrow-47",
        "song_sparrow",
        118804811,
        72823547,
        "heibudas",
        191643,
        "8dbe5a56d5fea83b6c25257aa0f9e552ea971f1e0311f583691d24fc9668556f",
        "jpeg",
    ),
    (
        "song_sparrow-48",
        "song_sparrow",
        118498513,
        72671334,
        "seanwashington1",
        80800,
        "940762ff238d94815922d06051ca00fe4a025279b1da08921a44733084adcdb6",
        "jpeg",
    ),
    (
        "song_sparrow-49",
        "song_sparrow",
        119696805,
        73327300,
        "rambryum",
        68983,
        "65f7007d6b90654b77b7f4ec0557ff2d0f310c572368cb3492dd69f9d082e90d",
        "jpeg",
    ),
    (
        "song_sparrow-50",
        "song_sparrow",
        4251416,
        3670620,
        "swells",
        49271,
        "8c53fa5c02e603227117293720d269c59a13555abd537a83640210805fbd844d",
        "JPG",
    ),
    (
        "song_sparrow-51",
        "song_sparrow",
        303954368,
        174929167,
        "dianeclark6280",
        157478,
        "14446920b7c55472de3b172f417fbbeac312090add7356a3c947cfb16911dc95",
        "png",
    ),
    (
        "song_sparrow-52",
        "song_sparrow",
        392544542,
        221653008,
        "emckenziewhat",
        138683,
        "7fd2f3ad3c835e541a7c720372d662a8f953f88f4d8223864a03e4b0f70a42d3",
        "jpeg",
    ),
    (
        "song_sparrow-53",
        "song_sparrow",
        270614122,
        156495271,
        "radkins21",
        92783,
        "7bf8ceb5944de93cbf01aebe4d464b9acb848569c07fbe6e732b701d309f0b25",
        "jpg",
    ),
    (
        "song_sparrow-54",
        "song_sparrow",
        139634990,
        84920252,
        "raffib128",
        56293,
        "865cf92a61d89cb3420c9d3cbf585a0724df2e314918b05d705804aee356f1a5",
        "jpeg",
    ),
    (
        "song_sparrow-55",
        "song_sparrow",
        25251350,
        16733911,
        "kemper",
        54034,
        "99f19fa1a37d23712538e43f6a82339f1fd37defe283de3baacc17719f05857e",
        "jpeg",
    ),
    (
        "song_sparrow-56",
        "song_sparrow",
        139129879,
        84639025,
        "wewantashrubbery",
        104736,
        "aabbffe279d90ffdf737175cc720bea92d634483b830c90acecf920901419643",
        "jpg",
    ),
    (
        "song_sparrow-57",
        "song_sparrow",
        128242284,
        78367372,
        "stevestevens",
        112226,
        "918dfc8094d0a697beb1dcd04499cfd7b0c4b12f9dac4d915d1566ab5690b29b",
        "jpeg",
    ),
    (
        "song_sparrow-58",
        "song_sparrow",
        478733235,
        266446909,
        "saintaardvark",
        298253,
        "d3c8d3a58b44d28c6af4748e4e4195a9a2af7dc6fe91541429dc44aa3a659d21",
        "jpg",
    ),
    (
        "song_sparrow-59",
        "song_sparrow",
        623006598,
        342147975,
        "ryman56",
        60213,
        "73e18df350a69ce00158f03c7ec4fda8672539e0a7292b75f1dfa22dbec7ee30",
        "jpg",
    ),
    (
        "chipping_sparrow-00",
        "chipping_sparrow",
        198992636,
        117809422,
        "k-simpkins",
        62563,
        "cf9f3b0c1863808e21af596b2e609b047ddbc28cb2ed076625e2546425ff0adf",
        "jpg",
    ),
    (
        "chipping_sparrow-01",
        "chipping_sparrow",
        248210057,
        144599194,
        "w_mark_c",
        193389,
        "497d0a0fef81c326bcc87b5d1eb97fe987d559b8f2b71bb60fe422ae34dce150",
        "jpg",
    ),
    (
        "chipping_sparrow-02",
        "chipping_sparrow",
        156350853,
        94266719,
        "ellyne",
        142332,
        "6a60ebac34476a372b4790a87d823432cdda8f72590930b5d18ae166cf4c7ba2",
        "jpeg",
    ),
    (
        "chipping_sparrow-03",
        "chipping_sparrow",
        16128796,
        11327134,
        "reuvenm",
        70532,
        "5ad36c9cdd6c92e225a1b8ab3c04d2f65bc4e971d0243958f8ac090b0996115c",
        "jpeg",
    ),
    (
        "chipping_sparrow-04",
        "chipping_sparrow",
        391300648,
        220982684,
        "carterdorscht",
        208567,
        "c974a676c3c227c2844ff822d429c784bc87bb41f40d5074a0ebadd4c6785b21",
        "jpeg",
    ),
    (
        "chipping_sparrow-05",
        "chipping_sparrow",
        339456083,
        193252042,
        "rawcomposition",
        46329,
        "41c9260ca9107430e3a8090ba01cebf3e3c25f2dad15bea4f77c8e824d9fc496",
        "jpg",
    ),
    (
        "chipping_sparrow-06",
        "chipping_sparrow",
        40457652,
        26076708,
        "andywilson",
        156894,
        "b70edd35e00fe672a39d5f0441ea4e9bb9fac19b0f2415ae13e22160bc6091a6",
        "jpeg",
    ),
    (
        "chipping_sparrow-07",
        "chipping_sparrow",
        84151914,
        52921135,
        "davidfbird",
        145714,
        "ea65ce5ed881ded9a8157b5756fa907948f41e0eefb6735b48a1c43993e977f9",
        "jpeg",
    ),
    (
        "chipping_sparrow-08",
        "chipping_sparrow",
        523674610,
        291074747,
        "rwp84",
        47983,
        "41eac0a5fb578b089f7524c4d2e6cb38c5508aa81815d6cfc46948c081daa800",
        "jpg",
    ),
    (
        "chipping_sparrow-09",
        "chipping_sparrow",
        292695018,
        168861389,
        "tim_kirsten",
        64812,
        "cbb2a03f5dbd2209d56f1cca8b49f342dfbaf44e51fe1014ead75b2018c6678d",
        "jpeg",
    ),
    (
        "chipping_sparrow-10",
        "chipping_sparrow",
        92501489,
        57964052,
        "tniernberger",
        78380,
        "44ddb9923026a97ee77ed362bc944c3d2cbbe4fbd4636000880e81613634e6c5",
        "jpg",
    ),
    (
        "chipping_sparrow-11",
        "chipping_sparrow",
        220257388,
        129611350,
        "gcart043",
        129377,
        "1bdce32e7ac58321dd6beacc084afaf45973469215d66b61a0f7c612da3db361",
        "jpeg",
    ),
    (
        "chipping_sparrow-12",
        "chipping_sparrow",
        478480946,
        266314208,
        "russnamitz",
        61359,
        "8200cc2ec779d47be1b5afa261d934910a251a39f5620d3dc705106fb2bf5e0b",
        "jpeg",
    ),
    (
        "chipping_sparrow-13",
        "chipping_sparrow",
        538480223,
        298914072,
        "hiltonward",
        201271,
        "2a6479556f14a20a8c9a69ff0c1026fe4deb376944234d1d4429c558042c4346",
        "jpg",
    ),
    (
        "chipping_sparrow-14",
        "chipping_sparrow",
        58192408,
        36778771,
        "bradenjudson",
        22352,
        "7c630d7a5b24d94e677e563a8ecb36f57fb29babe6ac1568f11537f10a5edf75",
        "jpeg",
    ),
    (
        "chipping_sparrow-15",
        "chipping_sparrow",
        80410971,
        50636049,
        "radrat",
        55646,
        "0c71ca735502e8c94db81302ecd006428206eb16d75472eb0c706e62e2656667",
        "jpg",
    ),
    (
        "chipping_sparrow-16",
        "chipping_sparrow",
        300376697,
        172991803,
        "matthias55",
        81986,
        "7a9a5cbfb6a7d0581c98c75b5fdeb53a049d60f352eda43a39b1f9c2a3347f3b",
        "jpeg",
    ),
    (
        "chipping_sparrow-17",
        "chipping_sparrow",
        370917657,
        209626382,
        "craigmartin",
        101736,
        "feb94d319fe5b12c01e63a80dc8e44c45e15a0bfb6534ac5d2c0641205dfe1d5",
        "jpg",
    ),
    (
        "chipping_sparrow-18",
        "chipping_sparrow",
        264119989,
        152960401,
        "laurelthrone",
        194107,
        "13e7336628f9ad784757dca82557c79eda22fdce561e0b52a10a503e72979b8e",
        "jpeg",
    ),
    (
        "chipping_sparrow-19",
        "chipping_sparrow",
        220323755,
        129631264,
        "enspring",
        85687,
        "886b7e6faa39943f0c9754e4aef637e2a2f5ad57fe8e8c7ec08360ea95af45b1",
        "jpeg",
    ),
    (
        "chipping_sparrow-20",
        "chipping_sparrow",
        210319996,
        124105091,
        "bunnymom20",
        188328,
        "2f6b6f7ac5ed9d91a361c8fed102484f4fca599ec39c9a2bfe9af1df53b28899",
        "jpeg",
    ),
    (
        "chipping_sparrow-21",
        "chipping_sparrow",
        117013164,
        71832078,
        "terrimewbornagain",
        49527,
        "4456adb52cdafdefdaeb6dd8cec5955f93c1b8f40abdbb1182b2f605a0e1d486",
        "jpeg",
    ),
    (
        "chipping_sparrow-22",
        "chipping_sparrow",
        660951395,
        362143224,
        "mike_cove",
        148079,
        "41d85eaa47b3a41a97ada54728a84c0bfbfed3bcff92de3b6ba88d4c9aface48",
        "jpg",
    ),
    (
        "chipping_sparrow-23",
        "chipping_sparrow",
        691160217,
        377927837,
        "ben142",
        235922,
        "f8137e51e90f0a1964c64fac1a775beaa9ae96c90a1d9aad4f1c870f143e34fd",
        "jpg",
    ),
    (
        "chipping_sparrow-24",
        "chipping_sparrow",
        707633397,
        386482138,
        "leannestacy",
        238800,
        "3fef93babe26eed119ab3b2363902007c46a58a02f91beb0d3bc650821f0dc28",
        "jpg",
    ),
    (
        "chipping_sparrow-25",
        "chipping_sparrow",
        696064427,
        380468188,
        "rdnwoods",
        309547,
        "b56f8f043112021367cbb02861021ac2ef6d2831359315f49d1445bf92477a07",
        "jpg",
    ),
    (
        "chipping_sparrow-26",
        "chipping_sparrow",
        702557361,
        383835541,
        "seanwashington1",
        159095,
        "2940c235b38f84101504b6f50e59c359d0f13e3dc384cadbd43976d1e5e9de91",
        "jpg",
    ),
    (
        "chipping_sparrow-27",
        "chipping_sparrow",
        660337945,
        361814902,
        "kristinpiston",
        158651,
        "a2272e4168edf260001ace410774fbe874ea6b1f6632b8502efc1bc02ae0f866",
        "jpg",
    ),
    (
        "chipping_sparrow-28",
        "chipping_sparrow",
        546647394,
        303298450,
        "kzoebel",
        86457,
        "c37a4aff4263de570ca82f94de92f82d9f9711922e5880547af836104f1883df",
        "jpg",
    ),
    (
        "chipping_sparrow-29",
        "chipping_sparrow",
        713495256,
        389515472,
        "lina47741",
        287986,
        "52e8a37dca8d985a57fdd0d90205e1d813388d961dbf4280d5cbccca9c6af5d3",
        "jpg",
    ),
    (
        "chipping_sparrow-30",
        "chipping_sparrow",
        269951645,
        156136769,
        "conhawn",
        106540,
        "11b48d8bf97084d9ff57aa69517b61539f03b72525022930192330fcb05a160c",
        "jpeg",
    ),
    (
        "chipping_sparrow-31",
        "chipping_sparrow",
        535714979,
        297462154,
        "dissectedfrog",
        200498,
        "cd2f2e1be8c76240a1eeff9535e9a60b8f4fb9d771c7e487d65cde48b7e24cf6",
        "jpg",
    ),
    (
        "chipping_sparrow-32",
        "chipping_sparrow",
        557301829,
        308874987,
        "mar_y_sierra_silvestre",
        115174,
        "ce58e87584df623a86643705b08750228903705ca9b02d93b0206401a24eee9e",
        "jpg",
    ),
    (
        "chipping_sparrow-33",
        "chipping_sparrow",
        652401709,
        357680572,
        "dianeclark6280",
        46782,
        "b77c0eea6899d49f8ab97fd1ad6fc772b6055e5195628fa9f6f989e2d51cedd1",
        "jpg",
    ),
    (
        "chipping_sparrow-34",
        "chipping_sparrow",
        657945260,
        360569392,
        "jasonleduc",
        36363,
        "e3471104742fa68b9eeb149255b0136c53306df23d14f2b2633cea80c9e46eea",
        "jpg",
    ),
    (
        "chipping_sparrow-35",
        "chipping_sparrow",
        434553038,
        243681621,
        "stariplativky",
        57489,
        "e253bd9d70dea9d0438b795f6bebcbc2b26c48b16a90729f0d3155695fa812fd",
        "jpg",
    ),
    (
        "chipping_sparrow-36",
        "chipping_sparrow",
        207498409,
        122551474,
        "askalotl",
        98446,
        "d84e28017be8142c22758e59f14b8c52990e79584a68e6f929123aab4997ac68",
        "jpeg",
    ),
    (
        "chipping_sparrow-37",
        "chipping_sparrow",
        179570776,
        106840941,
        "artemis224",
        138087,
        "74cad843089d2aada7a124f9e704ae1c1419b7cfcade7e2ae29e2c4eda0a6d36",
        "jpg",
    ),
    (
        "chipping_sparrow-38",
        "chipping_sparrow",
        81287617,
        51172999,
        "thyg",
        55659,
        "e1f075addb4bd1feea3ab256e37945a464d2e8c77768b7be131d0655382268d7",
        "jpg",
    ),
    (
        "chipping_sparrow-39",
        "chipping_sparrow",
        343116928,
        195090930,
        "umamimomma",
        52983,
        "a8948a0189d19b3d7b8df65271f4844b14bd5118f510d0b9ef458c942ac41ce2",
        "jpg",
    ),
    (
        "chipping_sparrow-40",
        "chipping_sparrow",
        480169680,
        267207764,
        "cvharris",
        144339,
        "d1ab8643c48f2ea823f9b61843016795def21b290c35ce3d6045e4933b5dfa0a",
        "jpeg",
    ),
    (
        "chipping_sparrow-41",
        "chipping_sparrow",
        352953833,
        200088501,
        "aster-asti",
        105786,
        "82c35e04e4e22fe35c9b364bce7737fa5226cb5c469c761adfc61ef0956da9db",
        "jpg",
    ),
    (
        "chipping_sparrow-42",
        "chipping_sparrow",
        622149696,
        341721858,
        "wilderbombyx",
        293652,
        "227c39a25941a81e9551b5ccef39f95d320e7daa02c96de247123ba62f7b9824",
        "jpg",
    ),
    (
        "chipping_sparrow-43",
        "chipping_sparrow",
        503555630,
        280377982,
        "jtdavis05",
        42352,
        "24ccb474659505cd785502295f385ec369a36c6248296aff85261ed3475a4372",
        "jpeg",
    ),
    (
        "chipping_sparrow-44",
        "chipping_sparrow",
        509135437,
        283351340,
        "martyndrabik",
        125948,
        "1604bf4f5512207c59637a84c78842285c40f76a4cf5335bf6d39afe896ed03c",
        "jpg",
    ),
    (
        "chipping_sparrow-45",
        "chipping_sparrow",
        8076820,
        6402608,
        "msieges",
        166588,
        "6c379f94141a0bcdeb8c9743ea140ded50f8c7864baecd6cef0ee1c724b9fd80",
        "jpeg",
    ),
    (
        "chipping_sparrow-46",
        "chipping_sparrow",
        131917390,
        80486493,
        "tom_lazar",
        137355,
        "10ab0a9c5af2ca2203a18e4e3df799dcf129494d7b637a3c46ffdcf0f3b3856c",
        "jpeg",
    ),
    (
        "chipping_sparrow-47",
        "chipping_sparrow",
        141348323,
        85865218,
        "j-dehoog",
        71473,
        "d3fae9ca7f515c43c216d749f085231d19ddfd101bd37cc4ffae25fc946a34fb",
        "jpeg",
    ),
    (
        "chipping_sparrow-48",
        "chipping_sparrow",
        28879424,
        18850970,
        "andy71",
        194983,
        "ea5af97ddef4f743596c176653c86e457f6bddde65d86f5ab16ddff939574683",
        "jpeg",
    ),
    (
        "chipping_sparrow-49",
        "chipping_sparrow",
        148831027,
        90084486,
        "ian-wolfe",
        181942,
        "cfab51f0c0598120f5312b354ae249bafb15124dd74723a8c29cdd7319206e49",
        "jpg",
    ),
    (
        "chipping_sparrow-50",
        "chipping_sparrow",
        641370647,
        351455126,
        "hrachski",
        130262,
        "9bc25019bf6a71d73daac63b810d18039d360887504014525598cffce2035908",
        "jpg",
    ),
    (
        "chipping_sparrow-51",
        "chipping_sparrow",
        648268506,
        355462611,
        "potatocthulhu",
        193138,
        "f526135f7b71e0f9f8d80c17d8e6c1a3cd89ba068333b9104e66d02ced7a7eaa",
        "jpg",
    ),
    (
        "chipping_sparrow-52",
        "chipping_sparrow",
        645412478,
        353758550,
        "ctlqh",
        154844,
        "e34228baa058ddb18c6aacf193c20574dab67d8c1f4d6940eec6e1acb03d993b",
        "jpg",
    ),
    (
        "chipping_sparrow-53",
        "chipping_sparrow",
        121777317,
        74504842,
        "hickl",
        48576,
        "a7f0980bf777cacb9f61855d157b94d895f6dd629485b2a038ce20519c53856c",
        "jpg",
    ),
    (
        "chipping_sparrow-54",
        "chipping_sparrow",
        147992761,
        89606238,
        "benkeen",
        461732,
        "8b6e8f03650c270624ebf82f85dd0b37db3e693157d4d3d7a1787c855846da9d",
        "png",
    ),
    (
        "chipping_sparrow-55",
        "chipping_sparrow",
        27976676,
        18310312,
        "schoenitz",
        66424,
        "64a0186d15abaf1537a34bd329b67625f7580b7aa544dded6bc626dd389b72fe",
        "jpg",
    ),
    (
        "chipping_sparrow-56",
        "chipping_sparrow",
        190706448,
        112884695,
        "jbeusmans",
        109296,
        "1777945eeb6f916ec6f9c0dc10c81014e8dc0ba22d87268e7c7dc1173193b1e6",
        "jpeg",
    ),
    (
        "chipping_sparrow-57",
        "chipping_sparrow",
        184174664,
        109291439,
        "chrismcv",
        91566,
        "c5d9a1dbaf1fad839680c417c7be7d4a680396e38e7c2aba0ee78fd213f7d32c",
        "jpeg",
    ),
    (
        "chipping_sparrow-58",
        "chipping_sparrow",
        72765715,
        45879059,
        "henryfrye",
        146855,
        "f0b957698f5a600e3a6c45471848013f9c74d1602895e40d513e2116a9b7320a",
        "jpg",
    ),
    (
        "chipping_sparrow-59",
        "chipping_sparrow",
        74100878,
        46721618,
        "aredmer",
        132045,
        "ce6f4d4324d684cfc0f40d6fe9c4a2b03a7024080e7448083ae7e3fc0c06ac6e",
        "jpg",
    ),
    (
        "white_throated_sparrow-00",
        "white_throated_sparrow",
        339621218,
        193338380,
        "rawcomposition",
        31152,
        "d1c08bfaca721bf0873437455b4cc010c6860d08b4777136c775007b0b9d07b6",
        "jpg",
    ),
    (
        "white_throated_sparrow-01",
        "white_throated_sparrow",
        166821399,
        99992799,
        "dziakj1",
        125954,
        "c595a41fbc8948b0d918b59117340dc320e2ba80d29e92bb9dacaaed5b404852",
        "jpeg",
    ),
    (
        "white_throated_sparrow-02",
        "white_throated_sparrow",
        469820434,
        261505977,
        "joy4birds",
        111767,
        "7ce091492c73c68395667bb45578dfff11457511b0958d1d0d320d1df3e55ceb",
        "jpg",
    ),
    (
        "white_throated_sparrow-03",
        "white_throated_sparrow",
        260413022,
        150969515,
        "andywilson",
        45958,
        "fb7c533c92239775da6a5b353ff398f6bdd8f65c13445fc1dd72986af5a46466",
        "jpeg",
    ),
    (
        "white_throated_sparrow-04",
        "white_throated_sparrow",
        99351488,
        62040646,
        "bradenjudson",
        23399,
        "e103968e2a6c9efb6f0bcb548a6457aef950a4a720838a624b6796b90411538e",
        "jpeg",
    ),
    (
        "white_throated_sparrow-05",
        "white_throated_sparrow",
        628148203,
        344731686,
        "lavenderdame",
        106872,
        "36379abf3af51d865ba6e6804ba3dd48fc9efd12ecc0b7d97e03fc04a16178a8",
        "jpg",
    ),
    (
        "white_throated_sparrow-06",
        "white_throated_sparrow",
        104660609,
        65043951,
        "allan7",
        42443,
        "10791891945e83a0c908c14fea07a257a0f25f51c0e708bee35184213833a635",
        "jpeg",
    ),
    (
        "white_throated_sparrow-07",
        "white_throated_sparrow",
        250718938,
        145903421,
        "stevestevens",
        138735,
        "bd52e6d2f48c247f72db0fa393ec4fa13af8510c95f0ca9134cbef71caddb6d4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-08",
        "white_throated_sparrow",
        15105971,
        10793852,
        "schylerbrown",
        31467,
        "6496e7e6d3abf13b1a538f769e3cc402280cf1e9a2a1ebdf81c68dcfd7ed01e2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-09",
        "white_throated_sparrow",
        171460784,
        102554447,
        "w_mark_c",
        251287,
        "3828c41af9209b408fe0d8ec6541edf35d13fc730b74fd27a82980d4f3771137",
        "jpg",
    ),
    (
        "white_throated_sparrow-10",
        "white_throated_sparrow",
        244260317,
        142471526,
        "deejay",
        75694,
        "d67efb1ec61f6700b8a6c6552e2da9cd981e68ae81af16d6f1e0ef17fcaff84f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-11",
        "white_throated_sparrow",
        194732675,
        115373159,
        "wildreturn",
        128989,
        "7ed4bde480b35734576bb4c5f9d1453e77c1f095380432b43b1b682050255831",
        "jpeg",
    ),
    (
        "white_throated_sparrow-12",
        "white_throated_sparrow",
        341990462,
        194541980,
        "ethologist",
        149260,
        "3e2710fac082cc347e6cd114d42d39f25ec47b82c25936922232eba916808cdb",
        "jpg",
    ),
    (
        "white_throated_sparrow-13",
        "white_throated_sparrow",
        267625208,
        154862375,
        "laurelthrone",
        148616,
        "0cac1bc7053c891ce5ae1c33e3f94b69f1b19b958bd08c686db2ccaa7ba0fbd1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-14",
        "white_throated_sparrow",
        330539586,
        188793537,
        "efalquet",
        119214,
        "c3db4583124472b28dbfb4c6fd9b3829e451d508a6b0a095eccc28f17e7982b4",
        "jpeg",
    ),
    (
        "white_throated_sparrow-15",
        "white_throated_sparrow",
        193091403,
        114346779,
        "ian-wolfe",
        73751,
        "e28974c8782fccb00f5ea5420140e01248ead859f151563a95a26bf67acba479",
        "jpg",
    ),
    (
        "white_throated_sparrow-16",
        "white_throated_sparrow",
        691050773,
        377873006,
        "dinomariobob",
        167744,
        "cc5e25afa5041ce2d6ea4e0d726793843f3a867f30b8d9ccf55892d6617da887",
        "jpg",
    ),
    (
        "white_throated_sparrow-17",
        "white_throated_sparrow",
        440986574,
        246671481,
        "suzannehale",
        141857,
        "4a354c189225de2e7b4e94a6df9cbd3a671dac0c8a7d2d4c75e933f93d8b83bf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-18",
        "white_throated_sparrow",
        113217820,
        69733707,
        "kemper",
        97802,
        "4de7da3ac53a086f2c343555be5a2b62a683715db9f80636a76673cc380fa7f6",
        "jpeg",
    ),
    (
        "white_throated_sparrow-19",
        "white_throated_sparrow",
        575307217,
        318327478,
        "k-simpkins",
        90416,
        "af2b4e72a098bd90b920c8a49632e1bbff18b73954d8ac541ca81ed4baf0530b",
        "jpg",
    ),
    (
        "white_throated_sparrow-20",
        "white_throated_sparrow",
        340420076,
        193765555,
        "don54",
        62714,
        "7f6eea434fc700166af1d343951d15f3a9c83eff06cfb0518c3c4291019a5556",
        "jpeg",
    ),
    (
        "white_throated_sparrow-21",
        "white_throated_sparrow",
        562344160,
        311510652,
        "rrfc",
        45842,
        "f7c6394649765db6e3313a03ae013291db1ae20d89d3b8f0bdf6641757f31ed0",
        "jpg",
    ),
    (
        "white_throated_sparrow-22",
        "white_throated_sparrow",
        74598266,
        47039878,
        "ianrwhyte",
        141502,
        "b0633999572a4499a80310955ab928a86f4fe08774e669b4c4949cd26254428b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-23",
        "white_throated_sparrow",
        588001234,
        324825124,
        "portablecity",
        370187,
        "11e3a97f43de5d61266d15028fe9023eef3900cfea2d027bd94ad847ecba9607",
        "jpg",
    ),
    (
        "white_throated_sparrow-24",
        "white_throated_sparrow",
        694103481,
        379467387,
        "memoosborne",
        120656,
        "82684f82f62f6e036e206eec315fff8e1ba36d308b667eba06750b75d1fada3a",
        "jpg",
    ),
    (
        "white_throated_sparrow-25",
        "white_throated_sparrow",
        655749108,
        359408087,
        "russnamitz",
        58449,
        "2a7d2b4226f1941d0950adc04bfd8fe62345f1138c521d0f5f0adf8bc5838da1",
        "jpg",
    ),
    (
        "white_throated_sparrow-26",
        "white_throated_sparrow",
        599110899,
        330395670,
        "jd_flores",
        121343,
        "5eeef4f5533a4b0a0222dcf7be000a2b34f3331f3be2b939703155ba8c8ba3ae",
        "jpg",
    ),
    (
        "white_throated_sparrow-27",
        "white_throated_sparrow",
        653888173,
        358443775,
        "toknowtheland",
        144197,
        "20a4933b00967fe4488296b2b6e89c12ecbcca0e40da591bd95927cca46ae2f7",
        "jpg",
    ),
    (
        "white_throated_sparrow-28",
        "white_throated_sparrow",
        295037357,
        170132018,
        "sturuss",
        117077,
        "d309f75ad5f4008c906ba7b3d6138aa017f919a0a17a1d5b431fa7ab47d5f2d3",
        "jpg",
    ),
    (
        "white_throated_sparrow-29",
        "white_throated_sparrow",
        405042885,
        228270930,
        "carterdorscht",
        122157,
        "f1fb32bcfb78f66f50ccd20f2418832afe72c7a5847bdeb8e4dc19546455cfaf",
        "jpeg",
    ),
    (
        "white_throated_sparrow-30",
        "white_throated_sparrow",
        84971412,
        53425343,
        "davidfbird",
        168442,
        "95c5363704d17d6076f3c7a4c9f5e41d235edf7d5dc9d6022116774f2fd544e6",
        "jpg",
    ),
    (
        "white_throated_sparrow-31",
        "white_throated_sparrow",
        476335868,
        265210970,
        "aster-asti",
        91852,
        "cd300cbc823e8ebc0902b7bd80c99eb323e5900bbabf63303b0d9399624c3e54",
        "jpg",
    ),
    (
        "white_throated_sparrow-32",
        "white_throated_sparrow",
        481170792,
        267732071,
        "kcthetc1",
        46021,
        "2194a78339aae99549df56d2d3d7fc91e0bf98a371e7d0da7958b4312fa3b487",
        "jpeg",
    ),
    (
        "white_throated_sparrow-33",
        "white_throated_sparrow",
        623602892,
        342440773,
        "imperialwoodpecker26",
        112678,
        "66653f47ce6a54d38cc1f54b0a85e965f5c5ee7cce65b02c78c1c185cb586c85",
        "jpg",
    ),
    (
        "white_throated_sparrow-34",
        "white_throated_sparrow",
        622732944,
        342010560,
        "tom_lazar",
        34257,
        "584070c4b5231db49087d1507aabec8f51adc4e09dcdda8b93b3a78159d47146",
        "jpg",
    ),
    (
        "white_throated_sparrow-35",
        "white_throated_sparrow",
        505347090,
        281327144,
        "erikschiff",
        85652,
        "91d50bda6afc3cd5596bf4d3176999bf473c31fb6680b64445ec1cbeaabea847",
        "jpg",
    ),
    (
        "white_throated_sparrow-36",
        "white_throated_sparrow",
        7072170,
        5706496,
        "jmutter",
        171582,
        "de90ea2739e8c65ca89f0e511abaf44219501e18377eb49d9e5c7f10d3a6d5b2",
        "jpeg",
    ),
    (
        "white_throated_sparrow-37",
        "white_throated_sparrow",
        16303076,
        11431253,
        "robw",
        61348,
        "e7d81cfe027c7b0914c79a0431f0ce1431db1ebe78368a2fe5f31e2aab59c3c7",
        "jpg",
    ),
    (
        "white_throated_sparrow-38",
        "white_throated_sparrow",
        7149657,
        5761350,
        "bradleysaul",
        108304,
        "adf755b8393fb6b4db99e788d4112c2c72197c33ecffeec2ca7b3ba5dd9d650e",
        "jpg",
    ),
    (
        "white_throated_sparrow-39",
        "white_throated_sparrow",
        27513220,
        17998912,
        "mefisher",
        49587,
        "f1716d49782f70455ec97ecdda634505b6b2afc375105b1d779353e704dda07b",
        "jpg",
    ),
    (
        "white_throated_sparrow-40",
        "white_throated_sparrow",
        642121459,
        351843522,
        "jeffcherry",
        53348,
        "c9e82d7e47ca2e51c856f9bd4d8f2965d8f69948f081c08c9fa6ebf44fcde711",
        "jpg",
    ),
    (
        "white_throated_sparrow-41",
        "white_throated_sparrow",
        642355950,
        351962427,
        "robinlanark",
        54504,
        "c0da16d5146afac102fa274fffea2e0b203282078ed692f3b7ad4a1340b0d4cd",
        "jpg",
    ),
    (
        "white_throated_sparrow-42",
        "white_throated_sparrow",
        638199267,
        349826931,
        "wildaboutwildlife",
        32981,
        "845f7d38d6f35d2e6e9dca61234c9566778fec1e274500c97e47977fcecd232f",
        "jpg",
    ),
    (
        "white_throated_sparrow-43",
        "white_throated_sparrow",
        119839965,
        73407944,
        "terrimewbornagain",
        61951,
        "dbfbaaa2f72caa812c026122f41512caef00f8f8fdd21c2307fcdf9d79921757",
        "jpeg",
    ),
    (
        "white_throated_sparrow-44",
        "white_throated_sparrow",
        6236370,
        5078910,
        "ruggedbynature",
        126403,
        "3d6030593bf57bc5734fd6c0706c1e350bb117a32370a16b194ef4488668f123",
        "jpeg",
    ),
    (
        "white_throated_sparrow-45",
        "white_throated_sparrow",
        32221828,
        20882923,
        "christinan",
        132485,
        "b92ce41128bb1c5b53a0c487b46ba80eea83b9a443ee9c72f100005e5d80e411",
        "jpeg",
    ),
    (
        "white_throated_sparrow-46",
        "white_throated_sparrow",
        36598789,
        23643589,
        "jessm-c",
        45861,
        "eda75171590418596e372c6cba058364acb64f1279d250dff6c5df61fa17868b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-47",
        "white_throated_sparrow",
        163198257,
        98065830,
        "ellyne",
        109261,
        "864fb42a82561684c1ac22ae71bd023ddeb15c6052b7732ebd2570a98b552a1a",
        "jpeg",
    ),
    (
        "white_throated_sparrow-48",
        "white_throated_sparrow",
        162150240,
        97493574,
        "c_burns802",
        188464,
        "0fb2fb2e04bca82c436315b77a974adc665d2d1fb4e419dd61afa068ab412254",
        "jpeg",
    ),
    (
        "white_throated_sparrow-49",
        "white_throated_sparrow",
        161756778,
        97282492,
        "raffib128",
        67237,
        "77384a65eb120bdb345c3015cb8c22fcec9c89df49b0e0c00954e3476834da9f",
        "jpeg",
    ),
    (
        "white_throated_sparrow-50",
        "white_throated_sparrow",
        165146429,
        99107203,
        "philippthompson",
        84518,
        "8582544ca8b3cb1c73174fc42a9d66971fdae0c90cd4739083eb4ef4983eb6e7",
        "jpeg",
    ),
    (
        "white_throated_sparrow-51",
        "white_throated_sparrow",
        177451999,
        105722646,
        "natepow",
        44983,
        "c81a68c46264d1a5fcae3042e7ab24653c3615665cef6293c31622d5f342d840",
        "jpg",
    ),
    (
        "white_throated_sparrow-52",
        "white_throated_sparrow",
        67630852,
        42591255,
        "schoenitz",
        138616,
        "34594c49a4f601b17e49dfc4d4098db7b73ff5a69d3830b773378e4d695ff181",
        "jpg",
    ),
    (
        "white_throated_sparrow-53",
        "white_throated_sparrow",
        99106006,
        61898962,
        "glennberry",
        122576,
        "fd8cd1c0b0c1fdd7636b8588c86c6d4f3cddc233e9bd4bccade5e2ed275d0e28",
        "jpg",
    ),
    (
        "white_throated_sparrow-54",
        "white_throated_sparrow",
        102162606,
        63634844,
        "eug302",
        73329,
        "692114df14fb8a75fdf38428a196467a4a394e5ded95605fbb48cf64903ccb1b",
        "jpeg",
    ),
    (
        "white_throated_sparrow-55",
        "white_throated_sparrow",
        370403033,
        209307218,
        "kuykenwil",
        124312,
        "d0700068b06ad0f79eeac331a92e5d633bd76be264c3d8801a9a6eb301193969",
        "jpg",
    ),
    (
        "white_throated_sparrow-56",
        "white_throated_sparrow",
        500790892,
        278893388,
        "quillipede",
        186273,
        "0db918123c7f529ca76c04129886eb9aeb947bd3e90cdfd968c4605931c16f85",
        "jpeg",
    ),
    (
        "white_throated_sparrow-57",
        "white_throated_sparrow",
        364738073,
        205369541,
        "sandra1142",
        144424,
        "4cd383751871426904f4d8ff7526a312ba33a6c894faf7121c6ec0ff6ce0d7b1",
        "jpeg",
    ),
    (
        "white_throated_sparrow-58",
        "white_throated_sparrow",
        374502828,
        211884837,
        "halliefromcali",
        62476,
        "0073652fddd7378dc936f280c18d7e44c4a7160aadc2f051e7b5e2896b1a476c",
        "jpg",
    ),
    (
        "white_throated_sparrow-59",
        "white_throated_sparrow",
        109241547,
        67591707,
        "blkillin",
        130154,
        "e5b8c1caa0f7cc7926d46eac42df582ccb0882b4a0733fc0740b8b4597a4e014",
        "jpeg",
    ),
    (
        "dark_eyed_junco-00",
        "dark_eyed_junco",
        172110799,
        102901486,
        "schylerbrown",
        182973,
        "185209c7a1111fc626a068e136ec3cfcdff3174d15f1c60af209d7b32fe7bef9",
        "jpeg",
    ),
    (
        "dark_eyed_junco-01",
        "dark_eyed_junco",
        46691943,
        29901256,
        "haida_gwaii",
        46823,
        "a92dca21e6e58c375fc313f0d1da2c86c11408beb0df8fd96fc60ae035ae80d5",
        "jpg",
    ),
    (
        "dark_eyed_junco-02",
        "dark_eyed_junco",
        707222551,
        386266764,
        "ben142",
        289608,
        "7bf320edf4d34a4848f3e9d175cfafa66cc5bab6a1a8f97c41c670263de3ff89",
        "jpg",
    ),
    (
        "dark_eyed_junco-03",
        "dark_eyed_junco",
        346777340,
        196961623,
        "zacharyfoster",
        46712,
        "ea8f9f0eebb4d2f86193c705344a8fab09bf634bc2217a0874ee57c4f0f5b4ab",
        "jpg",
    ),
    (
        "dark_eyed_junco-04",
        "dark_eyed_junco",
        8793471,
        6892999,
        "truthseqr",
        45302,
        "f8241eab39797c4e097a1432b13658466287d61ed515448457907c17e60cf5e2",
        "jpeg",
    ),
    (
        "dark_eyed_junco-05",
        "dark_eyed_junco",
        192557376,
        114006980,
        "k-simpkins",
        172398,
        "206f012add8a5d0fa434e07c51f1e7bd73a60c4d299a2202163fc662e1b03479",
        "jpeg",
    ),
    (
        "dark_eyed_junco-06",
        "dark_eyed_junco",
        243303909,
        141959574,
        "andywilson",
        71321,
        "ee5308b6f93fb40a6d795a7d8ca6f2ef55b4844513f4268ef34827e1c5e4c4a6",
        "jpeg",
    ),
    (
        "dark_eyed_junco-07",
        "dark_eyed_junco",
        274980085,
        159160633,
        "andy71",
        51209,
        "c54b45ca7fdc635bdb31eb89166c9fce84f0b0f8f0c17331fd5e42b582633c9b",
        "jpeg",
    ),
    (
        "dark_eyed_junco-08",
        "dark_eyed_junco",
        213798376,
        126031618,
        "nathanael15",
        57646,
        "d6b9e7d2dcc63cdd88b47cce328149aa9e8eb486ee2a5fc0f08b90601f2c7d9b",
        "jpg",
    ),
    (
        "dark_eyed_junco-09",
        "dark_eyed_junco",
        63482066,
        39977347,
        "chrisleearm",
        41870,
        "90573e814dbde965c70a934d9702110c40670aa22cc9f80ff8849db877d1fd72",
        "jpeg",
    ),
    (
        "dark_eyed_junco-10",
        "dark_eyed_junco",
        458710472,
        255914048,
        "joy4birds",
        90973,
        "fd25ecb1bf86896b84e9e4e8329c742011011e6883631f47dbe3744ba7463153",
        "jpg",
    ),
    (
        "dark_eyed_junco-11",
        "dark_eyed_junco",
        20784016,
        14046286,
        "gambolingquail",
        93957,
        "a454d6f987153317c9c57c05019b08ab0ebebc46d3cf72356b014231b69a1e9d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-12",
        "dark_eyed_junco",
        332842159,
        189933284,
        "jan-konilu",
        110145,
        "d18706f536164a72a4ec9bacf47437edd89d6b547f86eefdf10ed0166cebf0dd",
        "jpeg",
    ),
    (
        "dark_eyed_junco-13",
        "dark_eyed_junco",
        513004508,
        270404136,
        "thevertebratepokedex",
        141252,
        "854f75e7527932d0409e14756725388de9913c0b450536da717e54d683de169f",
        "jpg",
    ),
    (
        "dark_eyed_junco-14",
        "dark_eyed_junco",
        593499256,
        327583635,
        "orionid",
        108583,
        "7eab4cea6d46faa878732d82c5c2ece63ecc3e66781ea08d0bd950eee581d16d",
        "jpg",
    ),
    (
        "dark_eyed_junco-15",
        "dark_eyed_junco",
        256948665,
        149140989,
        "igor322",
        86925,
        "9339b8cf4aa347b4eb176ef1209dfaa4fcb11e277889cc17de399d6af2c8633a",
        "jpeg",
    ),
    (
        "dark_eyed_junco-16",
        "dark_eyed_junco",
        12533281,
        9255403,
        "braincellsgone",
        40857,
        "be34c51655f59612858d888b05a4f927da65e1aede850c9a5f1be68fa00bfcbe",
        "jpg",
    ),
    (
        "dark_eyed_junco-17",
        "dark_eyed_junco",
        12580989,
        9282523,
        "artemis224",
        216253,
        "15198c123c01fa0f8c03edc086408c8d95ffa553083c380e8ae236dbb7056093",
        "jpg",
    ),
    (
        "dark_eyed_junco-18",
        "dark_eyed_junco",
        63590391,
        40041059,
        "bobbyblackmore",
        82853,
        "87746be0bca30fc40ddba739c1a35ed1fdd41882ebfe3561518b69d285f5ae5d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-19",
        "dark_eyed_junco",
        106718005,
        66230973,
        "vicki936",
        41475,
        "e6903e9a69e987c45edd468ace0bf71adf8252063ff5d39c2130409d8fea68be",
        "jpeg",
    ),
    (
        "dark_eyed_junco-20",
        "dark_eyed_junco",
        611049594,
        336277421,
        "skylar_schell",
        15425,
        "7e876479febb3d44b1e494a25f778efe4e8bfd323279031cb08f9f6fe95f742e",
        "jpg",
    ),
    (
        "dark_eyed_junco-21",
        "dark_eyed_junco",
        591147962,
        326403963,
        "toknowtheland",
        106373,
        "07770f3313dc969802355fa8b3e62a86edb115a64771bd38938e1fab33220c39",
        "jpg",
    ),
    (
        "dark_eyed_junco-22",
        "dark_eyed_junco",
        459696953,
        256398190,
        "w_mark_c",
        262126,
        "5025e757c026e02160ff3f0e82f1f4434b888a6c9689e7ac76274c1fcec64f0c",
        "jpg",
    ),
    (
        "dark_eyed_junco-23",
        "dark_eyed_junco",
        484171775,
        269292238,
        "aschuman",
        59459,
        "89dd933d12cf21c0e73eb01e96a9279e2e57b36676a72a9c860281d0d7703282",
        "jpg",
    ),
    (
        "dark_eyed_junco-24",
        "dark_eyed_junco",
        469746672,
        261469406,
        "shannon_j",
        74167,
        "ec4c7ce15aa1c4eadd3bc5d46d433baf9560d6a0e4593d8d57aab8fa9bec9e18",
        "jpg",
    ),
    (
        "dark_eyed_junco-25",
        "dark_eyed_junco",
        357382685,
        202362003,
        "dougbrown",
        56324,
        "70275d8e07192c99e121b67d206de6823b194f043e65b9766f1f9ce772953c78",
        "jpeg",
    ),
    (
        "dark_eyed_junco-26",
        "dark_eyed_junco",
        7660371,
        6119391,
        "jeffreyleeisanaturalist",
        108394,
        "0f667e9af8e8d3585807da58a2f315c5dbcb7eece5f6862006fb9da68a4bef42",
        "jpg",
    ),
    (
        "dark_eyed_junco-27",
        "dark_eyed_junco",
        437957476,
        245430473,
        "eug302",
        106520,
        "394abfbcf192ecf5a76cb2f19dfa2a62070fa3874b084d1468219d6ee5db6cbf",
        "jpeg",
    ),
    (
        "dark_eyed_junco-28",
        "dark_eyed_junco",
        339439778,
        193239701,
        "rawcomposition",
        32258,
        "73fad57ed975df01c529e77eaca9eab9b00741ea2cfc91358856aefca620a10a",
        "jpg",
    ),
    (
        "dark_eyed_junco-29",
        "dark_eyed_junco",
        634100291,
        347744522,
        "zorthesosen",
        169264,
        "932237001fe68e3d6d19c496fd448b8a82f254316211ac6f9d35708a1395ed81",
        "jpg",
    ),
    (
        "dark_eyed_junco-30",
        "dark_eyed_junco",
        6198359,
        5055484,
        "glmory",
        108168,
        "d31767f42eaaf7f3527133fffb9a0271a9dbbe66fc036d5c694b605563d09965",
        "jpg",
    ),
    (
        "dark_eyed_junco-31",
        "dark_eyed_junco",
        6485959,
        5243663,
        "fake_id",
        148254,
        "ce4dab6c5b37b7ebac07f69d4040d4686fea9fd4b753f91bdf15a3a440820712",
        "jpg",
    ),
    (
        "dark_eyed_junco-32",
        "dark_eyed_junco",
        246318772,
        143610209,
        "wildreturn",
        62090,
        "839e57443659659522ed65dfd054a8fc0a6bfe400518d9f9a7519383d33a2c0d",
        "jpeg",
    ),
    (
        "dark_eyed_junco-33",
        "dark_eyed_junco",
        253199424,
        147174876,
        "sean579",
        121660,
        "6efc17eb4c0b184d64a605a76d3734d9bff5d26e8c8609d2f0b0dd4ccc8b3faa",
        "jpeg",
    ),
    (
        "dark_eyed_junco-34",
        "dark_eyed_junco",
        244315013,
        142500114,
        "enspring",
        63514,
        "1d5012c2b4504b4347fef36d8764eebd1bf07649a14cf874e6d0b3eba9fb3c4f",
        "jpg",
    ),
    (
        "dark_eyed_junco-35",
        "dark_eyed_junco",
        256191686,
        148740024,
        "nartb",
        194532,
        "e88b1449a989fc62c5da1c592e4987fa4db71eadf1ef29c0c5c1159b8f77a298",
        "jpg",
    ),
    (
        "dark_eyed_junco-36",
        "dark_eyed_junco",
        241013731,
        140770209,
        "nana10",
        196479,
        "fc773ca1b06e0e3673aac5c2213c66981ede11bf3ad4a0cdef2102c4f1d99843",
        "jpeg",
    ),
    (
        "dark_eyed_junco-37",
        "dark_eyed_junco",
        240712071,
        140608440,
        "saintaardvark",
        129995,
        "94c4dc30ca31804d6645a6be802cf87ccaffe974f0d3cb554b4fb3e9696ec6a5",
        "jpg",
    ),
    (
        "dark_eyed_junco-38",
        "dark_eyed_junco",
        242909841,
        141750179,
        "calinsdad",
        61773,
        "c49c5d22969552de0fc04ff95eefb8a2ffc76df528477b93c2e2c38cdebe8f6c",
        "jpg",
    ),
    (
        "dark_eyed_junco-39",
        "dark_eyed_junco",
        109385584,
        67667584,
        "ninetoes",
        64185,
        "64e9ee29cf017c292e1711b4c6482488a74c8f652e88e8b7c1df883dc91d75e2",
        "jpg",
    ),
    (
        "dark_eyed_junco-40",
        "dark_eyed_junco",
        102027986,
        63561089,
        "michaelnaumoff",
        114346,
        "eccaa3c60fe575e28fc60f8f6896a99065bff4aee0a4fdc057d7220e7870ca4d",
        "jpg",
    ),
    (
        "dark_eyed_junco-41",
        "dark_eyed_junco",
        105652410,
        65622040,
        "don54",
        159856,
        "5c4282e5e85af7a886eceeadc90977bcd11bf26658176ba5ef8182d2f4a2bfe1",
        "jpeg",
    ),
    (
        "dark_eyed_junco-42",
        "dark_eyed_junco",
        267851501,
        154986181,
        "shanebustapbj",
        203657,
        "217002a366779db0ccc3fddffa71e79472b1fcaa5d444049971dac4c31c910f5",
        "jpeg",
    ),
    (
        "dark_eyed_junco-43",
        "dark_eyed_junco",
        268597047,
        155389350,
        "j-dehoog",
        54916,
        "c0890844944f3d603f8220b0a75213051766c6a45b2ba1fed51381134832f69f",
        "jpeg",
    ),
    (
        "dark_eyed_junco-44",
        "dark_eyed_junco",
        515881506,
        286951393,
        "jubileej",
        108221,
        "9cbecae915f1c96dcc31d5fd77cb5902181152cc3b918693191cc11fce18822c",
        "jpg",
    ),
    (
        "dark_eyed_junco-45",
        "dark_eyed_junco",
        640462423,
        351001488,
        "russnamitz",
        44508,
        "63276eb90201f890b977a54b95bf1d65592eef98089dbb73f2e27e66aaf49a65",
        "jpg",
    ),
    (
        "dark_eyed_junco-46",
        "dark_eyed_junco",
        649025513,
        355899297,
        "rocksand2134",
        233215,
        "2f2545fa5e7901c0383578dea84131eb4be551b8f2bd55d9f308ebc838144f39",
        "jpg",
    ),
    (
        "dark_eyed_junco-47",
        "dark_eyed_junco",
        625101518,
        343192310,
        "jtdavis05",
        106387,
        "4642076ebc5e9bcde85dbba65eb16a49d3fabe6c82cc707b4863ad4b7a4531ae",
        "jpg",
    ),
    (
        "dark_eyed_junco-48",
        "dark_eyed_junco",
        509902416,
        283764048,
        "loarie",
        271588,
        "0c1e3897caca8ff3ff4f3eff7548e30ca50dfe2c2c972106303f9e21c259dbda",
        "jpg",
    ),
    (
        "dark_eyed_junco-49",
        "dark_eyed_junco",
        627787858,
        344552039,
        "margohj",
        54717,
        "a41e557077525b064fa16a7c0b140d608d323663519cfbf424f4454d79951244",
        "jpg",
    ),
    (
        "dark_eyed_junco-50",
        "dark_eyed_junco",
        630444682,
        345899876,
        "robinlanark",
        102578,
        "aa4ffd5dbd553d60ba687d57b21b52ce21572d3c711471abbc5e959bd1126da2",
        "jpg",
    ),
    (
        "dark_eyed_junco-51",
        "dark_eyed_junco",
        635784743,
        348597723,
        "lina47741",
        317439,
        "392d1114b7fc1bb3b041d9895ceb049432a96951764e8949d5939315b96bcc54",
        "jpg",
    ),
    (
        "dark_eyed_junco-52",
        "dark_eyed_junco",
        518407567,
        288278997,
        "zzphantom",
        231733,
        "de2a5be329b583911babe0758c9ce8c0982191f743b8082eb826623256709f71",
        "jpg",
    ),
    (
        "dark_eyed_junco-53",
        "dark_eyed_junco",
        620711072,
        341026776,
        "ethologist",
        186675,
        "13f47961b2a1654ec40f6d1a2005afbf34c75060987fbb662001e66e10a023ed",
        "jpg",
    ),
    (
        "dark_eyed_junco-54",
        "dark_eyed_junco",
        638143588,
        349800424,
        "tom_lazar",
        157629,
        "0be0b84bbc956c616ce35e9e704e8c9e01f0d24db7eeb157fcaa3b3952bd9b23",
        "jpg",
    ),
    (
        "dark_eyed_junco-55",
        "dark_eyed_junco",
        668612910,
        366183670,
        "gendereuphorbia",
        76372,
        "4bfa9f780e9fd0327afec60a8b42e3378b8a9917a0461fd77a6df73b1a94e6bb",
        "jpg",
    ),
    (
        "dark_eyed_junco-56",
        "dark_eyed_junco",
        116078128,
        71308227,
        "msieges",
        47042,
        "ecd4ade5abf058ac887fabe2706eb021ada3b9ab083d10728c9b1dc26de56bbc",
        "jpeg",
    ),
    (
        "dark_eyed_junco-57",
        "dark_eyed_junco",
        117580446,
        72145931,
        "hewwowhy",
        67000,
        "1a8ca3be627c192459440c2eb07bd85b348f0e876a7de32bfc6fe0fb2959a618",
        "jpg",
    ),
    (
        "dark_eyed_junco-58",
        "dark_eyed_junco",
        117044036,
        71848977,
        "radrat",
        81035,
        "7362dd0a27d11b18d5a79abcce9a80304dc80dc0ea1fa84265a1cf0638687641",
        "jpeg",
    ),
    (
        "dark_eyed_junco-59",
        "dark_eyed_junco",
        11110817,
        8359397,
        "giselle9",
        71741,
        "b2e74f27ddca8d1e6dd4807251b07867cbfec8b6bf51d93ef7f142109f705520",
        "jpeg",
    ),
    (
        "house_finch-00",
        "house_finch",
        697940852,
        381438133,
        "ben142",
        196735,
        "a89f8e0263fdabb404b462acaa592f5dd2ac88ee4615da444470de4a1fae82d5",
        "jpg",
    ),
    (
        "house_finch-01",
        "house_finch",
        176982307,
        105476125,
        "vicki936",
        22211,
        "c377fb361df0324c7a856d9344968886ece3b94bd67188c9325b8d2d284d3a2f",
        "jpeg",
    ),
    (
        "house_finch-02",
        "house_finch",
        117990649,
        72375345,
        "kristen163",
        75945,
        "eeafad0dd2e91ecfe45c9d1f27dd0392a01bd81099549c60fd2e36a4b4342a9f",
        "jpeg",
    ),
    (
        "house_finch-03",
        "house_finch",
        389479656,
        220010434,
        "aster-asti",
        82128,
        "a8848197b4e7890e07538d492480c4275b75d04e10c1ae95aee91aaafe3319c5",
        "jpg",
    ),
    (
        "house_finch-04",
        "house_finch",
        98576538,
        61594129,
        "enspring",
        46784,
        "8b355426d8fe6327f202c6a9458cee1b95de445bfbf7e48a4b5a2c7d0eb78a8c",
        "jpg",
    ),
    (
        "house_finch-05",
        "house_finch",
        72470599,
        45698380,
        "henrya",
        61724,
        "1b96d37a7078e1b725b80af4b10848da58b0d0c17a70c8ac01e326c0a749ee6b",
        "jpeg",
    ),
    (
        "house_finch-06",
        "house_finch",
        80751781,
        50842166,
        "leahmfulton",
        44269,
        "6d6202de26f042d83ee6c5af550cd74e6ab10eb796eed2f78c83ac9e2368e4b7",
        "jpg",
    ),
    (
        "house_finch-07",
        "house_finch",
        214612538,
        126483167,
        "hamiltonturner",
        124116,
        "6b7687640c4641b974865da04cf9eaf1f86b774ebc19678f2fc39e55c8648930",
        "jpeg",
    ),
    (
        "house_finch-08",
        "house_finch",
        630196420,
        345777550,
        "truthseqr",
        149455,
        "abb84d1e327dd82c07cbea3dd5583c07453e69b2cc220397b101e597da81bd6c",
        "jpg",
    ),
    (
        "house_finch-09",
        "house_finch",
        213077180,
        125637342,
        "jnicat",
        25823,
        "e1e3baff8d0bd72339e3e49089f7f4f2c1dd383ff005e49a964a1bacc4f8ebb8",
        "jpeg",
    ),
    (
        "house_finch-10",
        "house_finch",
        500744872,
        278868417,
        "pbaff",
        149626,
        "2684cfb1fc40d7766610a5920ead0ad0c27c338ccb4fb9ed18baddda150568b0",
        "jpeg",
    ),
    (
        "house_finch-11",
        "house_finch",
        268678834,
        155431721,
        "stevestevens",
        120325,
        "5d1e8c097c214d14ef7b895bf95769ae9bc25fa799a98bbfa6f12f2f84e6af79",
        "jpeg",
    ),
    (
        "house_finch-12",
        "house_finch",
        358373136,
        202884575,
        "kcthetc1",
        52329,
        "b9929163e4fdadef26c753437ac7af3ad550aa047b15cba131c05d4d335799fe",
        "jpeg",
    ),
    (
        "house_finch-13",
        "house_finch",
        104227663,
        64793290,
        "verdantpulsar",
        101003,
        "90fe37c477bad9ed30ab119e1a7445ffc2720e548a22bb65d66b2ff7b82337d5",
        "jpeg",
    ),
    (
        "house_finch-14",
        "house_finch",
        196156834,
        116218899,
        "kgarrett",
        56801,
        "ce03d1d70a89b5e6e0088f4307f8077573c3571b91997725ca2e6c69be80e2d0",
        "jpeg",
    ),
    (
        "house_finch-15",
        "house_finch",
        454332148,
        253709711,
        "rlaortiz",
        149985,
        "cf007ef8ac57bc0eb085c9eecf9fc95eb69a29df706998de237df24f9491c61d",
        "jpeg",
    ),
    (
        "house_finch-16",
        "house_finch",
        665287910,
        295246528,
        "dinomariobob",
        74972,
        "5f6121c1f8dbfaccbb61c279a579c22600744eb4118c2eeef09e69c86f6e1a49",
        "jpg",
    ),
    (
        "house_finch-17",
        "house_finch",
        509969159,
        283798927,
        "damienxw",
        104277,
        "56d3656be473c362f1ccd09d15e62cbcfe9137d83bef7a3312d72444921c7805",
        "jpg",
    ),
    (
        "house_finch-18",
        "house_finch",
        168402062,
        100871757,
        "k-simpkins",
        86922,
        "209a884cf7618d0b85679ae3a72f237a6d03a5a3096f6ab1b2e5503c38c50d9d",
        "jpg",
    ),
    (
        "house_finch-19",
        "house_finch",
        661042217,
        362189851,
        "dougbrown",
        56426,
        "ad704994fda99779aa340ca1c637b1371f0513df6f36973263bb9ee88cf6713b",
        "jpg",
    ),
    (
        "house_finch-20",
        "house_finch",
        701862058,
        383472210,
        "dblanco",
        25222,
        "8562ed9c4e889b34f83b40c55dd93b821a6b5829a3b1c681374a53c3ca9bf01f",
        "jpg",
    ),
    (
        "house_finch-21",
        "house_finch",
        709783562,
        387605980,
        "fraskar",
        99555,
        "60a1926358c4ec04aeec9414381177600580c8237899c852e03472e53cf87f31",
        "jpg",
    ),
    (
        "house_finch-22",
        "house_finch",
        703924789,
        384540425,
        "meshhy",
        110802,
        "2df38d1aaab2c268c7a47f05e55bef1e83690eeeb91f34874c5b479a5ff0a2cb",
        "jpg",
    ),
    (
        "house_finch-23",
        "house_finch",
        705516095,
        385380385,
        "m_aniket",
        77797,
        "ba009764b66ed89efc9b0aaacd9565830c65039967961488d774ce3c9add5d64",
        "jpg",
    ),
    (
        "house_finch-24",
        "house_finch",
        577527110,
        319485540,
        "glmory",
        94407,
        "6d4cd80f8ef77f41575befae0d8534e9bc397685dceb7f629b5cbc24447a3613",
        "jpg",
    ),
    (
        "house_finch-25",
        "house_finch",
        570686433,
        315887836,
        "emilyheaton",
        229263,
        "27e0bd21c1b322aa5d74c5504dbf27bfb5a0fdeea9718b6ffcee7926f2cb2733",
        "jpg",
    ),
    (
        "house_finch-26",
        "house_finch",
        266259557,
        154114419,
        "andywilson",
        47792,
        "b0dc2de7134cc5af50b1482681b450d32b1f58765990d5d553c4838ae7936db0",
        "jpeg",
    ),
    (
        "house_finch-27",
        "house_finch",
        269729570,
        156011712,
        "kbkash",
        59048,
        "3af407624c46ba4218d24a198ed22b812d4967380bd0e7f67e960f7fbb2defe7",
        "jpeg",
    ),
    (
        "house_finch-28",
        "house_finch",
        270277531,
        156183453,
        "aparrot1",
        83424,
        "93c92a1bb7cf89e7106b0c3df65ce030c82bccade181d661952e940ddcc6282a",
        "jpg",
    ),
    (
        "house_finch-29",
        "house_finch",
        658061500,
        360629741,
        "mike_cove",
        107594,
        "deaaefaf712059b6c8a929a83e1ffd0295e12f0f291bc9869011703eecfcf372",
        "jpg",
    ),
    (
        "house_finch-30",
        "house_finch",
        545518752,
        302693775,
        "cvharris",
        208917,
        "fc5ffdbd30bc76fa4371a99f59543c9137411ddac0a691e2e8122ae7796e038b",
        "jpg",
    ),
    (
        "house_finch-31",
        "house_finch",
        662587006,
        362998939,
        "curran",
        113662,
        "644931afff18c942930717ac3ca563b63c18a0c78ab12ff88557aeede18ead63",
        "jpg",
    ),
    (
        "house_finch-32",
        "house_finch",
        665549166,
        364566546,
        "sealgyu",
        130087,
        "e6fb3e42979ce2709433e8141c4c665a70c4942166153578c7f0173328ffb2ad",
        "jpg",
    ),
    (
        "house_finch-33",
        "house_finch",
        566433127,
        313653086,
        "fake_id",
        150986,
        "cb8a50f48f65795850f57c4d8f84e49debc3deed59b6a60d5bee8ef95299650f",
        "jpg",
    ),
    (
        "house_finch-34",
        "house_finch",
        404592864,
        228025269,
        "logan_artz",
        78971,
        "d0cf631b138d199951d96538903eeb1b56e3b8da3f28c7b042d8868060f27a44",
        "jpeg",
    ),
    (
        "house_finch-35",
        "house_finch",
        423118612,
        237726292,
        "conhawn",
        153347,
        "826ddd6fa8fad5bd1450c0c7beb77fa3261f41a0afa61d7681b7096bc4225c40",
        "jpeg",
    ),
    (
        "house_finch-36",
        "house_finch",
        290080883,
        167451466,
        "vijaybarve",
        89129,
        "f0b32b4af4bdf0bc7eaf298b37a9be6dbcb175d9f72553efc239b6afd2746a69",
        "jpeg",
    ),
    (
        "house_finch-37",
        "house_finch",
        296347840,
        170825854,
        "susanaber",
        96563,
        "47832989fc212f19631e55b63c23155817209c40c66af49e4e03e42863243770",
        "jpeg",
    ),
    (
        "house_finch-38",
        "house_finch",
        414288752,
        233136904,
        "wafflemaster135",
        29548,
        "9f2039a678a69e8f8f383d43f80e3f4e6fcc81d5b99285a0e5b5e54834cf9125",
        "jpeg",
    ),
    (
        "house_finch-39",
        "house_finch",
        284751592,
        164572867,
        "efalquet",
        80806,
        "b50e0a0a4a0bae79b5a8cd034645252044601a29b081f0842443848e802e82d2",
        "jpeg",
    ),
    (
        "house_finch-40",
        "house_finch",
        205914234,
        121669780,
        "darylnolan",
        44763,
        "c04e8d7c002ad0038dcb62843e776ffb8fc86f425c2399442dd1653bbaa4b904",
        "jpeg",
    ),
    (
        "house_finch-41",
        "house_finch",
        96688995,
        60467105,
        "erikschiff",
        145431,
        "dd056b1c0d89dcb9fa06b1c38936142e5c825eb2aeed0d7d2265dc7e147be792",
        "jpg",
    ),
    (
        "house_finch-42",
        "house_finch",
        732358790,
        399184296,
        "katrinamccollough",
        41331,
        "76c899fd9b9aa84ad41888c71a36ab99431fe7201dd417f2c19a66f77388420d",
        "jpg",
    ),
    (
        "house_finch-43",
        "house_finch",
        83824778,
        52720178,
        "beesbirdsbugs",
        48816,
        "72731eb63fa9c5058f9911d11be929af9a58cfea805159d54010e9095948c39a",
        "jpg",
    ),
    (
        "house_finch-44",
        "house_finch",
        349400403,
        198261102,
        "paulgraham",
        105965,
        "b0d1f556ee4926e6a6f87201f6b73955dd9c530788e0ce57d5e09ced74ede001",
        "jpeg",
    ),
    (
        "house_finch-45",
        "house_finch",
        484652323,
        269544283,
        "robinlanark",
        29863,
        "a1d6e8ffbcef10a83a2d3674c7228af89f8d39297f44cd569d1e031da02bc516",
        "jpeg",
    ),
    (
        "house_finch-46",
        "house_finch",
        341357046,
        194221751,
        "haconra",
        80643,
        "8e11937a5c6d5a47da47f562d52d4089d58ad3f2adefadc287b30c562c7ece85",
        "jpg",
    ),
    (
        "house_finch-47",
        "house_finch",
        622140195,
        341714166,
        "rachel141",
        133938,
        "aa776ef9fd3022b6b3db824a762bdde82eeb95518a8764763bd8199683bf1d04",
        "jpg",
    ),
    (
        "house_finch-48",
        "house_finch",
        515309527,
        286632225,
        "solunasilver",
        29507,
        "72a6b495ed99de80504c6b1a86ac1fbfc9e5736bc6f43274bd6093cc91f9ea05",
        "jpg",
    ),
    (
        "house_finch-49",
        "house_finch",
        514877896,
        286416757,
        "jubileej",
        153667,
        "26cd596525a244d2ba82ec2d575834c15671080814400cf7862f614c0db68d07",
        "jpg",
    ),
    (
        "house_finch-50",
        "house_finch",
        125605349,
        76831730,
        "sarahangulo",
        60174,
        "236eb345f1cf1ab4c82fb3b2cc48b948e614f8ddfd958c87a62ce5b44a04f080",
        "jpeg",
    ),
    (
        "house_finch-51",
        "house_finch",
        12881677,
        9467101,
        "artemis224",
        173956,
        "91b5f9f66e8b84cd96169894024ebc3519de6f737d3a32ee3b70100c22b039b9",
        "jpg",
    ),
    (
        "house_finch-52",
        "house_finch",
        124405788,
        76034586,
        "radrat",
        36008,
        "bb9b3fe0b37a1ab6342b1e573f3faf3a287669b70fa9f00b5bfce83de4a376c6",
        "jpeg",
    ),
    (
        "house_finch-53",
        "house_finch",
        17468099,
        12163814,
        "andy71",
        199644,
        "6bc02aaf56c1549dd78d6711f9121d48c33fbd76a8fb623895369f05604ad30d",
        "jpg",
    ),
    (
        "house_finch-54",
        "house_finch",
        134699271,
        82117365,
        "seanwashington1",
        35334,
        "d6e0fe4a8cc690fd6369deff8ea67f4810e1f1297a6f1b9f125aae905742db2a",
        "jpeg",
    ),
    (
        "house_finch-55",
        "house_finch",
        134287410,
        81854554,
        "omcelroy",
        90394,
        "6b4a5ea4c8aa1722332ccdf420af0ca0906cfdb87348e9f6c0904d5a92d8c203",
        "jpg",
    ),
    (
        "house_finch-56",
        "house_finch",
        17763955,
        12337763,
        "jennifer510",
        167368,
        "35355c603049fdf21fc3bf1fed827b50740cfdba134542707026bb4e1f32149b",
        "jpeg",
    ),
    (
        "house_finch-57",
        "house_finch",
        18946521,
        13021113,
        "deejay",
        60576,
        "2420363f6b02433f4b03f3323148ed5e4ac9d37c58a27e859f8e48354db866c8",
        "jpeg",
    ),
    (
        "house_finch-58",
        "house_finch",
        14590841,
        10492098,
        "silvercat",
        120515,
        "23934e0376c70dd6d5ce5e9a24c4176291c877432d25257f07e27694495c879a",
        "jpeg",
    ),
    (
        "house_finch-59",
        "house_finch",
        124702263,
        76228853,
        "janeyair",
        162431,
        "5c86d7cbd28e09d44814735f1e0a21e8266bd4e6a4396b047d092d2f5f3f2111",
        "jpeg",
    ),
    (
        "american_goldfinch-00",
        "american_goldfinch",
        84579952,
        53187208,
        "glennberry",
        59673,
        "72d36079e592e0a83c2f774f9073bfd4cc81253452c925d1673217ddd4b52a36",
        "jpeg",
    ),
    (
        "american_goldfinch-01",
        "american_goldfinch",
        12533322,
        9255418,
        "braincellsgone",
        55661,
        "6344e0125e74791f43ac6e07e5e1b9fbfce6d19bc62b6bb5d83b3caff9f7bcbc",
        "jpg",
    ),
    (
        "american_goldfinch-02",
        "american_goldfinch",
        131102823,
        80016788,
        "radrat",
        98990,
        "232a944f7e3351d4916a12ef2f6d598e7b007caaa95a9b064a14c1ba3af2a6ff",
        "jpeg",
    ),
    (
        "american_goldfinch-03",
        "american_goldfinch",
        175048222,
        104466897,
        "eug302",
        44231,
        "11c723482cc75fcc3a723ac1c0818a68e2fcf74c4ca684cf60195bf0d33f274e",
        "jpg",
    ),
    (
        "american_goldfinch-04",
        "american_goldfinch",
        68849595,
        43390778,
        "mefisher",
        154503,
        "66f07bc59bb3fdedd65a4537ebabd0cafd457826b8bf4bb633181f584a3edfd1",
        "jpg",
    ),
    (
        "american_goldfinch-05",
        "american_goldfinch",
        431916465,
        242278180,
        "k-simpkins",
        45413,
        "d76e7adf33e3a84ebec24dde5438965e62ebec8595755453973846340e4f460d",
        "jpg",
    ),
    (
        "american_goldfinch-06",
        "american_goldfinch",
        230801384,
        135330401,
        "enspring",
        42886,
        "78aebfb9b28c3e16dd9618a0e1ae06df67bfa4b8550c0879427d96915c475fed",
        "jpeg",
    ),
    (
        "american_goldfinch-07",
        "american_goldfinch",
        377136648,
        213398931,
        "nathan1177",
        66516,
        "7ad75838fdf2020a8e426e97507c7dd4355da93e6eece241128c28adbe302438",
        "jpg",
    ),
    (
        "american_goldfinch-08",
        "american_goldfinch",
        294667312,
        169935316,
        "dande",
        163470,
        "39e7892e81eeef6af4887e61bc0998e17797688eb6394fcc8c9438391e875ee0",
        "jpeg",
    ),
    (
        "american_goldfinch-09",
        "american_goldfinch",
        660472044,
        361884286,
        "ben142",
        270599,
        "733d64cd50c61334682f0862f5c7859ded34cae5776ee0bc6594fee400dc4876",
        "jpg",
    ),
    (
        "american_goldfinch-10",
        "american_goldfinch",
        143215717,
        86889530,
        "memoosborne",
        129438,
        "ef4a9a771cef9ba3c2c047eb106a6aa220236dd6aaa6aade5f4ef3a37c8abcba",
        "jpeg",
    ),
    (
        "american_goldfinch-11",
        "american_goldfinch",
        403873164,
        227647158,
        "drew_baxter",
        106009,
        "74e34c776f1b9a5d375a7dfae0e309cd42dbd133b82a1ecad4a49111ac7eed49",
        "jpeg",
    ),
    (
        "american_goldfinch-12",
        "american_goldfinch",
        481153019,
        267722510,
        "vicki936",
        255397,
        "3326ddbee3270241b681cb636466e742491f110e9841f453f5158864aadaea36",
        "jpg",
    ),
    (
        "american_goldfinch-13",
        "american_goldfinch",
        213311122,
        125763980,
        "hickl",
        24740,
        "0c295b3761bced4215519ff24a4f734773b54be98de456a4444d0819456af7dd",
        "jpeg",
    ),
    (
        "american_goldfinch-14",
        "american_goldfinch",
        417352824,
        234736320,
        "joy4birds",
        81109,
        "357014c108519543471b94f39591667d1a67d87fd1fdc4702a3a449235d6bddd",
        "jpeg",
    ),
    (
        "american_goldfinch-15",
        "american_goldfinch",
        72130720,
        45486482,
        "dctphoto",
        178111,
        "17b2f3599e20161acc17fd63bf61e9f40488b9c4d5bfaefc8cae54de42997509",
        "jpeg",
    ),
    (
        "american_goldfinch-16",
        "american_goldfinch",
        45148898,
        28952026,
        "megachile",
        49358,
        "6bbc6ca0ac074e486c20ce4c3fad5dc863cb2e908efc2b1ef97494403a351252",
        "jpeg",
    ),
    (
        "american_goldfinch-17",
        "american_goldfinch",
        133251099,
        81250513,
        "raffib128",
        128110,
        "863c76587fe1de80a84b97c2e72f38ae1cf4fa972789e006d9b0b544d696c6ef",
        "jpeg",
    ),
    (
        "american_goldfinch-18",
        "american_goldfinch",
        155797129,
        93957328,
        "nathanael15",
        74086,
        "7059ae3bdeeb48fe949e5b70b788bd28c96aecd0fe49cb7be22db9a8697bf5a5",
        "jpg",
    ),
    (
        "american_goldfinch-19",
        "american_goldfinch",
        11400438,
        8535447,
        "akneidel",
        26423,
        "4af74c1d04ddc7bbb7bb0e9eb2977a1daf48dd9b9477d71d69a7c4cb5d4f785b",
        "jpg",
    ),
    (
        "american_goldfinch-20",
        "american_goldfinch",
        55990791,
        35505213,
        "conhawn",
        84915,
        "69f768c39a2180440bdcbfc6addc5d426341e8080d4cf9ba8241d57564b3e6fb",
        "jpg",
    ),
    (
        "american_goldfinch-21",
        "american_goldfinch",
        145464774,
        88186208,
        "wildreturn",
        68204,
        "565d2a3b6d404e9ea0c24737ebcc5a3a82057b1452b4790df5e9552e8c50e592",
        "jpg",
    ),
    (
        "american_goldfinch-22",
        "american_goldfinch",
        637247336,
        289067166,
        "dinomariobob",
        142312,
        "01c63d305fce8edcc3a494f0543154c6bd6aa52368e02be84cf6c23e95b94cdd",
        "jpg",
    ),
    (
        "american_goldfinch-23",
        "american_goldfinch",
        460988055,
        257029007,
        "eric112",
        60314,
        "0db1b5f0e32d1faf861a437a20794fa18e80ea8966313933c145451c9319e2a9",
        "jpg",
    ),
    (
        "american_goldfinch-24",
        "american_goldfinch",
        59072170,
        37280587,
        "bradenjudson",
        23468,
        "827d77bf9ca9cc456e867434a07b68cdceb4a20e09eb9e6b67757c80cd31ff42",
        "jpeg",
    ),
    (
        "american_goldfinch-25",
        "american_goldfinch",
        66515260,
        41915252,
        "reuvenm",
        58833,
        "90f0b687d9626fdf5d0111b794cf960cf18f3de4c8eab601fcb23c41b45c4df7",
        "jpeg",
    ),
    (
        "american_goldfinch-26",
        "american_goldfinch",
        109875643,
        67928198,
        "artemis224",
        217909,
        "99e1d7d30eb19d47c7974e9ddd7efe4d06329c952a41e15d0a4b2095b747df61",
        "jpg",
    ),
    (
        "american_goldfinch-27",
        "american_goldfinch",
        219478221,
        129171341,
        "cgmayers",
        96993,
        "93bdee27bb526b60e0edfa518a923b3c10c41ba57221d6a23f4de15bf4a91600",
        "jpeg",
    ),
    (
        "american_goldfinch-28",
        "american_goldfinch",
        99106353,
        61899131,
        "thyg",
        50544,
        "0272bf89b379c2689247f1dbd6f4b36d7f197898baad91d7a8ebdf1c87f88fbd",
        "jpg",
    ),
    (
        "american_goldfinch-29",
        "american_goldfinch",
        223110598,
        131133187,
        "marissa3",
        89214,
        "6a2bb601ac3a70e80e6ccb33d4a24af43128d1e7afe3e92d42e803234bdf532e",
        "jpg",
    ),
    (
        "american_goldfinch-30",
        "american_goldfinch",
        104649681,
        65037800,
        "kemper",
        113821,
        "f4430daf9fca2cf6deb6b988148a5c2e5ae382af8c5db7e5cd37b26e8fbc9461",
        "jpeg",
    ),
    (
        "american_goldfinch-31",
        "american_goldfinch",
        98669936,
        61655828,
        "cloaca_enthusiast",
        31964,
        "3ddcb2e2e6863ef08a40fae7640e05b36ccec9fab9892ac494f095c4588b746f",
        "jpg",
    ),
    (
        "american_goldfinch-32",
        "american_goldfinch",
        347798463,
        197465311,
        "aster-asti",
        116039,
        "e1236dd9850a412ba68386ffd342d180f582db187b83c1d33cb1e09dd51d0043",
        "jpg",
    ),
    (
        "american_goldfinch-33",
        "american_goldfinch",
        461261879,
        257163676,
        "erikschiff",
        40881,
        "eb0b1ea08fcee160edd008ffc386b69b7b50bb89096c44873b978072c581d826",
        "jpg",
    ),
    (
        "american_goldfinch-34",
        "american_goldfinch",
        343726970,
        195390446,
        "robinlanark",
        53508,
        "8b1f4e496755d1a3c4f567c6a48673998b8480ce7d38e7c52bbd94d7ee9901ed",
        "jpeg",
    ),
    (
        "american_goldfinch-35",
        "american_goldfinch",
        352259924,
        199722919,
        "carterdorscht",
        122243,
        "11edfdb53f892b1123df734cb7300e002171baeef21671a31411ff30d156bdc6",
        "jpeg",
    ),
    (
        "american_goldfinch-36",
        "american_goldfinch",
        148829913,
        90084462,
        "ian-wolfe",
        172110,
        "4f6a493313fc596665eb99f8e303ab0ee980b4932a9957993cd00b03b5360fa4",
        "jpg",
    ),
    (
        "american_goldfinch-37",
        "american_goldfinch",
        276388190,
        159958623,
        "dianeclark6280",
        39786,
        "284ca7c9e814fede2c5a15e427a46d7dd82fa6984c4c862c380b0dddff2f86db",
        "jpg",
    ),
    (
        "american_goldfinch-38",
        "american_goldfinch",
        409778763,
        230765968,
        "squidtk",
        210988,
        "b92ccce145f7a320f25edf92633e0c56aa92e8c83890b4a439ce4b3d669cb1a1",
        "jpeg",
    ),
    (
        "american_goldfinch-39",
        "american_goldfinch",
        382773537,
        216432059,
        "rocksand2134",
        195020,
        "ad33d43d9e5650dd441bbd9adb9cc9854a6a1bcc7088a8802a7fb4ccc1ee8b51",
        "jpeg",
    ),
    (
        "american_goldfinch-40",
        "american_goldfinch",
        290287368,
        167563678,
        "sean579",
        105190,
        "8703f245d4b98476f67775ca0c19b7abd7c5edf756c50b2ff502b3d058c08014",
        "jpeg",
    ),
    (
        "american_goldfinch-41",
        "american_goldfinch",
        423112648,
        237722733,
        "andrew2285",
        55719,
        "10709f5df3f4c509afd260ba51c0db5cee89c7c9076c74e9911e9613303d7754",
        "jpeg",
    ),
    (
        "american_goldfinch-42",
        "american_goldfinch",
        422361237,
        237335568,
        "wildaboutwildlife",
        34855,
        "55b0962389b589037e03378472e65ffa0e7ed0fb7e227a7441106a009f093924",
        "jpg",
    ),
    (
        "american_goldfinch-43",
        "american_goldfinch",
        291016386,
        167956837,
        "samallonthesciencemon",
        128110,
        "489934bec3eb5682067cca11cb210c171198e616bb4920753b48bb0e1480bc32",
        "jpg",
    ),
    (
        "american_goldfinch-44",
        "american_goldfinch",
        419250127,
        235728207,
        "paulgraham",
        39211,
        "be1f9ae51a6c4ee92a543ecfe1654698581d7647a3a0b46859a7cf054d8b0fd9",
        "jpeg",
    ),
    (
        "american_goldfinch-45",
        "american_goldfinch",
        416004544,
        234039610,
        "randv",
        105453,
        "942a87723af8aedc86459479437bc474817f19429806f36d8e1236538448e24f",
        "jpeg",
    ),
    (
        "american_goldfinch-46",
        "american_goldfinch",
        416408092,
        234251817,
        "kmayner",
        143385,
        "55259e21e667b88864e2b88ba8dacc8ac5be0d2606699be35c46d2b17fe67fb2",
        "jpeg",
    ),
    (
        "american_goldfinch-47",
        "american_goldfinch",
        116188790,
        71368917,
        "leahmfulton",
        48440,
        "ae01ef8ba2fb714253543a5e773e4e047e95448f4ff4d1e51bc889a20b14e1bc",
        "jpg",
    ),
    (
        "american_goldfinch-48",
        "american_goldfinch",
        118530525,
        72675993,
        "seanwashington1",
        38505,
        "d688edde61a079907cdf5011a32914432f8f7f35c12d355e308aae29b1988738",
        "jpeg",
    ),
    (
        "american_goldfinch-49",
        "american_goldfinch",
        121016576,
        74071097,
        "geenance",
        110180,
        "fb4d1fef09311949043b3c9edc0d3a85d4c20386346093bb6ea39e1b95f756b9",
        "jpeg",
    ),
    (
        "american_goldfinch-50",
        "american_goldfinch",
        3279608,
        2872951,
        "joed",
        115964,
        "c2f1463bfef4c9b157f7e3ebd33fa37570592a4cdfa276f2b7c673f543cbc3ca",
        "jpeg",
    ),
    (
        "american_goldfinch-51",
        "american_goldfinch",
        118140282,
        72458899,
        "melaniegaddy",
        243713,
        "a3d70061032ac8c665f3d17a9118c8e592d53a3669182b23b502e3573a80d8f8",
        "jpg",
    ),
    (
        "american_goldfinch-52",
        "american_goldfinch",
        7669539,
        6125334,
        "cuihenggang",
        61287,
        "36d9570131caf33f3b5cfa6ab99a26c417e7cd8e38d0c1351d1a2938bb6e9bf9",
        "jpeg",
    ),
    (
        "american_goldfinch-53",
        "american_goldfinch",
        9258865,
        7182023,
        "fake_id",
        151987,
        "131d2a52097c9ab18fad318b62e8213c1bd33acddc72065ec0e404c76220c5ee",
        "jpeg",
    ),
    (
        "american_goldfinch-54",
        "american_goldfinch",
        12222432,
        9054511,
        "myacadianforest",
        107758,
        "69b1b5274d74bb28601afc2758e57a88f8d63743b5410c70284a9f4877c721fb",
        "jpeg",
    ),
    (
        "american_goldfinch-55",
        "american_goldfinch",
        128301260,
        78400716,
        "natepow",
        67416,
        "0c988080a47484802806ef14e0d1fdb9205680eacecedbe24f8bbac2aa2873f9",
        "jpg",
    ),
    (
        "american_goldfinch-56",
        "american_goldfinch",
        129842959,
        79296314,
        "nartb",
        117002,
        "f23b2e68f4c10e2679f0a9fe2f34a894bd2deea9c90c2143462e0aa5a83a3daa",
        "jpg",
    ),
    (
        "american_goldfinch-57",
        "american_goldfinch",
        78410307,
        49351195,
        "chrisleearm",
        108066,
        "8a4b273f1a1bd548c48abcf57b86ed35028cbfb949720180a43f24976a26ac8e",
        "jpg",
    ),
    (
        "american_goldfinch-58",
        "american_goldfinch",
        72300928,
        45598176,
        "henrya",
        68593,
        "a0c2869c330aaea08fba3bfeae99392c5588980b1ec21770c220c744125c3b42",
        "jpeg",
    ),
    (
        "american_goldfinch-59",
        "american_goldfinch",
        194724999,
        115367030,
        "vlatassa",
        159105,
        "c456a23238f8f81f93df7919b92dd3fa0123fa16a875c22ec1f9d0ae6ef17324",
        "jpeg",
    ),
)
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 36, "validation": 8, "test": 16}  # per species; 6 species -> 216 / 48 / 96
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MIN_CLASSES = 2
MAX_CLASSES = 100
MAX_LABEL_CHARS = 64
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_LABEL_RE = re.compile(r"^[A-Za-z0-9_ .:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def photo_url(photo_id: int, ext: str = "jpg") -> str:
    """The served object for a pinned photo; `ext` is its recorded original extension (jpg, jpeg or png,
    either case — the bucket key is case-sensitive)."""
    if ext.lower() not in ("jpg", "jpeg", "png"):
        raise ValueError(f"unsupported photo extension {ext!r}")
    return f"{CORPUS_BASE_URL}{photo_id}/medium.{ext}"


def observation_url(observation_id: int) -> str:
    return f"https://www.inaturalist.org/observations/{observation_id}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photo (bytes keyed by record id) from the cache or the open-data bucket."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _label, photo_id, _obs, _user, size, digest, ext in SAMPLE_RECORDS:
        local = cache / f"{photo_id}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(photo_id, ext)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "dimer-siglip2-tutorial/1.0"}
                )
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({photo_id}/medium.{ext}): fetched {len(data)} bytes with sha256 "
                    f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photo bytes into `{id, image, label}` records with their provenance."""
    out = []
    for rid, label, photo_id, obs_id, user, _size, _digest, _ext in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "label": label,
                "scientific_name": SPECIES[label][0],
                "common_name": SPECIES[label][1],
                "inat_photo_id": photo_id,
                "inat_observation_url": observation_url(obs_id),
                "observer": user,
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw per species: `sizes` counts per class for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record["label"]), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            out[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    for name in out:
        rng.shuffle(out[name])
        out[name] = [
            {**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int, *, require_labels: bool = True) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image/label")
    for key in ("id", "image") + (("label",) if require_labels else ()):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image, label = record["id"], record["image"], record.get("label")
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(
            f"{label_name}: image side outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}"
        )
    if label is None and not require_labels:
        item: dict[str, Any] = {"id": rid, "image": image.convert("RGB")}
    else:
        if not isinstance(label, str) or not _LABEL_RE.match(label.strip()):
            raise ValueError(
                f"{label_name}: label must be a non-empty string of at most {MAX_LABEL_CHARS} plain characters"
            )
        item = {"id": rid, "image": image.convert("RGB"), "label": label.strip()}
    for key in (
        "source_id",
        "observer",
        "inat_photo_id",
        "inat_observation_url",
        "scientific_name",
        "common_name",
    ):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of an image dataset; raises ValueError before any model import. Labels are
    required for the probe (`require_labels=True`) and optional for the reconstruction contract, which
    never reads them; a record that carries a label is always checked."""
    if (
        isinstance(records, Mapping)
        or not isinstance(records, Sequence)
        or isinstance(records, (str, bytes))
    ):
        raise ValueError("records must be a list of {id, image, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    counts: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index, require_labels=require_labels)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if "label" in item:
            counts[item["label"]] = counts.get(item["label"], 0) + 1
        checked.append(item)
    if require_labels and not MIN_CLASSES <= len(counts) <= MAX_CLASSES:
        raise ValueError(
            f"{len(counts)} distinct labels; {MIN_CLASSES}..{MAX_CLASSES} are required"
        )
    sides = [max(r["image"].size) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "classes": sorted(counts),
        "label_counts": dict(sorted(counts.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def class_names(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted label vocabulary of a dataset (the `classes` a head is trained for)."""
    names = sorted({str(r["label"]) for r in records})
    if len(names) < MIN_CLASSES:
        raise ValueError(f"a dataset needs at least {MIN_CLASSES} distinct labels")
    return names


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], image_digest(r["image"]), r.get("label")] for r in records]
    return _sha256_bytes(
        json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    )


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def observer_overlap(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:
    """How many observers contributed photos to more than one split (an observation, not an assertion)."""
    seen: dict[str, set[str]] = {}
    for name, records in splits.items():
        for record in records:
            if record.get("observer"):
                seen.setdefault(str(record["observer"]), set()).add(name)
    return {
        "observers": len(seen),
        "in_more_than_one_split": sum(1 for s in seen.values() if len(s) > 1),
    }


def _split_counts(per_label: int, val_fraction: float, test_fraction: float) -> tuple[int, int, int]:
    """(train, validation, test) that `split_dataset` cuts from one label holding `per_label` photographs."""
    n_test = max(1, round(per_label * test_fraction))
    n_val = round(per_label * val_fraction)
    return per_label - n_test - n_val, n_val, n_test


def min_byod_records(
    n_labels: int = 2, *, val_fraction: float = 0.15, test_fraction: float = 0.2
) -> dict[str, int]:
    """The smallest balanced dataset `split_dataset` accepts with `n_labels` labels and these fractions.

    Every label needs at least one training, one test and (when `val_fraction` > 0) one validation photograph,
    and the training split needs MIN_RECORDS in total. With the default fractions two labels need 12
    photographs (6 per label, split 4 / 1 / 1), three need 15 (5 per label), and four or more need 4 per
    label."""
    if isinstance(n_labels, bool) or not isinstance(n_labels, int) or n_labels < MIN_CLASSES:
        raise ValueError(f"n_labels must be an int >= {MIN_CLASSES}")
    for per_label in range(1, MAX_RECORDS + 1):
        train, val, test = _split_counts(per_label, val_fraction, test_fraction)
        if train >= 1 and (val >= 1 or val_fraction == 0) and train * n_labels >= MIN_RECORDS:
            return {
                "per_label": per_label,
                "total": per_label * n_labels,
                "train_per_label": train,
                "validation_per_label": val,
                "test_per_label": test,
            }
    raise ValueError("no dataset size satisfies these fractions")


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (
        0.0 <= val_fraction < 1.0
        and 0.0 < test_fraction < 1.0
        and val_fraction + test_fraction < 1.0
    ):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_label.setdefault(record["label"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        _n_train, n_val, n_test = _split_counts(len(pool), val_fraction, test_fraction)
        splits["test"].extend(pool[:n_test])
        splits["validation"].extend(pool[n_test : n_test + n_val])
        splits["train"].extend(pool[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    need = min_byod_records(
        max(len(by_label), MIN_CLASSES), val_fraction=val_fraction, test_fraction=test_fraction
    )
    advice = (
        f"with {len(by_label)} labels and these split fractions supply at least {need['total']} distinct "
        f"photographs, {need['per_label']} per label"
    )
    for part in ("train", "validation") if val_fraction > 0 else ("train",):
        present = {r["label"] for r in splits[part]}
        starved = sorted(label for label in by_label if label not in present)
        if starved:
            raise ValueError(
                f"split leaves no {part} photograph for labels {starved[:5]} "
                f"({', '.join(f'{k}: {len(by_label[k])}' for k in starved[:5])} distinct photographs); "
                f"{advice}"
            )
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required "
            f"({advice})"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label}` records from a directory or a zip holding `labels.csv` (columns `id`, `file`,
    `label`) beside the image files; images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        if not (source / "labels.csv").is_file():
            raise ValueError("BYOD folder must contain labels.csv")
        table = (source / "labels.csv").read_text(encoding="utf-8-sig")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        names = [
            n
            for n in archive.namelist()
            if not n.endswith("/")
            and not any(part == "__MACOSX" or part.startswith(".") for part in Path(n).parts)
        ]
        members = {Path(n).name: n for n in names}
        if "labels.csv" not in members:
            raise ValueError("BYOD zip must contain labels.csv")
        table = archive.read(members["labels.csv"]).decode("utf-8-sig")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding labels.csv and the image files")
    reader = csv.DictReader(io.StringIO(table))
    missing = {"id", "file", "label"} - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"labels.csv is missing columns {sorted(missing)} (it needs id, file, label)")
    rows = list(reader)
    if not rows:
        raise ValueError("labels.csv has no data rows: add one row per photograph (id, file, label)")
    out = []
    for line, row in enumerate(rows, start=2):  # line 1 is the header
        rid, name, label = ((row.get(k) or "").strip() for k in ("id", "file", "label"))
        where = f"labels.csv line {line} (file {name!r})"
        if not _ID_RE.match(rid):
            raise ValueError(f"{where}: id {rid!r} must match {_ID_RE.pattern}")
        if not _LABEL_RE.match(label):
            raise ValueError(
                f"{where}: label {label!r} must be 1..{MAX_LABEL_CHARS} plain characters "
                "(letters, digits, space and _ . : -)"
            )
        try:
            image = loader(name)
            if max(image.size) > MAX_IMAGE_SIDE:
                raise ValueError(
                    f"{where}: image side {max(image.size)} px is over MAX_IMAGE_SIDE={MAX_IMAGE_SIDE}"
                )
            image.load()
        except (KeyError, FileNotFoundError, IsADirectoryError):
            raise ValueError(
                f"{where}: that image file is not in the dataset; every labels.csv row must name "
                "an image file stored beside labels.csv"
            ) from None
        except ValueError:
            raise
        except (OSError, SyntaxError, Image.DecompressionBombError) as exc:
            raise ValueError(f"{where}: Pillow cannot decode the image ({exc})") from None
        out.append({"id": rid, "image": image.convert("RGB"), "label": label})
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the labels table of a split (id, file, label, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=["id", "file", "label", "observer", "inat_observation_url"]
        )
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": f"{record['inat_photo_id']}.jpg"
                    if record.get("inat_photo_id")
                    else record["id"],
                    "label": record["label"],
                    "observer": record.get("observer", ""),
                    "inat_observation_url": record.get("inat_observation_url", ""),
                }
            )
    return out

**Module 6/6:** `src/vit_mae_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""ViT-MAE base: masked-image reconstruction with the pinned ``facebook/vit-mae-base`` weights, a
reconstruction-supervised evaluation contract, a linear-probe reading of the encoder, and a bounded
continuation of the masked-autoencoding pretraining on a labelled photograph set.

The checkpoint is the full pretraining model (``ViTMAEForPreTraining``: a ViT-B/16 encoder and an
8-layer, 512-wide decoder). Its one user-facing operation is **reconstruction**: a seeded random 75 % of the
196 patches is hidden from the encoder, the decoder predicts every patch's pixels, and the loss is the mean
squared error on the hidden patches — the model's own training objective, which is also the only number it
can be read on without labels. ``embed`` exposes the encoder as a feature extractor (mean of the patch
tokens with nothing hidden, L2-normalised), and ``fit_probe`` / ``classify`` / ``evaluate`` read those
features through a linear head the way the fleet reads self-supervised encoders. ``adapt`` continues the
masked-autoencoding objective on the caller's photographs — the decoder and the last encoder blocks — with
validation-loss epoch selection, and the probe is fitted again afterwards so the effect of that continuation
on the features is measured, not assumed.
"""
# ruff: noqa: E501  -- docstrings and record literals kept on single lines at the fleet width

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from io import BytesIO
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .metrics import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import load_components, stage_missing_files, verify_snapshot` removed — names are kernel globals defined by the carried modules

ImageInput = str | Path | bytes | Image.Image

MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MIN_IMAGE_SIDE = 16
MAX_BATCH = 64  # images per embed() / reconstruct() call
DEFAULT_TRAINABLE_BLOCKS = 2  # the last two encoder blocks + the encoder LayerNorm train beside the whole decoder
PARAMETER_COUNT = 111_907_840
ENCODER_PARAMETERS = 85_798_656
DECODER_PARAMETERS = 26_109_184
PROBE_STEPS = 300
PROBE_LR = 0.01
PROBE_WEIGHT_DECAY = 1e-3
KNN_K = 5
MAX_EVAL_RECORDS = 5_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.vit-mae-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
GRID = IMAGE_SIZE // PATCH_SIZE

INPUT_SCHEMA: dict[str, Any] = {
    "images": (
        "PIL.Image.Image, raw bytes, or a local path decodable by Pillow; any mode, converted to RGB; "
        "remote URLs are refused"
    ),
    "image_size": (
        f"sides in [{MIN_IMAGE_SIDE}, {MAX_IMAGE_SIDE}] px; the processor resizes every image to {IMAGE_SIZE} x {IMAGE_SIZE} "
        f"(aspect ratio not kept) and normalises with the ImageNet mean / std"
    ),
    "mask_ratio": f"fraction of the {NUM_PATCHES} patches hidden from the encoder, in (0, 1); default {DEFAULT_MASK_RATIO}; the mask is seeded",
    "output": (
        "reconstruct: per image the masked-patch MSE (the model's loss), the visible-patch MSE, the mask, and the "
        "reconstructed image (decoder prediction on hidden patches, the input on visible ones); embed: one "
        f"{HIDDEN_SIZE}-d L2-normalised vector per image; classify (after fit_probe): a score per class"
    ),
    "validation": "size and decodability only; nothing checks what the image shows",
}


def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _coerce_image(value: ImageInput) -> Image.Image:
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, bytes):
        with Image.open(BytesIO(value)) as handle:
            handle.load()
            return handle.convert("RGB")
    text = str(value)
    if text.startswith(("http://", "https://")):
        raise ValueError("remote URLs are not accepted; pass a local path, bytes or a PIL image")
    path = Path(text)
    if not path.is_file():
        raise FileNotFoundError(f"image not found: {path}")
    with Image.open(path) as handle:
        handle.load()
        return handle.convert("RGB")


def _check_size(image: Image.Image, what: str) -> None:
    w, h = image.size
    if min(w, h) < MIN_IMAGE_SIDE or max(w, h) > MAX_IMAGE_SIDE:
        raise ValueError(f"{what}: sides must be in [{MIN_IMAGE_SIDE}, {MAX_IMAGE_SIDE}] px, got {w} x {h}")


def _check_images(images: Sequence[ImageInput] | ImageInput) -> list[Image.Image]:
    if isinstance(images, str | Path | bytes | Image.Image):
        images = [images]
    if not images:
        raise ValueError("images must contain at least one image")
    if len(images) > MAX_BATCH:
        raise ValueError(f"at most {MAX_BATCH} images per call")
    out = []
    for index, image in enumerate(images):
        coerced = _coerce_image(image)
        _check_size(coerced, f"image {index}")
        out.append(coerced)
    return out


def _check_mask_ratio(mask_ratio: float) -> float:
    if isinstance(mask_ratio, bool) or not isinstance(mask_ratio, int | float) or not 0.0 < float(mask_ratio) < 1.0:
        raise ValueError("mask_ratio must be a number in (0, 1)")
    return float(mask_ratio)


def validate_inputs(
    images: Sequence[ImageInput] | ImageInput,
    *,
    mask_ratio: float = DEFAULT_MASK_RATIO,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: the input manifest (schema, per-image observations, verdict), raising exactly what
    `reconstruct` / `embed` would raise."""
    coerced = _check_images(images)
    ratio = _check_mask_ratio(mask_ratio)
    if names is not None and len(names) != len(coerced):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size)} for i, image in enumerate(coerced)],
        "mask_ratio": ratio,
        "hidden_patches": int(NUM_PATCHES * ratio),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def _seeded_noise(seed: int, count: int, generator_device: str = "cpu") -> torch.Tensor:
    """The random argument HF's ViTMAE uses to choose the hidden patches, seeded per image."""
    g = torch.Generator(device=generator_device).manual_seed(int(seed))
    return torch.rand(count, NUM_PATCHES, generator=g)


def _record_seed(record_id: str, seed: int) -> int:
    digest = hashlib.sha256(f"{seed}:{record_id}".encode()).digest()
    return int.from_bytes(digest[:4], "big")


class ViTMAEPipeline:
    def __init__(
        self,
        model: Any,
        processor: Any,
        *,
        device: str | torch.device = "cpu",
        checkpoint_path: Path | str | None = None,
        checkpoint_source: str | None = None,
        manifest_verified: bool = False,
        weight_sha256: str | None = None,
        weight_size_bytes: int | None = None,
    ) -> None:
        self.model = model
        self.processor = processor
        self.device = torch.device(device)
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.checkpoint_source = checkpoint_source
        self.manifest_verified = manifest_verified
        self.weight_sha256 = weight_sha256
        self.weight_size_bytes = weight_size_bytes
        self.adapter: dict[str, Any] | None = None
        self.probe: dict[str, Any] | None = None  # {"classes", "weight", "bias", ...} after fit_probe
        # Pinned-base values of every tensor adapt() or load_artifact() has changed, kept the first time each is
        # about to change: every adaptation starts from the verified base, never from a previous run (MAE-M2).
        self._base_state: dict[str, Any] = {}
        if hasattr(self.model, "parameters"):  # injected fakes in the offline tests carry none
            self.model.eval()
            for param in self.model.parameters():
                param.requires_grad_(False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | torch.device | None = None,
        cache_dir: str | Path | None = None,
        weights_path: str | Path | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ViTMAEPipeline:
        """Load the one supported checkpoint (fleet snapshot directory → stage absent entries when allowed →
        verify against the manifest and the pinned digests → load the safetensors file strictly)."""
        if weights_dir is not None:
            if weights_path is not None:
                raise ValueError("pass either weights_dir or weights_path, not both")
            stage_missing_files(weights_dir, allow_download=allow_download)
            verify_snapshot(weights_dir)
            weights_path = weights_dir
        model, processor, target_device, _, metadata = load_components(device=device, cache_dir=cache_dir, weights_path=weights_path, return_metadata=True)
        return cls(
            model,
            processor,
            device=target_device,
            checkpoint_path=metadata.get("checkpoint_path"),
            checkpoint_source=metadata.get("checkpoint_source"),
            manifest_verified=metadata.get("manifest_verified", False),
            weight_sha256=metadata.get("weight_sha256"),
            weight_size_bytes=metadata.get("weight_size_bytes"),
        )

    # ------------------------------------------------------------------ tensors

    def _pixels(self, images: Sequence[Image.Image]) -> torch.Tensor:
        return self.processor(images=list(images), return_tensors="pt")["pixel_values"].to(self.device)

    def _patchify(self, pixels: torch.Tensor) -> torch.Tensor:
        return self.model.patchify(pixels)

    def _forward(self, pixels: torch.Tensor, noise: torch.Tensor, mask_ratio: float) -> Any:
        """One masked forward with the given noise; the mask ratio is set on the config for the call."""
        previous = self.model.config.mask_ratio
        self.model.config.mask_ratio = mask_ratio
        try:
            return self.model(pixel_values=pixels, noise=noise.to(pixels.device))
        finally:
            self.model.config.mask_ratio = previous

    # ------------------------------------------------------------------ inference contract

    def reconstruct(
        self,
        images: Sequence[ImageInput] | ImageInput,
        *,
        mask_ratio: float = DEFAULT_MASK_RATIO,
        seed: int = 0,
        return_images: bool = True,
    ) -> dict[str, Any]:
        """Hide a seeded random `mask_ratio` of the patches, reconstruct, and score against the true pixels."""
        coerced = _check_images(images)
        ratio = _check_mask_ratio(mask_ratio)
        pixels = self._pixels(coerced)
        noise = _seeded_noise(seed, len(coerced))
        with torch.no_grad():
            out = self._forward(pixels, noise, ratio)
            target = self._patchify(pixels)
            pred = out.logits
            mask = out.mask
            filled = pred * mask.unsqueeze(-1) + target * (1 - mask.unsqueeze(-1))
            recon = self.model.unpatchify(filled) if return_images else None
        results = []
        target_np, pred_np, mask_np = target.cpu().numpy(), pred.float().cpu().numpy(), mask.cpu().numpy()
        for i, image in enumerate(coerced):
            masked = masked_mse(pred_np[i], target_np[i], mask_np[i])
            visible = masked_mse(pred_np[i], target_np[i], 1.0 - mask_np[i])
            entry: dict[str, Any] = {
                "input_size": list(image.size),
                "mask_ratio": ratio,
                "hidden_patches": int(mask_np[i].sum()),
                "masked_mse": masked,
                "visible_mse": visible,
                "mask": mask_np[i].astype(bool).tolist(),
            }
            if return_images:
                entry["reconstruction"] = self._to_image(recon[i])
            results.append(entry)
        return {
            "results": results,
            "model_loss": float(out.loss) if out.loss is not None else None,
            "mask_ratio": ratio,
            "seed": int(seed),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_license": MODEL_LICENSE,
        }

    def _to_image(self, pixels: torch.Tensor) -> Image.Image:
        mean = torch.tensor(self.processor.image_mean, dtype=pixels.dtype, device=pixels.device).view(3, 1, 1)
        std = torch.tensor(self.processor.image_std, dtype=pixels.dtype, device=pixels.device).view(3, 1, 1)
        array = ((pixels * std + mean).clamp(0, 1) * 255).round().byte().permute(1, 2, 0).cpu().numpy()
        return Image.fromarray(array)

    def embed(self, images: Sequence[ImageInput] | ImageInput) -> dict[str, Any]:
        """Encoder features with nothing hidden: the mean of the patch tokens after the final LayerNorm, L2-normalised."""
        coerced = _check_images(images)
        with torch.no_grad():
            features = self._features(self._pixels(coerced))
        return {"embeddings": features.cpu().numpy(), "dim": HIDDEN_SIZE, "normalized": True, "pooling": "mean of patch tokens, nothing hidden", "model_id": MODEL_ID, "model_revision": MODEL_REVISION}

    def _features(self, pixels: torch.Tensor) -> torch.Tensor:
        previous = self.model.config.mask_ratio
        self.model.config.mask_ratio = 0.0
        try:
            hidden = self.model.vit(pixel_values=pixels).last_hidden_state
        finally:
            self.model.config.mask_ratio = previous
        pooled = hidden[:, 1:, :].mean(dim=1)
        return torch.nn.functional.normalize(pooled.float(), dim=-1)

    def features(self, records: Sequence[Mapping[str, Any]], *, batch_size: int = 16) -> np.ndarray:
        out = []
        for start in range(0, len(records), batch_size):
            chunk = [_coerce_image(r["image"]) for r in records[start : start + batch_size]]
            with torch.no_grad():
                out.append(self._features(self._pixels(chunk)).cpu().numpy())
        return np.concatenate(out, axis=0) if out else np.zeros((0, HIDDEN_SIZE), dtype=np.float32)

    # ------------------------------------------------------------------ reconstruction evaluation

    def evaluate_reconstruction(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        mask_ratio: float = DEFAULT_MASK_RATIO,
        seed: int = 0,
        batch_size: int = 16,
    ) -> dict[str, Any]:
        """Masked-patch MSE over a validated `{id, image, ...}` set with one seeded mask per record id, beside the
        mean-patch and blur fills scored on the same masks."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS, require_labels=False)["records"]
        ratio = _check_mask_ratio(mask_ratio)
        rows, mean_rows, blur_rows = [], [], []
        for start in range(0, len(checked), batch_size):
            chunk = checked[start : start + batch_size]
            pixels = self._pixels([_coerce_image(r["image"]) for r in chunk])
            noise = torch.stack([_seeded_noise(_record_seed(str(r["id"]), seed), 1)[0] for r in chunk])
            with torch.no_grad():
                out = self._forward(pixels, noise, ratio)
                target = self._patchify(pixels).cpu().numpy()
            pred, mask = out.logits.float().cpu().numpy(), out.mask.cpu().numpy()
            for i, record in enumerate(chunk):
                rows.append({"id": str(record["id"]), "masked_mse": masked_mse(pred[i], target[i], mask[i]), "visible_mse": masked_mse(pred[i], target[i], 1.0 - mask[i]), "hidden_patches": int(mask[i].sum())})
                mean_rows.append({"id": str(record["id"]), "masked_mse": masked_mse(mean_patch_fill(target[i], mask[i]), target[i], mask[i]), "visible_mse": 0.0})
                blur_rows.append({"id": str(record["id"]), "masked_mse": masked_mse(blur_fill(target[i], mask[i], grid=GRID), target[i], mask[i]), "visible_mse": 0.0})
        result = reconstruction_metrics(rows, mask_ratio=ratio)
        result["per_image"] = rows
        result["seed"] = int(seed)
        result["baselines"] = {
            "mean_patch_fill": {**reconstruction_metrics(mean_rows, mask_ratio=ratio), "baseline": "every hidden patch = the mean colour of the visible patches"},
            "blur_fill": {**reconstruction_metrics(blur_rows, mask_ratio=ratio), "baseline": "every hidden patch = the mean colour of its visible neighbours"},
        }
        result["verdict"] = "measured" if len(rows) >= MIN_SCORED_RECORDS else "small-sample"
        result["adapted"] = self.adapter is not None
        return result

    # ------------------------------------------------------------------ linear probe

    def fit_probe(
        self,
        train: Sequence[Mapping[str, Any]],
        *,
        steps: int = PROBE_STEPS,
        lr: float = PROBE_LR,
        weight_decay: float = PROBE_WEIGHT_DECAY,
        seed: int = 0,
    ) -> dict[str, Any]:
        """A multinomial logistic-regression head on the frozen, standardised features of a labelled set
        (full-batch Adam, seeded)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import class_names, validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(train, min_records=MIN_SCORED_RECORDS // 5)["records"]
        classes = class_names(checked)
        gold = torch.tensor([classes.index(str(r["label"])) for r in checked])
        started = time.perf_counter()
        raw = torch.from_numpy(self.features(checked)).float()
        # The MAE paper probes through a BatchNorm without affine parameters; the same standardisation with the
        # training set's statistics, applied to every query, is what makes a linear head on MAE features usable.
        mean, std = raw.mean(dim=0), raw.std(dim=0) + 1e-6
        feats = (raw - mean) / std
        torch.manual_seed(seed)
        head = torch.nn.Linear(HIDDEN_SIZE, len(classes))
        optimiser = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=weight_decay)
        losses = []
        for _ in range(steps):
            optimiser.zero_grad()
            loss = torch.nn.functional.cross_entropy(head(feats), gold)
            loss.backward()
            optimiser.step()
            losses.append(float(loss.detach()))
        self.probe = {
            "classes": classes,
            "weight": head.weight.detach().clone(),
            "bias": head.bias.detach().clone(),
            "feature_mean": mean,
            "feature_std": std,
            "steps": steps,
            "lr": lr,
            "weight_decay": weight_decay,
            "n_train": len(checked),
            "train_loss": [losses[0], losses[-1]],
            "standardisation": "training-set mean / std per feature (the paper's affine-free BatchNorm)",
            "seconds": round(time.perf_counter() - started, 2),
            "train_features": raw.numpy(),
            "train_gold": gold.numpy(),
        }
        return {k: v for k, v in self.probe.items() if k not in ("weight", "bias", "feature_mean", "feature_std", "train_features", "train_gold")}

    def _scores(self, features: np.ndarray) -> np.ndarray:
        if self.probe is None:
            raise ValueError("no probe fitted: call fit_probe() first")
        feats = (torch.from_numpy(np.asarray(features)).float() - self.probe["feature_mean"]) / self.probe["feature_std"]
        logits = feats @ self.probe["weight"].T + self.probe["bias"]
        return torch.softmax(logits, dim=-1).numpy()

    def classify(self, images: Sequence[ImageInput] | ImageInput) -> dict[str, Any]:
        """Scores per class from the fitted probe on the encoder features."""
        coerced = _check_images(images)
        with torch.no_grad():
            feats = self._features(self._pixels(coerced)).cpu().numpy()
        scores = self._scores(feats)
        classes = self.probe["classes"]
        return {
            "classes": list(classes),
            "results": [{"top1": classes[int(row.argmax())], "scores": {c: float(row[j]) for j, c in enumerate(classes)}} for row in scores],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]], *, k: int = KNN_K) -> dict[str, Any]:
        """Probe accuracy / macro F1 on a validated labelled set, with a k-NN on the same features beside it."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self.probe is None:
            raise ValueError("no probe fitted: call fit_probe() first")
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        classes = list(self.probe["classes"])
        for record in checked:
            if str(record["label"]) not in classes:
                raise ValueError(f"record label {record['label']!r} is not a probe class")
        gold = [classes.index(str(r["label"])) for r in checked]
        feats = self.features(checked)
        probe = classification_metrics(self._scores(feats), gold, classes)
        if self.probe.get("train_features") is not None:
            knn = classification_metrics(knn_scores(self.probe["train_features"], self.probe["train_gold"], feats, len(classes), k=k), gold, classes)
            knn["baseline"] = f"{k}-nearest-neighbour vote on the same features"
            probe["knn"] = knn
        else:
            probe["knn"] = None  # a reloaded probe carries its head and statistics, not the training features
        probe["classes"] = classes
        probe["predictions"] = [classes[int(row.argmax())] for row in self._scores(feats)]
        probe["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "small-sample"
        probe["adapted"] = self.adapter is not None
        return probe

    # ------------------------------------------------------------------ adaptation contract

    def _remember_base(self, names: Sequence[str]) -> None:
        state = self.model.state_dict()
        for name in names:
            if name not in self._base_state:
                self._base_state[name] = state[name].detach().clone()

    def restore_base(self) -> list[str]:
        """Put the pipeline back to the pinned base: copy the base values into every tensor an earlier adapt() or
        load_artifact() changed and drop the adapter record and the probe, so `reconstruct`, `features` and a new
        adapt() read the untouched checkpoint. Returns the names of the tensors that differed from the base."""
        state = self.model.state_dict()
        # Only tensors whose live value differs from the base count as restored, so a call on an untouched
        # model reports nothing (t5-base-text2text-pipeline 93a578f).
        restored = sorted(n for n, base in self._base_state.items() if not bool((state[n] == base).all()))
        if restored:
            self.model.load_state_dict({**state, **self._base_state}, strict=True)
            self.model.eval()
        self.adapter, self.probe = None, None
        return restored

    def _trainable_names(self, trainable_blocks: int) -> list[str]:
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int) or not 0 <= trainable_blocks <= ENCODER_LAYERS:
            raise ValueError(f"trainable_blocks must be an int in 0..{ENCODER_LAYERS}")
        first = ENCODER_LAYERS - trainable_blocks
        prefixes = tuple(f"vit.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS)) + ("decoder.",)
        if trainable_blocks:
            prefixes += ("vit.layernorm.",)
        return [name for name, _p in self.model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 5,
        lr: float = 1e-5,
        batch_size: int = 8,
        trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,
        mask_ratio: float = DEFAULT_MASK_RATIO,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded continuation of the masked-autoencoding objective on validated photographs.

        Trains the decoder and the last `trainable_blocks` encoder blocks (with the encoder LayerNorm) on the
        checkpoint's own loss — the masked-patch MSE under a fresh random mask every step (`mask_ratio`) — with
        AdamW at a fixed learning rate, gradient clipping at 1.0, seeded order and masks, no scheduler. Epoch 0
        records the frozen model's validation reconstruction; every epoch is scored on the validation set with
        its fixed seeded masks, and the epoch with the lowest validation masked MSE is kept. Labels are not used.
        Every call starts from the pinned base: tensors an earlier adapt() or load_artifact() changed are restored
        first, so epoch 0 is the frozen model whatever ran before. Transactional: any failure puts back the
        weights, adapter record and probe this call found. The fitted probe (if any) is discarded on success,
        because the features it was fitted on no longer exist."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        ratio = _check_mask_ratio(mask_ratio)
        names = self._trainable_names(trainable_blocks)
        train_checked = validate_dataset(train, require_labels=False)["records"]
        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS, require_labels=False)["records"] if val else []
        model = self.model
        # The weights, adapter and probe as this call found them: a failed call puts them back (the transactional
        # contract), while a successful one starts from the pinned base.
        current = model.state_dict()
        previous_state = {k: current[k].detach().clone() for k in self._base_state}
        previous_adapter, previous_probe = self.adapter, self.probe
        restored = self.restore_base()
        self._remember_base(names)
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
        started = time.perf_counter()
        pixels_all = torch.cat([self._pixels([_coerce_image(r["image"])]).cpu() for r in train_checked])

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            result = self.evaluate_reconstruction(val_checked, mask_ratio=ratio, seed=seed)
            return {k: result[k] for k in ("masked_mse", "masked_mse_visible", "psnr_masked", "n")}

        def key(entry: dict[str, Any]) -> float:
            return -entry["val"]["masked_mse"] if entry["val"] else -math.inf

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    chunk = order[start : start + batch_size]
                    pixels = pixels_all[chunk].to(self.device)
                    noise = torch.rand(len(chunk), NUM_PATCHES, generator=generator)
                    optimiser.zero_grad(set_to_none=True)
                    loss = self._forward(pixels, noise, ratio).loss
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": float(np.mean(losses)), "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked:
                    if key(entry) > key(history[best_epoch]):
                        best_epoch = epoch
                        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                else:
                    best_epoch = epoch
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
            if progress is not None and history:
                pass
        except BaseException:
            model.load_state_dict({**model.state_dict(), **initial_state, **previous_state}, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter, self.probe = previous_adapter, previous_probe
            raise
        model.load_state_dict({**model.state_dict(), **best_state}, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.probe = None
        self.adapter = {
            "objective": "masked-autoencoding continuation (masked-patch MSE, the checkpoint's own loss)",
            "mask_ratio": ratio,
            "trainable_blocks": trainable_blocks,
            "n_trainable": int(n_trainable),
            "n_total": int(sum(p.numel() for p in model.parameters())),
            "epochs": epochs,
            "lr": lr,
            "batch_size": batch_size,
            "seed": seed,
            "started_from": "pinned base"
            + (f" (restored {len(restored)} tensors changed by an earlier run)" if restored else ""),
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "best_epoch": best_epoch,
            "selection": "lowest validation masked MSE (validation split, fixed seeded masks)" if val_checked else "final epoch (no validation split)",
            "seconds": round(time.perf_counter() - started, 2),
            "history": history,
            "trainable_names": names,
        }
        return dict(self.adapter)

    # ------------------------------------------------------------------ artifact

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors (and the fitted probe, if any) as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        probe_record = None
        if self.probe is not None:
            tensors["probe.weight"] = self.probe["weight"].detach().cpu().contiguous()
            tensors["probe.bias"] = self.probe["bias"].detach().cpu().contiguous()
            tensors["probe.feature_mean"] = self.probe["feature_mean"].detach().cpu().contiguous()
            tensors["probe.feature_std"] = self.probe["feature_std"].detach().cpu().contiguous()
            probe_record = {k: v for k, v in self.probe.items() if k not in ("weight", "bias", "feature_mean", "feature_std", "train_features", "train_gold")}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": DEFAULT_MODEL_KEY, "weight_file": MODEL_FILENAME, "weight_sha256": MODEL_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "probe": probe_record,
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [{"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries (and restore the probe head when the artifact carries one)."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = _check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_blocks"]))
        probe_record = manifest.get("probe")
        if probe_record:
            expected = sorted(expected + ["probe.weight", "probe.bias", "probe.feature_mean", "probe.feature_std"])
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = self.model.state_dict()
        model_tensors = {k: v for k, v in tensors.items() if not k.startswith("probe.")}
        for key, value in model_tensors.items():
            if key not in state or not key.startswith(("vit.encoder.layer.", "vit.layernorm.", "decoder.")):
                raise ValueError(f"artifact tensor {key} is not an adaptable tensor of the base")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}")
        self.restore_base()
        self._remember_base(sorted(model_tensors))
        state = self.model.state_dict()
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in model_tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": [k for k in manifest["tensors"] if not k.startswith("probe.")], "history": manifest.get("history", [])}
        self.probe = None
        if probe_record:
            weight, bias = tensors["probe.weight"], tensors["probe.bias"]
            mean, std = tensors["probe.feature_mean"], tensors["probe.feature_std"]
            if tuple(weight.shape) != (len(probe_record["classes"]), HIDDEN_SIZE) or tuple(bias.shape) != (len(probe_record["classes"]),) or tuple(mean.shape) != (HIDDEN_SIZE,) or tuple(std.shape) != (HIDDEN_SIZE,):
                raise ValueError("artifact probe head has an unexpected shape")
            self.probe = {**probe_record, "weight": weight.float(), "bias": bias.float(), "feature_mean": mean.float(), "feature_std": std.float(), "train_features": None, "train_gold": None}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | torch.device | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ViTMAEPipeline:
        """A fresh pipeline from the pinned base with an adapter overlaid."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def _check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> Path:
    """Refuse an artifact whose manifest is not exactly the one this package writes. Nothing is deserialised
    here; the digest check that follows detects corruption or drift relative to the adjacent manifest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
        raise ValueError(f"artifact format_version {manifest.get('format_version')!r} is not the supported {ARTIFACT_FORMAT_VERSION!r}")
    base = manifest.get("base_model", {})
    if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (MODEL_ID, MODEL_REVISION, MODEL_SHA256):
        raise ValueError("artifact was adapted from a different base model, revision or weight file")
    if base.get("weight_file", MODEL_FILENAME) != MODEL_FILENAME:
        raise ValueError("artifact was adapted from a different base weight file")
    files = manifest.get("files")
    if not isinstance(files, list) or len(files) != 1:
        raise ValueError("artifact manifest must list exactly one file")
    entry = files[0]
    if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
        raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
    weights_path = (root / entry["path"]).resolve()
    if weights_path.parent != root.resolve():
        raise ValueError("artifact weight path must resolve inside the artifact directory")
    adapter = manifest.get("adapter")
    blocks = adapter.get("trainable_blocks") if isinstance(adapter, Mapping) else None
    if isinstance(blocks, bool) or not isinstance(blocks, int) or not 0 <= blocks <= ENCODER_LAYERS:
        raise ValueError("artifact manifest does not record an in-range integer trainable_blocks")
    probe = manifest.get("probe")
    if probe is not None and (not isinstance(probe, Mapping) or not isinstance(probe.get("classes"), list) or not probe["classes"]):
        raise ValueError("artifact probe record must list its classes")
    if not isinstance(manifest.get("tensors"), list):
        raise ValueError("artifact manifest must list its tensors")
    return weights_path


def load_pipeline(**kwargs: Any) -> ViTMAEPipeline:
    return ViTMAEPipeline.from_pretrained(**kwargs)


def evaluation_report(result: Mapping[str, Any], *, sample_kind: str = "synthetic") -> dict[str, Any]:
    """The per-call reading of a `reconstruct` result: masked and visible MSE per image and their means, with a
    `sample-sanity` verdict on a drawing (plumbing evidence) or `measured` on a scored set."""
    rows = result.get("results", [])
    if not rows:
        raise ValueError("no results to report")
    masked = [float(r["masked_mse"]) for r in rows]
    visible = [float(r["visible_mse"]) for r in rows]
    metrics = [
        {"id": "masked_mse", "value": float(np.mean(masked)), "definition": RECONSTRUCTION_DEFINITIONS["masked_mse"]},
        {"id": "masked_mse_visible", "value": float(np.mean(visible)), "definition": RECONSTRUCTION_DEFINITIONS["masked_mse_visible"]},
        {"id": "mask_ratio", "value": float(result.get("mask_ratio", DEFAULT_MASK_RATIO)), "definition": RECONSTRUCTION_DEFINITIONS["mask_ratio"]},
    ]
    return {
        "metrics": metrics,
        "n": len(rows),
        "sample_kind": sample_kind,
        "verdict": "sample-sanity" if sample_kind.startswith("synthetic") or len(rows) < MIN_SCORED_RECORDS else "measured",
        "note": "a reconstruction loss is the model's own objective, not a quality judgement; lower is better and nothing here is calibrated",
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


__all__ = [
    "ARTIFACT_FORMAT",
    "DEFAULT_TRAINABLE_BLOCKS",
    "INPUT_SCHEMA",
    "MAX_BATCH",
    "MAX_IMAGE_SIDE",
    "MIN_IMAGE_SIDE",
    "PARAMETER_COUNT",
    "ViTMAEPipeline",
    "evaluation_report",
    "load_pipeline",
    "validate_inputs",
]

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `25b184bea553…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ViTMAEPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "vit-mae-base",
  "modelId": "facebook/vit-mae-base",
  "revision": "25b184bea5538bf5c4c852c79d221195fdd2778d",
  "files": [
    {
      "path": "README.md",
      "bytes": 2968,
      "sha256": "16ae81a13455621be2a1e323fcef6824663c011ebd4e75ead22c5b8f13b9d2c5"
    },
    {
      "path": "config.json",
      "bytes": 676,
      "sha256": "1f7a507e1f7f63a262c5500b86470c3b22ff1a9e42de81bbba27105396572dae"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 217,
      "sha256": "a250969c94afba52d785a0e08dd36e13aeda97c4dd2b7fd0d24b457288536cea"
    },
    {
      "path": "model.safetensors",
      "bytes": 447670680,
      "sha256": "479dcef4bd5df06259399027b789f21e9d9a1b79f37155a64176d55bc26fdae8"
    }
  ],
  "totalBytes": 447674541
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ViTMAEPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'facebook/vit-mae-base', 'revision': '25b184bea5538bf5c4c852c79d221195fdd2778d', 'license': 'Apache-2.0', 'files': 4, 'total_bytes': 447674541}


{'weights_dir': '/content/weights/vit-mae-base', 'fetched': ['README.md', 'config.json', 'preprocessor_config.json', 'model.safetensors']}


{'verified_files': 4, 'revision': '25b184bea5538bf5c4c852c79d221195fdd2778d'}


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


{'device': device(type='cuda'), 'source': 'local-snapshot'}


## 4. iNaturalist photographs and split

`fetch_corpus` returns the 360 pinned photographs from the cache under `weights/inat-birds/` or the iNaturalist open-data bucket — every cached file is re-hashed and every fetched file refused on any byte-size or SHA-256 mismatch — and `read_corpus` decodes them into `{id, image, label}` records with their observation page, observer and species names. `build_sample_dataset` draws a seeded stratified split per species (36 / 8 / 16 → 216 / 48 / 96). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) is shared, `observer_overlap` reports how many observers contributed to more than one split (an observation about the draw, not an assertion), and the training split's labels table is written to `outputs/vit_mae_pretraining_train.csv` in the shape BYOD expects. The labels are for the probe in Sections 6 and 8; the continuation in Section 7 never reads them.

Look for: 360 photographs, the six species with 36 / 8 / 16 each, three digests, and four refusal probes — a duplicate id, an image over the side ceiling, a dataset with one label and one too small to split — each rejected before the model does anything.

*Evaluation practice.* **Bring your own data (optional):** set `USE_BYOD = True` and either `BYOD_PATH` (a zip or a folder holding `labels.csv` and the images, as a path in this runtime — this works on Colab, Kaggle and Jupyter) or leave `BYOD_PATH` empty to upload exactly one zip through the Colab dialog; then choose **Run after** from this cell. This cell first puts the model back to the pinned base, so Section 6 reads the frozen model and Section 7's epoch 0 is the frozen model. The effective minimum is 12 photographs for two labels.

**Predict before running:** the reconstruction objective never reads the labels. Why split by species at all?

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
# A re-run after Section 7 (BYOD, or a new split): Section 6 and Section 7's epoch 0 must read the pinned base.
had_adapter = pipe.adapter is not None
restored_tensors = pipe.restore_base()
if had_adapter or restored_tensors:
    print({'restored_pinned_base': len(restored_tensors), 'note': 'the continued decoder and blocks were put back to the checkpoint and the probe was dropped; Sections 5-7 start from it again'})
if USE_BYOD:
    if BYOD_PATH.strip():
        byod_zip = Path(BYOD_PATH.strip()).expanduser()
        if not byod_zip.exists():
            raise FileNotFoundError(f'BYOD_PATH {BYOD_PATH!r} does not exist (relative paths start at {Path.cwd()}): give a .zip or a folder holding labels.csv and the images.')
        file_name = byod_zip.name
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('USE_BYOD is True but BYOD_PATH is empty, and the upload dialog exists only in Google Colab: on Kaggle or Jupyter put the zip (or folder) in the runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload() or {}
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one .zip file (received {len(uploaded)}; a cancelled dialog sends none): run this cell again.')
        file_name, payload = next(iter(uploaded.items()))
        if not file_name.lower().endswith('.zip'):
            raise ValueError(f'{file_name}: upload one .zip holding labels.csv and the images.')
        byod_zip = Path('work') / 'byod.zip'
        byod_zip.parent.mkdir(parents=True, exist_ok=True)
        byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    display_names = {}
    n_labels = len({r['label'] for r in records})
    raw_rows = {'byod': len(records), 'labels': n_labels, 'duplicate_images_dropped': len(records) - sum(len(part) for part in splits.values()), 'effective_minimum': min_byod_records(max(n_labels, 2))['total']}
    if len(splits['test']) < 5 * n_labels:
        print({'caution': f"only {len(splits['test'])} held-out test photographs for {n_labels} labels (fewer than 5 per label): probe accuracy moves in large steps; add photographs before reading it"})
else:
    corpus_files = fetch_corpus(cache_dir='weights/inat-birds')
    corpus = read_corpus(corpus_files)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME}: {CORPUS_RELEASE} ({CORPUS_LICENSE})'
    display_names = {key: common for key, (_scientific, common) in SPECIES.items()}
    raw_rows = {'photographs': len(corpus), 'bytes': sum(len(v) for v in corpus_files.values()), 'observers': len({r['observer'] for r in corpus})}
# The training split must hold MIN_RECORDS; validation and test only need one photograph each (split_dataset checks the labels).
dataset_manifests = {name: validate_dataset(part, min_records=MIN_RECORDS if name == 'train' else 1) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
classes = class_names(train_records)
write_dataset_csv(train_records, 'outputs/vit_mae_pretraining_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'observer_overlap': observer_overlap(splits), 'classes': classes})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'label_counts': manifest['label_counts'], 'image_side': manifest['image_side'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {'id': example['id'], 'label': example['label'], 'display_name': display_names.get(example['label'], example['label']), 'size': list(example['image'].size), 'observation': example.get('inat_observation_url')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'image over the side ceiling': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'one label only': [{**r, 'label': 'bird'} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': "iNaturalist CC0 bird photographs (six species): iNaturalist open-data bucket, research-grade CC0 photos selected 2026-09-19 (60 per species) (CC0 1.0 (each photo's own license_code on iNaturalist; observers credited in the records))", 'raw_rows': {'photographs': 360, 'bytes': 39223447, 'observers': 222}, 'splits': {'train': 216, 'validation': 48, 'test': 96}, 'observer_overlap': {'observers': 222, 'in_more_than_one_split': 53}, 'classes': ['american_goldfinch', 'chipping_sparrow', 'dark_eyed_junco', 'house_finch', 'song_sparrow', 'white_throated_sparrow']}
{'train': {'n': 216, 'label_counts': {'american_goldfinch': 36, 'chipping_sparrow': 36, 'dark_eyed_junco': 36, 'house_finch': 36, 'song_sparrow': 36, 'white_throated_sparrow': 36}, 'image_side': {'min': 342, 'max': 500}, 'digest': 'fa4ce9eb2a674c17...'}}
{'validation': {'n': 48, 'label_counts': {'american_goldfinch': 8, 'chipping_sparrow': 8, 'dark_eyed_junco': 8, 'house_finch': 8, 'song_sparrow': 8, 'white_throated_

**What to notice:** 360 photographs, 216 / 48 / 96, the three digests, `observer_overlap`, and the four refusals.

<details><summary>Check your reasoning</summary>Because the probe in Sections 6 and 8 does read them: stratifying keeps every species in every split, so the probe is scored on all six. The reconstruction numbers would be valid under any image-disjoint split. A review probe counted 53 of 222 observers in more than one split, so a probe can partly learn a photographer's style.</details>

## 5. Mask and reconstruct through the inference contract

*Core concept.* A masked autoencoder hides most of an image and predicts the hidden pixels from the visible ones; its loss is computed on the hidden patches only. **Predict before running:** on a flat-coloured drawn square, will the model's masked MSE be lower or higher than on a photograph?

The inference contract is exercised on three 32×32 synthetic shapes — a red square, a green circle and a blue triangle — rendered in code as ASCII PPM files exactly as the repository's `examples/sample-data/generate_samples.py` renders them and digest-asserted against `SHA256SUMS`, a different image family from the photographs, and images the model will be asked to reconstruct again after adaptation. `validate_inputs` applies exactly the checks the public operations apply and returns an input manifest; a remote URL is validated too and its rejection recorded as a finding. `reconstruct` hides a seeded random 75 % of the 196 patches (147 of them), runs the encoder on the visible 49 and the decoder on all 196, and returns per image the **masked MSE** (over the hidden patches, in the processor's normalised pixel space), the **visible MSE** (the decoder also predicts the patches the encoder saw), the mask itself and the reconstruction composited over the visible patches. `embed` returns the mean of the 196 patch tokens with nothing hidden, L2-normalised. The per-call `evaluation_report` on three drawn shapes is `sample-sanity` — plumbing evidence, not a measurement; whether the model reconstructs *well* is what Section 6 measures on 96 photographs.

Look for: the same seed hiding the same 147 patches twice, a different seed hiding different ones, and the pipeline's mean masked MSE equal to the model's own `loss` — the pipeline computes what the checkpoint was trained on. On these flat drawings the hidden patches are trivial to fill (masked MSE about 0.0005–0.02 in the build record) while the visible ones are not (about 0.55–0.64): the decoder was never trained on the patches the encoder saw, which is why the visible-patch error is reported but never read as quality.

In [ ]:
SAMPLE_DIGESTS = {  # examples/sample-data/SHA256SUMS
    'red_square.ppm': 'b38ff0c9131677ed6cf09832eff40a22841e1a4725d426fad3b1bf6a1dbdb096',
    'green_circle.ppm': '2e3e657686a0f6a6df3f3621d21a410d20d9a47b109f06f9405faa4f747a663f',
    'blue_triangle.ppm': 'f0f4c38c7af92b3a6b1d55a25272156edd87b41e029e116cfa059003dae029a3',
}
WIDTH, HEIGHT, BACKGROUND = 32, 32, (245, 245, 245)


def shape_mask(shape, x, y):
    if shape == 'square':
        return 8 <= x < 24 and 8 <= y < 24
    if shape == 'circle':
        return (x - 16) ** 2 + (y - 16) ** 2 <= 9**2
    if not 7 <= y < 26:
        return False
    half = (y - 7) // 2
    return 16 - half <= x <= 16 + half


def render_ppm(foreground, shape):
    # The repository's generate_samples.py rendering: ASCII P3, 24 values per line.
    lines = ['P3', f'{WIDTH} {HEIGHT}', '255']
    for y in range(HEIGHT):
        row = []
        for x in range(WIDTH):
            pixel = foreground if shape_mask(shape, x, y) else BACKGROUND
            row.extend(str(v) for v in pixel)
        for start in range(0, len(row), 24):
            lines.append(' '.join(row[start : start + 24]))
    return '\n'.join(lines) + '\n'


Path('outputs/sample-data').mkdir(parents=True, exist_ok=True)
specs = {'red_square.ppm': ((220, 40, 40), 'square'), 'green_circle.ppm': ((40, 170, 75), 'circle'), 'blue_triangle.ppm': ((40, 90, 220), 'triangle')}
shape_images = []
for name, (foreground, shape) in specs.items():
    path = Path('outputs/sample-data') / name
    path.write_bytes(render_ppm(foreground, shape).encode('ascii'))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != SAMPLE_DIGESTS[name]:
        raise ValueError(f'Synthetic sample digest mismatch for {name}: {digest} != {SAMPLE_DIGESTS[name]}')
    shape_images.append(path)
shape_sha256 = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in shape_images}
print({'ceilings': {'DEFAULT_MASK_RATIO': DEFAULT_MASK_RATIO, 'NUM_PATCHES': NUM_PATCHES, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'MIN_CLASSES': MIN_CLASSES, 'image_contract': '224x224 RGB after processor resize, 196 patches of 16x16', 'device': str(pipe.device)}})
input_manifest = validate_inputs(shape_images, names=[p.name for p in shape_images])
try:
    validate_inputs('https://example.invalid/not-allowed.png')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'remote-url-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/vit_mae_pretraining_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'shapes': [p.name for p in shape_images], 'sha256': {k: v[:16] + '...' for k, v in shape_sha256.items()}, 'manifest_verdict': input_manifest['verdict'], 'hidden_patches': input_manifest['hidden_patches'], 'findings': len(input_manifest['findings'])})
SCENE_SEED = 7
t0 = time.perf_counter()
scene = pipe.reconstruct(shape_images, seed=SCENE_SEED)
again = pipe.reconstruct(shape_images, seed=SCENE_SEED, return_images=False)
other = pipe.reconstruct(shape_images, seed=SCENE_SEED + 1, return_images=False)
shape_embeddings = pipe.embed(shape_images)['embeddings']
scene_rows = []
for path, entry in zip(shape_images, scene['results'], strict=True):
    entry['reconstruction'].save(f'outputs/vit_mae_pretraining_frozen_{path.stem}.png')
    scene_rows.append({'image': path.name, 'hidden_patches': entry['hidden_patches'], 'masked_mse': entry['masked_mse'], 'visible_mse': entry['visible_mse']})
    print(path.name, '->', {'hidden_patches': entry['hidden_patches'], 'masked_mse': round(entry['masked_mse'], 4), 'visible_mse': round(entry['visible_mse'], 4)})
checks = {
    'one_result_per_image': len(scene['results']) == len(shape_images),
    'hidden_patches_match_ratio': all(e['hidden_patches'] == int(NUM_PATCHES * DEFAULT_MASK_RATIO) for e in scene['results']),
    'errors_non_negative': all(e['masked_mse'] >= 0.0 and e['visible_mse'] >= 0.0 for e in scene['results']),
    'same_seed_same_mask': all(a['mask'] == b['mask'] for a, b in zip(scene['results'], again['results'], strict=True)),
    'other_seed_other_mask': any(a['mask'] != b['mask'] for a, b in zip(scene['results'], other['results'], strict=True)),
    'pipeline_mse_is_model_loss': abs(scene['model_loss'] - float(np.mean([e['masked_mse'] for e in scene['results']]))) < 1e-4,
    'embeddings_unit_norm': bool(np.allclose(np.linalg.norm(shape_embeddings, axis=1), 1.0, atol=1e-4)),
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
frozen_scene = evaluation_report(scene, sample_kind='synthetic')
print({'checks': checks, 'seconds': round(time.perf_counter() - t0, 2), 'frozen_scene': {m['id']: round(m['value'], 4) for m in frozen_scene['metrics']}, 'verdict': frozen_scene['verdict'], 'embedding_shape': list(shape_embeddings.shape)})

{'ceilings': {'DEFAULT_MASK_RATIO': 0.75, 'NUM_PATCHES': 196, 'MAX_IMAGE_SIDE': 4096, 'MIN_IMAGE_SIDE': 16, 'MAX_BATCH': 64, 'MIN_RECORDS': 8, 'MAX_RECORDS': 20000, 'MIN_CLASSES': 2, 'image_contract': '224x224 RGB after processor resize, 196 patches of 16x16', 'device': 'cuda'}}
{'shapes': ['red_square.ppm', 'green_circle.ppm', 'blue_triangle.ppm'], 'sha256': {'red_square.ppm': 'b38ff0c9131677ed...', 'green_circle.ppm': '2e3e657686a0f6a6...', 'blue_triangle.ppm': 'f0f4c38c7af92b3a...'}, 'manifest_verdict': 'accepted', 'hidden_patches': 147, 'findings': 1}


red_square.ppm -> {'hidden_patches': 147, 'masked_mse': 0.0005, 'visible_mse': 0.5519}
green_circle.ppm -> {'hidden_patches': 147, 'masked_mse': 0.0113, 'visible_mse': 0.5696}
blue_triangle.ppm -> {'hidden_patches': 147, 'masked_mse': 0.0228, 'visible_mse': 0.6383}
{'checks': {'one_result_per_image': True, 'hidden_patches_match_ratio': True, 'errors_non_negative': True, 'same_seed_same_mask': True, 'other_seed_other_mask': True, 'pipeline_mse_is_model_loss': True, 'embeddings_unit_norm': True}, 'seconds': 0.62, 'frozen_scene': {'masked_mse': 0.0115, 'masked_mse_visible': 0.5866, 'mask_ratio': 0.75}, 'verdict': 'sample-sanity', 'embedding_shape': [3, 768]}


**What to notice:** the masked and visible MSE per shape, and `hidden_patches` (147 of 196 for each).

<details><summary>Check your reasoning</summary>Much lower. In the CPU build record the three drawn shapes scored masked MSE 0.0005, 0.0113 and 0.0228 against 0.2281 for the test photographs: a hidden patch of a flat-coloured drawing on a white ground is predictable from its visible neighbours, while a photograph's hidden feathers and background are not. A low error on drawings says the task is easy, not that the model is good.</details>

## 6. Baselines and the frozen model on the test photographs, two ways

**Reconstruction.** `pipe.evaluate_reconstruction` hides one seeded 75 % mask per photograph (the seed is derived from the record id, so the same photograph gets the same mask in every evaluation of this notebook) and reports the mean and median **masked MSE**, the **visible MSE** and the **PSNR of the hidden patches** over the 96 test photographs. Two non-neural fills are scored on exactly the same masks: the **mean-patch fill** (every hidden patch is the mean colour of the visible patches — what a model that learned nothing about images would do) and the **blur fill** (every hidden patch is the mean colour of its visible neighbours — the cheapest use of locality). **What to look for:** the frozen model against both fills, on the mean *and* the median.

**Linear probe.** `pipe.fit_probe` embeds the 216 training photographs with nothing hidden, standardises the features with the training set's mean and standard deviation (the paper's affine-free BatchNorm) and fits a multinomial logistic-regression head by full-batch Adam; `pipe.evaluate` reports its **accuracy** and **macro F1** on the test photographs with a per-species breakdown, and beside it a **5-nearest-neighbour** vote on the same features. The **majority floor** answers every photograph with the most frequent training label (accuracy 1/6 on a balanced split), and the **colour nearest neighbour** answers with the label of the training photograph whose 3×3 mean-colour grid is closest. MAE features are known to probe poorly before fine-tuning (the paper reports 67.8 % linear-probe top-1 on ImageNet against 83.6 % after fine-tuning), and six bird species from 216 photographs is a hard probe. Read the per-species recall: it is where the probe's number comes from.

*Evaluation practice.* The cell records **verdicts** — whether the frozen model beats both fills, and whether the probe beats the majority floor — instead of asserting them: on a small BYOD test split a miss is a finding, and the notebook continues to the continuation, export and reload.

**Predict before running:** MAE features are known to probe poorly before fine-tuning. Will the linear probe beat a k-NN on the same features, and by how much will it beat the 1/6 floor?

In [ ]:
RK = ('masked_mse', 'masked_mse_median', 'masked_mse_visible', 'psnr_masked')
CK = ('accuracy', 'macro_f1')
t0 = time.perf_counter()
frozen_rec = pipe.evaluate_reconstruction(test_records, seed=0)
frozen_rec_seconds = round(time.perf_counter() - t0, 1)
print({'frozen_reconstruction_test': {k: round(frozen_rec[k], 4) for k in RK}, 'n': frozen_rec['n'], 'mask_ratio': frozen_rec['mask_ratio'], 'verdict': frozen_rec['verdict'], 'seconds': frozen_rec_seconds})
for name, fill in frozen_rec['baselines'].items():
    print({name: {k: round(fill[k], 4) for k in RK}, 'note': fill['baseline']})
print({'definitions': frozen_rec['definitions']})
# Reported verdicts, not assertions: on your images a fill may come close, and that is a finding.
frozen_rec_verdict = 'frozen below both fills' if frozen_rec['masked_mse'] < min(frozen_rec['baselines']['blur_fill']['masked_mse'], frozen_rec['baselines']['mean_patch_fill']['masked_mse']) else 'a fill matches or beats the frozen model'
print({'frozen_vs_fills': frozen_rec_verdict})

baseline_majority = majority_baseline(train_records, test_records, classes)
baseline_neighbour = colour_neighbour_baseline(train_records, test_records, classes)
print({'majority_baseline': {k: round(baseline_majority[k], 3) for k in CK}, 'n': baseline_majority['n'], 'note': baseline_majority['baseline']})
print({'colour_neighbour_baseline': {k: round(baseline_neighbour[k], 3) for k in CK}, 'note': baseline_neighbour['baseline']})
t0 = time.perf_counter()
frozen_probe_fit = pipe.fit_probe(train_records)
frozen_probe = pipe.evaluate(test_records)
probe_seconds = round(time.perf_counter() - t0, 1)
print({'frozen_probe_test': {k: round(frozen_probe[k], 3) for k in CK}, 'n': frozen_probe['n'], 'verdict': frozen_probe['verdict'], 'train_loss': [round(v, 4) for v in frozen_probe_fit['train_loss']], 'standardisation': frozen_probe_fit['standardisation'], 'seconds': probe_seconds})
print({'knn_on_the_same_features': {k: round(frozen_probe['knn'][k], 3) for k in CK}, 'note': frozen_probe['knn']['baseline']})
frozen_fields = {c: {'n': v['support'], 'recall': round(v['recall'], 2), 'f1': round(v['f1'], 2)} for c, v in frozen_probe['per_class'].items()}
print({'by_species_frozen': frozen_fields})
frozen_probe_verdict = 'above floor' if frozen_probe['accuracy'] > baseline_majority['accuracy'] else 'at or below floor'
print({'frozen_probe_vs_floor': frozen_probe_verdict})

{'frozen_reconstruction_test': {'masked_mse': 0.2281, 'masked_mse_median': 0.1801, 'masked_mse_visible': 0.2794, 'psnr_masked': 19.3365}, 'n': 96, 'mask_ratio': 0.75, 'verdict': 'measured', 'seconds': 1.8}
{'mean_patch_fill': {'masked_mse': 0.7652, 'masked_mse_median': 0.6763, 'masked_mse_visible': 0.0, 'psnr_masked': 14.0797}, 'note': 'every hidden patch = the mean colour of the visible patches'}
{'blur_fill': {'masked_mse': 0.5452, 'masked_mse_median': 0.4753, 'masked_mse_visible': 0.0, 'psnr_masked': 15.5516}, 'note': 'every hidden patch = the mean colour of its visible neighbours'}
{'definitions': {'masked_mse': "mean over images of the mean squared error between predicted and true pixels over the patches hidden from the encoder, in the processor's normalised pixel space (ImageNet mean / std); lower is better", 'masked_mse_visible': 'the same error over the patches the encoder saw (the decoder also predicts those); lower is better', 'mask_ratio': 'fraction of the patches hidden fro

{'majority_baseline': {'accuracy': 0.167, 'macro_f1': 0.048}, 'n': 96, 'note': "majority floor ('american_goldfinch' for every photograph)"}
{'colour_neighbour_baseline': {'accuracy': 0.26, 'macro_f1': 0.261}, 'note': 'colour nearest neighbour (216 training photographs)'}


{'frozen_probe_test': {'accuracy': 0.365, 'macro_f1': 0.365}, 'n': 96, 'verdict': 'measured', 'train_loss': [2.0043, 0.0067], 'standardisation': "training-set mean / std per feature (the paper's affine-free BatchNorm)", 'seconds': 5.0}
{'knn_on_the_same_features': {'accuracy': 0.219, 'macro_f1': 0.216}, 'note': '5-nearest-neighbour vote on the same features'}
{'by_species_frozen': {'american_goldfinch': {'n': 16, 'recall': 0.69, 'f1': 0.56}, 'chipping_sparrow': {'n': 16, 'recall': 0.25, 'f1': 0.22}, 'dark_eyed_junco': {'n': 16, 'recall': 0.38, 'f1': 0.36}, 'house_finch': {'n': 16, 'recall': 0.31, 'f1': 0.45}, 'song_sparrow': {'n': 16, 'recall': 0.31, 'f1': 0.33}, 'white_throated_sparrow': {'n': 16, 'recall': 0.25, 'f1': 0.26}}}
{'frozen_probe_vs_floor': 'above floor'}


**What to notice:** the masked MSE mean and median against both fills, the probe against the three classifiers, and the per-species recall.

<details><summary>Check your reasoning</summary>In the Kaggle T4 release run (21 September 2026) the frozen model's masked MSE was 0.2281 (median 0.1801) against 0.5452 for the blur fill and 0.7652 for the mean fill, PSNR 19.3 dB on the hidden patches. The probe scored accuracy 0.365 and macro F1 0.365 against the majority floor's 0.167, the colour neighbour's 0.260 and the k-NN's 0.219 — above every baseline, but with a 95 % interval of roughly ±0.10 on 96 photographs, and far from a classifier you would ship.</details>

## 7. Bounded continuation of the masked-autoencoding objective

`pipe.adapt` continues pre-training: it trains the whole decoder, the last `TRAINABLE_BLOCKS` encoder blocks and the encoder's final LayerNorm — two blocks by default, 40,286,464 of 111,907,840 parameters; the patch embedding, the position embeddings and the first ten blocks stay frozen — on the checkpoint's own loss, the masked-patch MSE under a fresh seeded random 75 % mask every step. AdamW at a fixed learning rate with weight decay 0.05, gradient clipping at 1.0, seeded shuffling and masks, no scheduler, and **no labels**. Epoch 0 records the frozen model's validation reconstruction; every epoch is scored on the 48 validation photographs with their fixed seeded masks, and the epoch with the lowest validation masked MSE is kept — so the selection can return the frozen model itself (epoch 0) when nothing beats it, and never returns one worse *on the validation split*; the test split can still move either way. Every call starts from the **pinned base**: tensors an earlier call (or an artifact) changed are put back first, so epoch 0 is always the frozen model and re-running Sections 7–8 with a changed field repeats the comparison validly. The fitted probe is discarded, because the features it was fitted on no longer exist.

Watch the numbers move very little: the pre-trained model has seen 1,600 epochs of ImageNet, and 216 photographs of birds are not a new domain to it. The build record's learning-rate sweep found `1e-4` makes the validation masked MSE **worse** from the first epoch (batches of eight with random masks are a noisy gradient), and `1e-5` moves it from 0.2335 to 0.2332 at epoch 3 of 5. That is the honest default: the configuration that did not hurt, chosen on the validation split. To compare a change side by side without replacing the default exports, use Section 10.

**Predict before running:** will any epoch beat the frozen model's validation masked MSE, and by how much?

In [ ]:
EPOCHS = 5  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 4) for k in ('masked_mse', 'masked_mse_visible', 'psnr_masked')})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


settings = {'epochs': EPOCHS, 'lr': LEARNING_RATE, 'batch_size': BATCH_SIZE, 'trainable_blocks': TRAINABLE_BLOCKS}
if settings != {'epochs': 5, 'lr': 1e-5, 'batch_size': 8, 'trainable_blocks': 2}:
    print({'note': 'changed settings: this run starts again from the pinned base and replaces the default results of Sections 8-9; Section 10 compares a change side by side instead', 'settings': settings})
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=TRAINABLE_BLOCKS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'objective': adapt_result['objective'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})
val_history = {h['epoch']: h['val']['masked_mse'] for h in adapt_result['history']}
if val_history[adapt_result['best_epoch']] > val_history[0]:  # a contract check on the selector, not a quality claim
    raise RuntimeError(f"the selector returned epoch {adapt_result['best_epoch']}, worse on validation than the frozen model")
print({'started_from': adapt_result['started_from'], 'validation_masked_mse_by_epoch': {k: round(v, 4) for k, v in val_history.items()}})

{'epoch': 1, 'train_loss': 0.2456, 'val_masked_mse': 0.2338, 'val_masked_mse_visible': 0.247, 'val_psnr_masked': 19.2288}


{'epoch': 2, 'train_loss': 0.2445, 'val_masked_mse': 0.2338, 'val_masked_mse_visible': 0.2416, 'val_psnr_masked': 19.2287}


{'epoch': 3, 'train_loss': 0.2442, 'val_masked_mse': 0.2332, 'val_masked_mse_visible': 0.2439, 'val_psnr_masked': 19.2392}


{'epoch': 4, 'train_loss': 0.2438, 'val_masked_mse': 0.2334, 'val_masked_mse_visible': 0.2475, 'val_psnr_masked': 19.2356}


{'epoch': 5, 'train_loss': 0.24, 'val_masked_mse': 0.2337, 'val_masked_mse_visible': 0.2425, 'val_psnr_masked': 19.2308}
{'trainable_parameters': 40286464, 'total_parameters': 111907840, 'objective': "masked-autoencoding continuation (masked-patch MSE, the checkpoint's own loss)", 'best_epoch': 3, 'selection': 'lowest validation masked MSE (validation split, fixed seeded masks)', 'seconds': 26.3}
{'started_from': 'pinned base', 'validation_masked_mse_by_epoch': {0: 0.2335, 1: 0.2338, 2: 0.2338, 3: 0.2332, 4: 0.2334, 5: 0.2337}}


**What to notice:** the validation masked MSE per epoch and `best_epoch`.

<details><summary>Check your reasoning</summary>Barely. In the release run validation masked MSE went 0.2335 → 0.2338 → 0.2338 → 0.2332 → 0.2334 → 0.2337 and epoch 3 was kept — a gain of 0.0003 on 48 photographs, with no dispersion estimate. A model trained for 1,600 ImageNet epochs is already converged on natural photographs.</details>

## 8. Held-out evaluation, two ways

The test photographs were never used for training or epoch selection, and no photograph appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6: the same seeded masks per photograph for the reconstruction (so a delta is the model's, not the mask's), and a probe re-fitted on the adapted features of the same 216 training photographs with the same seed. The systems are put side by side — three on reconstruction (mean fill, blur fill, frozen, adapted), five on the probe (majority, colour neighbour, k-NN, frozen probe, adapted probe) — the per-species breakdown is repeated and the evaluation report is written as JSON. Read it in this order: the **validation masked MSE** that selected the epoch, then the **test masked MSE** — its **mean and its median together**, and `paired_per_image`, how many test photographs got better or worse under the same masks — then the probe and the per-species recall. The selector's guarantee holds on the validation split only; held-out error can move either way. The cell records verdicts instead of asserting a gain. Ninety-six photographs from one seeded split give **no dispersion estimate**; accuracy moves in steps of one photograph; and a masked MSE that moved in the fourth decimal is the honest reading of continued pre-training on a converged model — sample-sanity evidence that the adaptation contract works, not a benchmark, and not a claim about your images until you measure them.

**Predict before running:** if the mean test masked MSE barely moves, will the median move the same way?

In [ ]:
adapted_rec = pipe.evaluate_reconstruction(test_records, seed=0)
adapted_val_rec = pipe.evaluate_reconstruction(val_records, seed=0)
adapted_probe_fit = pipe.fit_probe(train_records)
adapted_probe = pipe.evaluate(test_records)
adapted_fields = {c: {'n': v['support'], 'recall': round(v['recall'], 2), 'f1': round(v['f1'], 2)} for c, v in adapted_probe['per_class'].items()}
comparison = {
    'reconstruction': {k: {'mean_patch_fill': round(frozen_rec['baselines']['mean_patch_fill'][k], 4), 'blur_fill': round(frozen_rec['baselines']['blur_fill'][k], 4), 'frozen': round(frozen_rec[k], 4), 'adapted': round(adapted_rec[k], 4)} for k in RK},
    'reconstruction_delta_vs_frozen': {k: round(adapted_rec[k] - frozen_rec[k], 5) for k in RK},
    'validation_masked_mse': {'frozen': round(val_history[0], 4), 'selected_epoch': adapt_result['best_epoch'], 'selected': round(val_history[adapt_result['best_epoch']], 4), 'rescored': round(adapted_val_rec['masked_mse'], 4)},
    'probe': {k: {'majority': round(baseline_majority[k], 3), 'colour_neighbour': round(baseline_neighbour[k], 3), 'knn_frozen': round(frozen_probe['knn'][k], 3), 'knn_adapted': round(adapted_probe['knn'][k], 3), 'frozen': round(frozen_probe[k], 3), 'adapted': round(adapted_probe[k], 3)} for k in CK},
    'probe_delta_vs_frozen': {k: round(adapted_probe[k] - frozen_probe[k], 3) for k in CK},
    'by_species': {c: {'n': frozen_fields[c]['n'], 'frozen_recall': frozen_fields[c]['recall'], 'adapted_recall': adapted_fields[c]['recall'], 'frozen_f1': frozen_fields[c]['f1'], 'adapted_f1': adapted_fields[c]['f1']} for c in classes},
}
frozen_by_id = {row['id']: row['masked_mse'] for row in frozen_rec['per_image']}
paired_deltas = [row['masked_mse'] - frozen_by_id[row['id']] for row in adapted_rec['per_image']]
comparison['paired_per_image'] = {'better': sum(d < 0 for d in paired_deltas), 'worse': sum(d > 0 for d in paired_deltas), 'same': sum(d == 0 for d in paired_deltas), 'of': len(paired_deltas)}
def direction(new, old, lower_is_better):
    if new == old:
        return 'no change'
    return 'improved' if (new < old) == lower_is_better else 'worse'
# Reported verdicts, not assertions: a continuation that does not help is a result to record, and export and reload still run.
comparison['verdicts'] = {
    'frozen_vs_fills': frozen_rec_verdict,
    'frozen_probe_vs_floor': frozen_probe_verdict,
    'adapted_below_blur_fill': bool(adapted_rec['masked_mse'] < adapted_rec['baselines']['blur_fill']['masked_mse']),
    'test_masked_mse_mean': direction(adapted_rec['masked_mse'], frozen_rec['masked_mse'], True),
    'test_masked_mse_median': direction(adapted_rec['masked_mse_median'], frozen_rec['masked_mse_median'], True),
    'probe_accuracy': direction(adapted_probe['accuracy'], frozen_probe['accuracy'], False),
}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': DEFAULT_MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'classes': classes,
    'display_names': display_names,
    'reconstruction': {'frozen_test': {k: v for k, v in frozen_rec.items() if k != 'per_image'}, 'adapted_test': {k: v for k, v in adapted_rec.items() if k != 'per_image'}, 'adapted_validation': {k: v for k, v in adapted_val_rec.items() if k != 'per_image'}},
    'probe': {'baselines': {'majority': baseline_majority, 'colour_neighbour': baseline_neighbour}, 'frozen_test': {k: v for k, v in frozen_probe.items() if k != 'predictions'}, 'frozen_fit': frozen_probe_fit, 'adapted_test': {k: v for k, v in adapted_probe.items() if k != 'predictions'}, 'adapted_fit': adapted_probe_fit},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/vit_mae_pretraining_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
if abs(adapted_val_rec['masked_mse'] - val_history[adapt_result['best_epoch']]) >= 1e-4:  # a contract check: the kept epoch re-scores to the number that selected it
    raise RuntimeError('the kept epoch does not re-score to the validation masked MSE that selected it')
print({'report': 'outputs/vit_mae_pretraining_evaluation_report.json'})

{'reconstruction': {'masked_mse': {'mean_patch_fill': 0.7652, 'blur_fill': 0.5452, 'frozen': 0.2281, 'adapted': 0.228}, 'masked_mse_median': {'mean_patch_fill': 0.6763, 'blur_fill': 0.4753, 'frozen': 0.1801, 'adapted': 0.1827}, 'masked_mse_visible': {'mean_patch_fill': 0.0, 'blur_fill': 0.0, 'frozen': 0.2794, 'adapted': 0.2789}, 'psnr_masked': {'mean_patch_fill': 14.0797, 'blur_fill': 15.5516, 'frozen': 19.3365, 'adapted': 19.3375}}}
{'reconstruction_delta_vs_frozen': {'masked_mse': -5e-05, 'masked_mse_median': 0.00261, 'masked_mse_visible': -0.00054, 'psnr_masked': 0.00104}}
{'validation_masked_mse': {'frozen': 0.2335, 'selected_epoch': 3, 'selected': 0.2332, 'rescored': 0.2332}}
{'probe': {'accuracy': {'majority': 0.167, 'colour_neighbour': 0.26, 'knn_frozen': 0.219, 'knn_adapted': 0.219, 'frozen': 0.365, 'adapted': 0.365}, 'macro_f1': {'majority': 0.048, 'colour_neighbour': 0.261, 'knn_frozen': 0.216, 'knn_adapted': 0.216, 'frozen': 0.365, 'adapted': 0.365}}}
{'probe_delta_vs_frozen

**What to notice:** the mean and median rows of `reconstruction_delta_vs_frozen`, `paired_per_image`, the probe and the `verdicts`.

<details><summary>Check your reasoning</summary>No. In the release run the mean test masked MSE moved from 0.2281 to 0.2280 (−0.00005) while the median rose from 0.1801 to 0.1827 (+0.0026) — about fifty times larger and in the opposite direction: the typical photograph got slightly worse while the mean stayed flat. The probe stayed at 0.365 accuracy and 0.365 macro F1. The selector's guarantee is about the 48 validation photographs, not the test split; a single metric does not characterise the change.</details>

## 9. Re-reconstruct the drawn shapes, export the adapter and reload it

The three shapes from Section 5 are reconstructed again by the adapted model under the same seeded masks — drawings, a different image family from the photographs it was tuned on, so this is a small look at what the continuation did *outside* its corpus (the build record's numbers are in `docs/release-verification.md`; a changed error here is a finding to record, not a failure) — and reported with the per-call `evaluation_report` (`sample-sanity`). Both readings are written as JSON with the reconstructions as PNG.

`pipe.save_artifact` writes the trained tensors — the decoder, the last two encoder blocks and the encoder LayerNorm, about 161 MB — plus the probe head and its standardisation statistics as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the objective and mask ratio, the probe record (classes, steps, learning rate), the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `ViTMAEPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the declared blocks, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical masked MSE on eight test photographs under the same masks and identical probe decisions (VER4) — a contract check, so it stays a hard check.

**Predict before running:** the continuation saw only photographs. Will the drawn shapes reconstruct better or worse afterwards?

In [ ]:
import shutil

adapted_scene = pipe.reconstruct(shape_images, seed=SCENE_SEED)
adapted_scene_rows = []
for path, before, after in zip(shape_images, scene['results'], adapted_scene['results'], strict=True):
    after['reconstruction'].save(f'outputs/vit_mae_pretraining_adapted_{path.stem}.png')
    adapted_scene_rows.append({'image': path.name, 'hidden_patches': after['hidden_patches'], 'masked_mse': after['masked_mse'], 'visible_mse': after['visible_mse']})
    print({'image': path.name, 'frozen_masked_mse': round(before['masked_mse'], 4), 'adapted_masked_mse': round(after['masked_mse'], 4), 'same_mask': before['mask'] == after['mask']})
adapted_scene_report = evaluation_report(adapted_scene, sample_kind='synthetic')
print({'scene_after_adaptation': {m['id']: round(m['value'], 4) for m in adapted_scene_report['metrics']}, 'verdict': adapted_scene_report['verdict']})
with open('outputs/vit_mae_pretraining_shapes.json', 'w', encoding='utf-8') as handle:
    json.dump({'seed': SCENE_SEED, 'frozen': scene_rows, 'adapted': adapted_scene_rows}, handle, indent=2)

artifact_dir = Path('outputs/vit_mae_pretraining_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'vit_mae_pretraining', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'probe_classes': artifact_manifest['probe']['classes'], 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = ViTMAEPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_images = [r['image'] for r in test_records[:8]]
before = pipe.reconstruct(parity_images, seed=11, return_images=False)['results']
after = reloaded.reconstruct(parity_images, seed=11, return_images=False)['results']
before_cls, after_cls = pipe.classify(parity_images)['results'], reloaded.classify(parity_images)['results']
parity = {
    'max_abs_masked_mse_difference': float(max(abs(a['masked_mse'] - b['masked_mse']) for a, b in zip(before, after, strict=True))),
    'identical_reconstructions': int(sum(abs(a['masked_mse'] - b['masked_mse']) < 1e-6 for a, b in zip(before, after, strict=True))),
    'identical_probe_decisions': int(sum(a['top1'] == b['top1'] for a, b in zip(before_cls, after_cls, strict=True))),
    'max_abs_probe_score_difference': float(max(abs(a['scores'][c] - b['scores'][c]) for a, b in zip(before_cls, after_cls, strict=True) for c in a['scores'])),
    'of': len(parity_images),
}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch'], 'reloaded_probe_classes': reloaded.probe['classes']})
assert parity['identical_reconstructions'] == parity['of'] and parity['identical_probe_decisions'] == parity['of'] and parity['max_abs_probe_score_difference'] < 1e-4

write_provenance('outputs/provenance.json', pipeline=pipe)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'fetched_this_run': fetched, 'weight_file': MODEL_FILENAME, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': MODEL_SHA256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'base_url': CORPUS_BASE_URL, 'bytes': CORPUS_BYTES, 'pinned_photographs': len(SAMPLE_RECORDS), 'species': {k: list(v) for k, v in SPECIES.items()}},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'shapes': {'names': [p.name for p in shape_images], 'sha256': shape_sha256, 'seed': SCENE_SEED}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene_report, 'frozen_rows': scene_rows, 'adapted_rows': adapted_scene_rows},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'device': str(pipe.device), 'dtype': 'float32', 'checkpoint_source': pipe.checkpoint_source},
}
with open('outputs/vit_mae_pretraining_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

{'image': 'red_square.ppm', 'frozen_masked_mse': 0.0005, 'adapted_masked_mse': 0.0006, 'same_mask': True}
{'image': 'green_circle.ppm', 'frozen_masked_mse': 0.0113, 'adapted_masked_mse': 0.0117, 'same_mask': True}
{'image': 'blue_triangle.ppm', 'frozen_masked_mse': 0.0228, 'adapted_masked_mse': 0.0219, 'same_mask': True}
{'scene_after_adaptation': {'masked_mse': 0.0114, 'masked_mse_visible': 0.5838, 'mask_ratio': 0.75}, 'verdict': 'sample-sanity'}


{'artifact': 'outputs/vit_mae_pretraining_adapter', 'format': 'org.valcorza.vit-mae-base.adapter.v1', 'tensors': 174, 'probe_classes': ['american_goldfinch', 'chipping_sparrow', 'dark_eyed_junco', 'house_finch', 'song_sparrow', 'white_throated_sparrow'], 'bytes': 161190912, 'sha256': '1c361926a2509af2...'}


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


{'reload_parity': {'max_abs_masked_mse_difference': 0.0, 'identical_reconstructions': 8, 'identical_probe_decisions': 8, 'max_abs_probe_score_difference': 1.1920928955078125e-06, 'of': 8}, 'reloaded_best_epoch': 3, 'reloaded_probe_classes': ['american_goldfinch', 'chipping_sparrow', 'dark_eyed_junco', 'house_finch', 'song_sparrow', 'white_throated_sparrow']}
['provenance.json', 'sample-data', 'vit_mae_pretraining_adapted_blue_triangle.png', 'vit_mae_pretraining_adapted_green_circle.png', 'vit_mae_pretraining_adapted_red_square.png', 'vit_mae_pretraining_adapter', 'vit_mae_pretraining_evaluation_report.json', 'vit_mae_pretraining_frozen_blue_triangle.png', 'vit_mae_pretraining_frozen_green_circle.png', 'vit_mae_pretraining_frozen_red_square.png', 'vit_mae_pretraining_input_manifest.json', 'vit_mae_pretraining_result.json', 'vit_mae_pretraining_shapes.json', 'vit_mae_pretraining_train.csv']


**What to notice:** `frozen_masked_mse` against `adapted_masked_mse` for each shape (`same_mask` should be true), and the reload parity line.

<details><summary>Check your reasoning</summary>In the CPU build record the three shapes moved from 0.0005 / 0.0113 / 0.0228 to 0.0006 / 0.0117 / 0.0219 under the same masks — two slightly worse, one slightly better — and three drawings cannot tell you which is typical — record the change as a finding about the continuation outside its corpus, not as a measurement. Reload parity held in the release run: identical reconstructions and probe decisions on eight of eight test photographs.</details>

## 10. Change one thing: a ten-times-higher learning rate, or another mask ratio (optional)

*Evaluation practice.* A **Predict → Change one thing → Run → Observe → Explain** activity, off by default so Run all is unaffected. Set `RUN_EXPERIMENT = True`, change **one** field — by default the learning rate goes from 1e-5 to 1e-4; `EXPERIMENT_MASK_RATIO` changes the mask ratio for both the continuation and its own frozen control — and run this cell after Sections 4–9. The experiment loads its **own** pipeline from the verified snapshot, so it starts from the checkpoint and never touches the default `pipe`; it scores its own frozen control at the same mask ratio, writes only to `outputs/vit_mae_pretraining_experiment/`, prints the default and the changed run side by side (including epoch 0, which must equal the default's at the default ratio), and checks that the default exports are byte-identical afterwards.

**Predict:** at 1e-4, which epoch will the selector keep?

In [ ]:
RUN_EXPERIMENT = False  # @param {type:"boolean"}
EXPERIMENT_LEARNING_RATE = 1e-4  # @param {type:"number"}
EXPERIMENT_MASK_RATIO = 0.75  # @param {type:"number"}
EXPERIMENT_TRAINABLE_BLOCKS = 2  # @param {type:"integer"}

if not RUN_EXPERIMENT:
    print({'experiment': 'skipped (RUN_EXPERIMENT = False); the default path above is complete'})
else:
    canonical_files = {'adapter': artifact_dir / 'adapter.safetensors', 'evaluation_report': Path('outputs/vit_mae_pretraining_evaluation_report.json'), 'result': Path('outputs/vit_mae_pretraining_result.json')}
    canonical = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in canonical_files.items()}
    experiment_dir = Path('outputs/vit_mae_pretraining_experiment')
    shutil.rmtree(experiment_dir, ignore_errors=True)
    experiment_dir.mkdir(parents=True)
    # Its own pipeline from the verified snapshot: the experiment starts from the checkpoint and the default pipe is untouched.
    experiment_pipe = ViTMAEPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
    experiment_control = experiment_pipe.evaluate_reconstruction(test_records, mask_ratio=EXPERIMENT_MASK_RATIO, seed=0)
    experiment_result = experiment_pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=EXPERIMENT_LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=EXPERIMENT_TRAINABLE_BLOCKS, mask_ratio=EXPERIMENT_MASK_RATIO, progress=report)
    experiment_rec = experiment_pipe.evaluate_reconstruction(test_records, mask_ratio=EXPERIMENT_MASK_RATIO, seed=0)
    side_by_side = {
        'settings': {'default': {'lr': adapt_result['lr'], 'mask_ratio': adapt_result['mask_ratio'], 'trainable_blocks': adapt_result['trainable_blocks']}, 'experiment': {'lr': EXPERIMENT_LEARNING_RATE, 'mask_ratio': EXPERIMENT_MASK_RATIO, 'trainable_blocks': EXPERIMENT_TRAINABLE_BLOCKS}},
        'validation_masked_mse_by_epoch': {'default': [round(h['val']['masked_mse'], 4) for h in adapt_result['history']], 'experiment': [round(h['val']['masked_mse'], 4) for h in experiment_result['history']]},
        'best_epoch': {'default': adapt_result['best_epoch'], 'experiment': experiment_result['best_epoch']},
        'test_masked_mse': {'default_frozen': round(frozen_rec['masked_mse'], 4), 'default_adapted': round(adapted_rec['masked_mse'], 4), 'experiment_frozen_control': round(experiment_control['masked_mse'], 4), 'experiment_adapted': round(experiment_rec['masked_mse'], 4)},
        'test_masked_mse_median': {'default_frozen': round(frozen_rec['masked_mse_median'], 4), 'default_adapted': round(adapted_rec['masked_mse_median'], 4), 'experiment_frozen_control': round(experiment_control['masked_mse_median'], 4), 'experiment_adapted': round(experiment_rec['masked_mse_median'], 4)},
    }
    for key, row in side_by_side.items():
        print({key: row})
    with open(experiment_dir / 'experiment_report.json', 'w', encoding='utf-8') as handle:
        json.dump({'side_by_side': side_by_side, 'history': experiment_result['history']}, handle, indent=2, ensure_ascii=False, default=str)
    unchanged = {name: hashlib.sha256(path.read_bytes()).hexdigest() == canonical[name] for name, path in canonical_files.items()}
    if not all(unchanged.values()):
        raise RuntimeError(f'the experiment changed a default export: {unchanged}')
    print({'default_exports_unchanged': unchanged, 'experiment_outputs': str(experiment_dir)})
    del experiment_pipe

{'experiment': 'skipped (RUN_EXPERIMENT = False); the default path above is complete'}


**Observe → Explain.** Compare the two validation curves (epoch 0 must be equal at the default mask ratio) and the test rows against each run's own frozen control.

<details><summary>Check your reasoning</summary>The build record's sweep found that `1e-4` makes the validation masked MSE worse from the first epoch — batches of eight with random masks are a noisy gradient — so the selector keeps epoch 0, the frozen model, and the experiment's adapted row equals its control. That is the selector doing its job. With a lower mask ratio the encoder sees more of each image, so both the control and the adapted masked MSE fall; compare each run only with its own control. No experiment run is recorded on the release runtime.</details>

## Interpretation and limits

The frozen model reconstructs hidden patches of photographs it was never trained on far better than the two non-neural fills (masked MSE 0.2281 against 0.5452 and 0.7652 in the build record), and its mean-pooled features carry enough to lift a linear probe over six bird species above the majority floor, the colour neighbour and a k-NN on the same features (0.365 against 0.167, 0.260 and 0.219) — while still being, as the paper says of MAE features, a poor linear-probe backbone until fine-tuned. A bounded continuation of the masked-autoencoding objective on 216 photographs, chosen on the validation split, moved the held-out mean masked MSE from 0.2281 to 0.2280 while the median rose from 0.1801 to 0.1827 (Kaggle T4 release run, 21 September 2026), left the probe at 0.365, and exported a 161 MB adapter that reloads to identical reconstructions and probe decisions. That is the claim: the adaptation contract works end to end on a real photograph set, the selector is honest enough to keep the frozen model when nothing beats it on validation — a guarantee about the validation split, not the test split —  and the numbers it produces are read on the objective and on a probe, per species, against non-neural baselines and the frozen model rather than in isolation.

What the numbers say is that continued pre-training on a few hundred in-domain photographs does not change a model that has seen 1,600 epochs of ImageNet: the build record's learning-rate sweep (`docs/release-verification.md`) found `1e-4` hurts from the first epoch and `1e-5` moves the validation masked MSE in the fourth decimal. Where continuation earns its place is a domain the checkpoint has not seen — medical, satellite, microscopy, line art — and that is what BYOD is for; the check in Section 7 (the kept epoch is never worse than the frozen model on validation) and the baselines in Sections 6 and 8 are what make such a run readable. The test split is 96 photographs from one seeded draw of one sample, the validation split that picks the epoch is 48, a masked MSE is the model's own objective and not a perceptual judgement, and the probe's accuracy moves in steps of one photograph.

Three things to carry to real data. **Baselines first:** the two fills and the three non-neural classifiers on *your* images are the numbers to read before any adapted one. **Leakage:** keep every photograph in one split (the contract de-duplicates by decoded pixels) and split by photographer or session when your images come from few sources — the sample's observer overlap is printed for exactly that reason. **The readout is not the product:** a linear probe measures the features; a classifier you would ship is a full fine-tuning of the encoder, which this repository does not expose.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real photograph set, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded continuation of the pre-training objective, evaluate against non-neural baselines and the frozen model on an image-disjoint split two ways, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, representation quality on any other population or camera, or production fitness.

**Optional experiments (off by default; each names its field and what to run):** Section 10 runs a changed continuation in its own pipeline beside the default — `EXPERIMENT_LEARNING_RATE` (1e-4 by default: watch the selector keep epoch 0), `EXPERIMENT_MASK_RATIO` (for example 0.5, scored against its own frozen control) or `EXPERIMENT_TRAINABLE_BLOCKS` (0 trains the decoder alone; compare the artifact size); change one and run that cell again. Changing `EPOCHS`, `LEARNING_RATE` or `TRAINABLE_BLOCKS` and choosing **Run after** from Section 7 also starts from the pinned base — every `adapt` restores it first — but replaces the default results and exports. BYOD: `USE_BYOD` and `BYOD_PATH` in Section 4, then **Run after** from Section 4; bring images from an unfamiliar domain and read the fills before the adapted number.

## Troubleshooting

- **Section 1 stops with "This notebook needs a Linux x86_64 runtime"** — you are on Windows, macOS or an ARM machine. Use Google Colab, Kaggle or a Linux x86_64 Jupyter server.
- **The uv wheel fails its size/SHA-256 check, or a download in Section 1 times out** — run Section 1 again; a complete environment is reused, an incomplete one is finished. If it repeats, the network is blocking or altering `files.pythonhosted.org` or `pypi.org`.
- **"The isolated environment's Python process exited"** — usually out of memory. Restart the session and choose **Run all**; leave the optional experiment off on a small runtime.
- **You re-ran Section 1 on its own** — nothing is lost: it keeps the running worker and every variable, so the cells after it keep working. After a session restart, run from the top.
- **Section 3 reports a size or SHA-256 mismatch, or cannot reach the Hub** — the message names the file. Delete it from the snapshot folder Section 3 prints and run Section 3 again; the snapshot comes from `huggingface.co`.
- **Section 4 cannot fetch a photograph, or one fails its digest** — `fetch_corpus` names it; the default path needs `inaturalist-open-data.s3.amazonaws.com`. Run Section 4 again (cached photographs are re-hashed); delete `weights/inat-birds/` if a cached file is corrupt.
- **Out of memory** — lower `BATCH_SIZE` in Section 7, or restart the session and choose **Run all**; leave Section 10 off on a small runtime.
- **BYOD: "BYOD_PATH … does not exist"** — the path is relative to the working directory printed in the message.
- **BYOD: "the upload dialog exists only in Google Colab"** — on Kaggle or Jupyter, put the zip in the runtime (or attach it as a dataset) and set `BYOD_PATH`.
- **BYOD: "Upload exactly one .zip file"** — the dialog was cancelled or several files were chosen; run the cell again.
- **BYOD: "labels.csv line N (file …): that image file is not in the dataset"** — fix the `file` column of that row, or add the image to the zip.
- **BYOD: "… Pillow cannot decode the image"** — the file on that line is corrupt or not an image.
- **BYOD: "split leaves no train/validation photograph for labels …" or "… training records"** — add photographs; the message names the minimum for your label count.
## Glossary

- **Mask / mask ratio** — the random set of hidden patches; 75 % of 196 by default, seeded per photograph here.
- **Masked MSE / visible MSE** — the mean squared error over the hidden patches (the model's objective) and over the visible ones, in the processor's normalised pixel space.
- **PSNR** — peak signal-to-noise ratio of the hidden patches, in dB; higher is better.
- **Mean-patch fill / blur fill** — two non-neural reconstructions: the mean visible colour, and the mean of each hidden patch's visible neighbours.
- **Mean pooling / standardisation** — averaging the patch tokens into one vector; scaling each feature by the training set's mean and standard deviation.
- **Linear probe / k-NN** — a logistic-regression head on frozen features; a nearest-neighbour vote on the same features.
- **Accuracy / macro F1** — the fraction right; the mean of per-class F1.
- **Epoch selection** — keeping the epoch with the lowest validation masked MSE; a guarantee on validation only.
- **Paired per-image count** — how many test photographs got better or worse under the same masks.
- **Adapter / reload parity** — the trained tensors and the probe head (safetensors) overlaid on the pinned base; the reloaded pipeline gives identical outputs.
- **BYOD** — bring your own data: your photographs through the same cells.

## Conclusion (your notes)

Optional — fill in from **your** run, not the recorded one:

- The data was ___ photographs over ___ labels; test split ___.
- Frozen masked MSE ___ (median ___) against the blur fill ___ and the mean fill ___.
- Probe accuracy ___ against the floor ___, the colour neighbour ___ and the k-NN ___.
- After continuation (epoch ___ kept): masked MSE ___ (median ___), ___ photographs better and ___ worse; probe ___.
- What I would need before claiming continuation helps: ___ (for example an unfamiliar domain, more test images, several seeds).

## References

- Repository README: https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/MODEL_CARD.md
- Sample dataset card (synthetic shapes): https://github.com/kurtvalcorza/vit-mae-pretraining-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream model: https://huggingface.co/facebook/vit-mae-base
- Upstream code: https://github.com/facebookresearch/mae — served through https://github.com/huggingface/transformers (`ViTMAEForPreTraining`)
- Masked Autoencoders Are Scalable Vision Learners (He, Chen, Xie, Li, Dollár and Girshick, CVPR 2022): https://arxiv.org/abs/2111.06377
- The SigLIP zero-shot row in this fleet, sharing the corpus and the code shape: https://github.com/kurtvalcorza/siglip-v1-zero-shot-pipeline
- iNaturalist open data (CC0 photographs, each observer's own licence): https://www.inaturalist.org/pages/developers — bucket https://inaturalist-open-data.s3.amazonaws.com/
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)